# ⚽ More than a Hack 2027 — Phase 1: Ball Possession
## Pipeline Multi-Skenario: LightGBM · XGBoost · CatBoost · Random Forest · Ensemble · YOLO

**Pertandingan:** FC Barcelona (`H`) vs Athletic Club (`A`) — LaLiga 2024-25, matchday 2
**Output:** satu label `H` / `A` / `D` / `X` per frame (199.319 baris), **satu paket submission per skenario**
**Metrik peringkat resmi:** `spellF1` (possession sebagai interval, IoU 0,3–0,7); pemecah seri `macroF1`

---

### Arsitektur pipeline

```
  Video taktis ──► [YOLO: deteksi bola] ──┐            (hanya S6)
                                          ▼
  ATD (tracking) ──► A. Pra-pemrosesan bola ──► B. M1: siapa dekat bola di dalam gap ──┐
                               │                                                       │
                               └──► C. Kontak & kontrol (Jalur A dwell / Jalur B rantai)├──► E. Viterbi H/A/D ──► CSV
  Smart Tagging ──► label lemah ──► D. M2: bola mati ──────────────────────────────────┘
```

### Skenario yang dibandingkan

| ID | Nama | Sumber bola | Model M1 & M2 | Perangkat |
|---|---|---|---|---|
| S1 | LightGBM | ATD | LightGBM | CPU |
| S2 | XGBoost | ATD | XGBoost | CPU (opsional GPU) |
| S3 | CatBoost | ATD | CatBoost | CPU (opsional GPU) |
| S4 | RandomForest | ATD | Random Forest | CPU |
| S5 | Ensemble | ATD | *Stacking* S1–S4 (regresi logistik di atas prediksi out-of-fold) | CPU |
| S6 | YOLO_Ensemble | ATD **+ deteksi bola YOLO dari video** | *Stacking* S1–S4 | **GPU** |

Semua skenario memakai **fitur, fold validasi, aturan kontrol, dan parameter decoding yang sama**,
sehingga perbedaan hasil berasal dari model (S1–S5) atau dari sumber bola (S6).

### Hasil yang dihasilkan untuk setiap skenario
```
work/runs_v3/<RUN_ID>/scenarios/S1_LightGBM/
  ├─ <team_name>.csv          file possession (sudah divalidasi validator resmi)
  ├─ <team_name>.md           draf dokumen teknis (English) khusus skenario ini
  ├─ validation.txt           output validate_submission.py resmi
  ├─ scores.json              semua metrik skenario
  └─ figures/                 gambar khusus skenario
work/runs_v3/<RUN_ID>/comparison/
  ├─ comparison_table.csv     tabel perbandingan semua skenario
  ├─ vs_previous.csv          selisih tiap skenario terhadap run v2 (otomatis)
  ├─ README.md                ringkasan perbandingan + panduan memilih
  └─ figures/                 grafik perbandingan
```
**Keputusan skenario mana yang dikirim ada di tangan tim** setelah membaca Bagian 15.

### Perubahan versi 2 (setelah run pertama pada data asli)
| Masalah di run 1 | Perbaikan v2 |
|---|---|
| YOLO tidak belajar (mAP50 0,03): kotak label 8 px memotong bola | kotak minimal 20 px (`min_box_px`, `box_scale`); gambar train/val lama **dipakai ulang**, hanya label ditulis ulang |
| Ambang confidence tetap 0,25 -> 0 deteksi terpilih | ambang dipilih otomatis via F1 pada frame evaluasi (`conf_select = "auto"`) |
| M2 (bola mati) terlemah: AUC 0,72–0,80 | label lemah gol→kick-off, fitur bola diam, jarak ke titik restart, pemain berjalan (`m2_version = 2`) |
| Bola mati sangat pendek memecah possession | durasi minimum bola mati 2 s di Viterbi (`min_dead_s`) |
| Timing transisi sulit dibaca | kolom `pub_transF1`, `pub_trans_match`, `pub_latency_med` di tabel perbandingan |
| Hasil run lama tertimpa? | hasil v2 di `<work_dir>/runs_v2/`; hasil lama di `runs/` tetap utuh dan dibandingkan otomatis (Bagian 15.1) |

### Perubahan versi 3 (fokus: YOLO / S6) — hasil v2 tidak disentuh
Pada v2, S6 justru terendah (spellF1 publik 0,337): 31% deteksi terpilih salah (rumput, kaki, sebagian bola) dan
ambang terbaik jatuh di 0,05. Penyebab utama: **pusat label meleset** (posisi ATD tidak tepat di atas bola, galat
kira-kira sebesar diameter bola), sehingga isi kotak label sering rumput/kaki. v3 memperbaiki dari hulu ke hilir:

| # | Masalah v2 | Perbaikan v3 | Parameter |
|---|---|---|---|
| 1 | Pusat label = proyeksi ATD (meleset) | **Self-training**: model v2 menjadi "guru" yang mencari bola di sekitar posisi ATD; pusat dihaluskan dengan detektor blob (DoG) dan dicek kontrasnya. Crop tanpa bola yang terkonfirmasi **dibuang** | `yolo_v3.relabel_*` |
| 2 | Kotak tetap ≥ 20 px (banyak rumput) | Kotak **mengikuti ukuran bola tampak** + margin, minimal 15 px; dikalibrasi dari ukuran bola terukur (13 px di sisi dekat kamera) dan skala homography per posisi | `ball_px_ref`, `min_box_px` |
| 3 | Model menandai kaki/rumput | **Contoh negatif**: crop yang sama dengan bola dihapus (inpainting) → "kaki tanpa bola = bukan bola" | `neg_frac` |
| 4 | Ambang dipilih via F1 (0,05) | Ambang dipilih agar **presisi ≥ 0,90** pada segmen evaluasi | `target_precision` |
| 5 | Deteksi diterima per frame | **Filter lintasan**: deteksi harus membentuk lintasan yang mungkin secara fisik (kecepatan bola) atau tersambung ke posisi ATD di ujung gap | `track_*` |
| 6 | Posisi YOLO = setara ATD | **Fusi hati-hati**: bukti kontak dari posisi YOLO diberi bobot lebih kecil, kontrol butuh dwell lebih lama, M1 hanya dilatih dari frame ATD | `yolo_fusion` |
| 7 | Evaluasi frame tunggal | Evaluasi pada **segmen 2 s** (bola ATD terlihat, disimulasikan sebagai gap) → seluruh rantai seleksi ikut diuji | `eval_segments` |

**Pemisahan folder v3:** hasil di `<work_dir>/runs_v3/`, checkpoint baru di `<work_dir>/checkpoints_v3/`, dataset & run
YOLO baru di `<folder YOLO>/dataset_v3` dan `runs_v3`. **Cache sumber dibaca bersama tanpa diubah**: checkpoint v2
(`checkpoints/`: parsing ATD, fitur pemain, model S1–S5, info dataset YOLO, bobot YOLO v2 sebagai guru) dan gambar
crop YOLO v2 (`dataset/`). Karena itu S1–S5 dimuat dari cache v2 (hasilnya identik dengan v2, jadi menjadi pembanding
yang adil) dan yang dihitung ulang hanya rantai YOLO serta sumber `yolo`.

### Versi 2 (ringkas)
Checkpoint yang tidak terpengaruh (parsing ATD, fitur pemain, data & model M1) **dipakai ulang**; yang dihitung ulang
hanya M2, dataset/training/deteksi YOLO, dan tahap setelahnya.

### Cara pakai
1. Isi **Bagian 1 (CONFIG)**: `team_name`, `env`, dan `data_dir` (lokasi folder data di Google Drive).
2. Jalankan semua sel. Tahap berat disimpan sebagai **checkpoint**; jalankan ulang untuk melanjutkan.
3. Baca tabel perbandingan (Bagian 15), lalu pilih skenario. Opsional: isi `submit_scenario` agar file
   terpilih disalin ke `work/submission_final/`.

## 0. ⚠️ Aturan kepatuhan (wajib dibaca tim)

Dari dokumen resmi kompetisi (`FAQ.md`, `OVERVIEW.md`, `RULES.md`):

- **Dilarang** mengunggah video, dataset, atau cuplikannya ke **LLM cloud pihak ketiga** (ChatGPT, Claude, Gemini, dll.).
  Jangan memakai fitur AI assistant pada sel yang berisi data. **Gambar dari Bagian 12 (YOLO) berisi potongan video
  — jangan dibagikan ke asisten AI mana pun.**
- Yang aman dibagikan: **skor**, output validator, pesan error tanpa baris data, dan statistik agregat.
- **Label harus dihasilkan algoritma.** Data latih YOLO dibuat otomatis dari tracking (pseudo-label), bukan anotasi tangan.
- **Jangan commit data** atau turunannya (frame video, dataset YOLO, ekspor tracking) ke repo publik.
  Yang masuk repo: notebook, kode, dan dokumentasi.
- Repo publik wajib bisa **mereproduksi file submission yang sama**. Seed, versi pustaka, commit repo resmi,
  dan hash file tercatat di `run_manifest.json` dan di dokumen teknis setiap skenario.
- **Lisensi YOLO:** Ultralytics memakai AGPL-3.0. Repo submission yang publik memenuhi syarat ini; cantumkan lisensinya.

## 1. Konfigurasi utama

**Satu-satunya sel yang perlu diubah.** Untuk **Colab dengan runtime lokal (GPU lokal)**: set `env = "local"` dan isi
`data_dir` dengan path folder Drive di komputer, misalnya `G:/My Drive/MTAH2027/data` (Google Drive for Desktop, Windows)
atau `~/Library/CloudStorage/GoogleDrive-<email>/My Drive/MTAH2027/data` (macOS). Bila dibiarkan `None`,
notebook mencoba mencarinya otomatis.

> ⚠️ **Path Windows wajib ditulis sebagai raw string `r"..."` atau dengan garis miring `/`.**
> Di Python, `"H:\.shortcut\1_ABC"` biasa akan rusak: `\1` dibaca sebagai karakter kontrol (octal), `\n` sebagai
> baris baru, `\t` sebagai tab. Benar: `r"H:\.shortcut-targets-by-id\1_ABC\Dataset"` atau
> `"H:/.shortcut-targets-by-id/1_ABC/Dataset"`. Folder **pintasan (shortcut) dari folder yang dibagikan** bisa dipakai
> langsung lewat path `.shortcut-targets-by-id`; disarankan set foldernya "Available offline" di Google Drive for Desktop
> agar file tidak diunduh ulang saat dibaca.

In [ ]:
CONFIG = {
    # ---------------- Identitas & environment ----------------
    "team_name": "V3locity",               # nama file <team_name>.csv / .md di setiap skenario
    "env": "auto",                          # "auto" | "colab" (runtime hosted) | "local" (runtime lokal / GPU lokal)
    "drive_root": "/content/drive/MyDrive/MTAH2027",   # dipakai bila env = colab (hosted)
    "data_dir": None,                       # folder berisi 'tactical/'. None = cari otomatis di ./data
    "auto_extract_zip": True,               # bila data masih berupa .zip di data_dir, ekstrak otomatis ke work_dir
    "work_dir": None,                       # folder kerja (checkpoint + hasil). None = ~/MTAH2027_work;
                                            # mis. r"H:\My Drive\mtah_2027". None = ~/MTAH2027_work
    "storage_mode": "auto",                 # "auto" | "drive" | "local". auto: "drive" bila work_dir di folder sinkron cloud
                                            # mode drive: data dibaca langsung dari Drive (tanpa cache), semua I/O
                                            # dibaca/ditulis ulang otomatis bila Drive sedang sinkron, disk lokal minimal
    "results_mirror_dir": None,             # opsional: folder kedua untuk salinan hasil akhir
    "results_dirname": "runs_v3",           # hasil run disimpan di <work_dir>/<results_dirname>/<RUN_ID>;
                                            # hasil lama di runs/ dan runs_v2/ TIDAK disentuh.
    "checkpoint_dirname": "checkpoints_v3", # checkpoint BARU ditulis ke <work_dir>/<checkpoint_dirname>
    "shared_checkpoint_dirs": ["checkpoints"],  # cache sumber yang DIBACA saja (checkpoint v2), tidak pernah ditulis/dihapus
    "compare_with_previous": "auto",        # "auto" = run terbaru di compare_prefer_dirname | path folder run | None
    "compare_prefer_dirname": "runs_v2",    # folder hasil yang diutamakan sebagai pembanding
    "min_free_gb_local": 3.0,               # batas aman ruang kosong disk sistem (dipakai Drive for Desktop utk streaming)
    "io_retries": 6, "io_retry_wait_s": 2.0, # percobaan ulang baca/tulis file di Drive
    "local_cache_dir": "auto",              # "auto": mode local -> <work_dir>/data_cache (disalin sekali); mode drive -> tidak ada
                                            # cache (baca langsung). None = selalu baca langsung. Atau path folder
    "official_repo_url": "https://github.com/metrica-sports/more-than-a-hack-2027",
    "official_repo_commit": "3de6b60aeabd2475d95879a651d38b72bc4fba00",
    "force_stages": [],                     # contoh ["m1_xgb_atd"] untuk memaksa hitung ulang;
                                            # ["yolo_train"] = training YOLO dari epoch 1 (tanpa resume)
    "seed": 2027,
    "n_threads": 8,                         # dikunci agar hasil deterministik; catat nilai ini di dokumen
    "gbdt_device": "cpu",                   # "cpu" (deterministik, disarankan) | "cuda" (XGBoost/CatBoost di GPU)
    "fig_dpi": 150,
    "show_figures": True,

    # ---------------- Skenario ----------------
    "scenarios": {
        "S1": {"name": "LightGBM",      "source": "atd",  "model": "lgbm"},
        "S2": {"name": "XGBoost",       "source": "atd",  "model": "xgb"},
        "S3": {"name": "CatBoost",      "source": "atd",  "model": "cat"},
        "S4": {"name": "RandomForest",  "source": "atd",  "model": "rf"},
        "S5": {"name": "Ensemble",      "source": "atd",  "model": "ensemble"},
        "S6": {"name": "YOLO_Ensemble", "source": "yolo", "model": "ensemble"},
        # Tambahan tanpa biaya latih ekstra (model anggota sudah dilatih untuk S6), contoh:
        # "S7": {"name": "YOLO_LightGBM", "source": "yolo", "model": "lgbm"},
    },
    "ensemble_members": ["lgbm", "xgb", "cat", "rf"],
    "ensemble_method": "auto",              # "auto" (pilih via log-loss OOF) | "stack" | "mean"
    "submit_scenario": None,                # mis. "S5": salin paket skenario itu ke work/submission_final/

    # ---------------- Lapangan & waktu ----------------
    "pitch": [105.3, 68.0],
    "fps": 25,

    # ---------------- Tahap A: pra-pemrosesan bola ----------------
    "ball_max_speed": 40.0, "ball_min_run": 3, "ball_max_outside": 8.0,
    "savgol_window": 7, "savgol_order": 2,

    # ---------------- Tahap B: M1 (gap bola) ----------------
    "short_gap_max": 10, "gap_aug_per_frame": 2, "gap_max_train": 400000,
    "gap_cls_max_dist": 2.0, "cv_folds": 4, "cv_block_s": 120,

    # ---------------- Hiperparameter model ----------------
    "lgbm": {"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 63, "min_child_samples": 50,
             "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8},
    "xgb": {"n_estimators": 400, "learning_rate": 0.05, "max_depth": 8, "min_child_weight": 5,
            "subsample": 0.8, "colsample_bytree": 0.8, "max_bin": 256},
    "cat": {"iterations": 500, "learning_rate": 0.08, "depth": 8, "l2_leaf_reg": 3.0},
    "rf": {"n_estimators": 200, "min_samples_leaf": 20, "max_features": "sqrt", "max_samples": 0.5},

    # ---------------- Tahap C: kontak & kontrol ----------------
    "r_pz": 1.0, "r_dz": 1.0, "contact_merge_gap": 3, "t_dwell": 8, "pass_window": 100,

    # ---------------- Tahap D: bola mati ----------------
    "dead_lead_s": 1.0, "dead_weak_len_s": 2.5, "alive_weak_margin_s": 1.0,
    "m2_version": 2,                        # v2: label gol->kick-off, fitur bola diam & titik restart
    "goal_dead_cap_s": 120,                 # batas panjang label "mati" setelah gol

    # ---------------- Tahap E: decoding Viterbi ----------------
    "p_control": 0.95, "p_contact": 0.65, "min_mass_frames": 6, "gap_weight": 0.10, "st_weight": 0.0,
    "dead_temp": 0.25, "lam_switch": 10.0, "lam_to_dead": 8.0, "lam_from_dead": 8.0,
    "min_dead_s": 2.0,                      # v2: bola mati minimal 2 s (restart butuh waktu); 0 = nonaktif
    "switch_delay_frames": 0,               # eksperimen: geser bukti pergantian tim (frame); 0 = tanpa geser
    "abstain_in_gaps": False,

    # ---------------- YOLO (S6) ----------------
    "yolo": {
        "enabled": True,
        "require_gpu": True,          # True: S6 dilewati bila tidak ada GPU
        "device": "auto",             # "auto" | 0 | "cpu"
        "weights": ["yolo26s.pt", "yolo11s.pt"],   # dicoba berurutan (diunduh otomatis oleh ultralytics)
        "crop": 960,                  # ukuran potongan latih (piksel asli, tanpa resize)
        "upscale": 1.0,               # >1 memperbesar gambar (membantu bola kecil, lebih lambat)
        "n_train": 3000, "n_val": 600, "n_eval": 2000,
        "epochs": 40, "patience": 10, "batch": 8, "workers": None,    # batch 8 aman untuk GPU 6 GB; None: 0 di Windows
        "label_max_speed": 15.0,      # m/s; frame latih hanya bola "di tanah" (kecepatan rendah)
        "ball_diam_m": 0.22, "box_scale": 4.0, "min_box_px": 20,   # v2: kotak 8 px memotong bola -> minimal 20 px
        "conf_store": 0.05, "conf_select": "auto", "max_det": 5,  # "auto": ambang dipilih via F1 pada frame evaluasi
        "max_prior_dist_m": 40.0, "match_radius_m": 1.5,
        "infer_batch": 4, "save_every": 2000,   # infer_batch 4 untuk frame 1920x1080 di GPU 6 GB (turun otomatis bila OOM)
        "dataset_location": "auto",   # "auto": temp disk lokal bila ruang cukup, selain itu di work_dir (Drive) + baca ulang
                                      # "local_temp" | "work_dir"
        "jpeg_quality": 90,           # ukuran dataset ~0,25 MB/gambar pada crop 960
        "cleanup_local_temp": False,  # v2: dataset disimpan agar bisa dipakai ulang (±1,7 GB di temp lokal)
        "reuse_previous_images": True, # pakai ulang gambar train/val run sebelumnya; hanya label yang ditulis ulang
        "max_infer_frames": None,     # batasi jumlah frame inferensi (hanya untuk uji cepat)
        # Catatan v3: parameter di atas (crop, n_train, box_scale, min_box_px, ...) adalah parameter DATASET DASAR v2.
        # Jangan diubah: dengan nilai yang sama, gambar crop v2 dipakai ulang tanpa mendekode video lagi.
    },

    # ---------------- YOLO v3: label bersih, kotak terkalibrasi, negatif, presisi, lintasan ----------------
    "yolo_v3": {
        "teacher_weights": "auto",    # "auto": bobot YOLO terbaik (mAP50) dari checkpoint v2 | path .pt | None (tanpa guru)
        "relabel": True,              # perbaiki pusat label dengan guru + detektor blob
        "relabel_conf": 0.10,         # confidence minimum deteksi guru di sekitar posisi ATD
        "relabel_conf_min": 0.03,     # confidence yang disimpan saat inferensi guru (untuk diagnostik)
        "relabel_radius_m": 1.5,      # radius pencarian bola di sekitar posisi ATD (meter, dikonversi ke piksel per frame)
        "relabel_max_shift_frac": 0.5,# geseran maksimum oleh detektor blob, x diameter bola tampak
        "relabel_min_contrast": 5.0,  # bola harus lebih terang dari cincin sekitarnya (skala abu 0-255); None = tanpa cek
        "ball_px_ref": 13.0,          # diameter bola terukur (px) ...
        "ball_px_ref_where": "near",  # ... di sisi lapangan dekat kamera ("near") / jauh ("far"), tengah panjang lapangan,
        "ball_px_ref_inset_m": 5.0,   # ... sekitar 5 m ke dalam dari garis samping
        "box_margin_px": 1.5,         # margin per sisi di luar diameter bola tampak
        "min_box_px": 15, "max_box_px": 40,
        "neg_frac": 0.20,             # jumlah contoh negatif (bola dihapus) = 20% dari crop positif
        "init_weights": "pretrained", # "pretrained" (bobot COCO di CONFIG['yolo']['weights']) | "teacher" (lanjut dari v2)
        "epochs": 50, "patience": 15,
        "eval_segments": 40, "eval_seg_len": 50,   # 40 segmen x 2 s pada blok validasi (bola ATD terlihat)
        "target_precision": 0.90,     # ambang confidence terkecil yang mencapai presisi ini pada segmen evaluasi
        "thr_grid": [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80],
        "track_topk": 3,              # kandidat per frame yang dipertimbangkan
        "track_vmax": 35.0,           # m/s, kecepatan bola maksimum yang masuk akal
        "track_tol_m": 0.75,          # toleransi posisi antar-deteksi (meter)
        "track_max_skip": 3,          # frame kosong yang boleh dilompati dalam satu lintasan
        "track_min_len": 5,           # lintasan lepas minimal 5 deteksi (0,2 s)
        "track_min_len_anchored": 2,  # lintasan yang tersambung ke bola ATD di ujung gap cukup 2 deteksi
    },
    "yolo_fusion": {
        "evidence_weight": 0.5,       # bobot bukti kontak dari posisi YOLO (1 = setara ATD, seperti v2)
        "t_dwell_weak": 16,           # kontrol dari posisi YOLO butuh dwell >= 16 frame (ATD: t_dwell = 8)
        "m1_train_on_atd_only": True, # M1 sumber yolo dilatih hanya dari frame bola ATD (YOLO hanya memperpendek gap)
    },

    # ---------------- Evaluasi ----------------
    "bootstrap_n": 1000,
    "sweep": {"lam_switch": [4, 6, 8, 10, 14, 20, 28], "dead_temp": [0.1, 0.25, 0.5]},
    "enable_video_overlay": False,
    "extra_gt_csv": None,             # biarkan None (hanya untuk menguji pipeline dengan data sintetis)
}

## 2. Environment, instalasi, dan lokasi data

- **colab**: runtime hosted Google; Drive di-*mount* otomatis.
- **local**: Jupyter/Colab yang terhubung ke **runtime lokal** (GPU di komputer sendiri). Drive dibaca dari folder
  Google Drive for Desktop.

### Mode penyimpanan
| | **mode `drive`** (otomatis bila `work_dir` di Google Drive Desktop) | **mode `local`** |
|---|---|---|
| Data (ATD, video 5 GB) | dibaca **langsung** dari Drive, tidak disalin | disalin sekali ke `<work_dir>/data_cache` |
| Checkpoint & hasil | di `work_dir` (Drive) | di `work_dir` |
| Dataset YOLO (±1 GB) | temp disk lokal **bila ruang cukup**, lalu dihapus; bila tidak, di Drive | `work_dir` |
| Baca/tulis file | **dicoba ulang otomatis** bila Drive sedang sinkron/mengunduh | biasa |

Google Drive for Desktop adalah *file system virtual*: file bisa sedang diunduh, diunggah, atau dikunci. Penyebab error
`Image Not Found` sebelumnya: Ultralytics membaca gambar sekali saja dan langsung gagal. Mode `drive` menambahkan
pembacaan ulang (Ultralytics, video, checkpoint), memverifikasi setiap file yang ditulis, dan memberi **pesan error
terstruktur** (tahap, penyebab, langkah perbaikan). Drive for Desktop tetap butuh ruang kosong di disk sistem untuk
*streaming*; notebook memeriksa ini di awal (`min_free_gb_local`).

**PyTorch tidak dipasang otomatis**, karena `pip install torch` di Windows memasang versi CPU. Bila memakai GPU lokal,
pasang dulu versi CUDA dari <https://pytorch.org/get-started/locally/>, lalu jalankan ulang sel ini.

Struktur data yang dicari (sesuai `data/README.md` resmi):
```
<data_dir>/tactical/BAR-ATH_tactical.mp4 · _atd.xml · _atd.txt · _homography.csv · _smart_tagging.xml
```

In [ ]:
import os, sys, io, re, json, time, glob, shutil, pickle, hashlib, platform, warnings, itertools, zipfile
import subprocess, importlib, importlib.util, urllib.request

_override = os.environ.get("MTAH_NB_CONFIG")          # dipakai untuk pengujian otomatis
if _override:
    _ov = json.loads(_override)
    for _k, _v in _ov.items():
        if isinstance(_v, dict) and isinstance(CONFIG.get(_k), dict) and _k != "scenarios":
            CONFIG[_k].update(_v)
        else:
            CONFIG[_k] = _v


def detect_env(choice):
    if choice in ("colab", "local"):
        return choice
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        return "local"


ENV = detect_env(CONFIG["env"])
IS_WINDOWS = os.name == "nt"
print("Environment:", ENV, "| OS:", platform.system())
if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")

REQUIRED = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
            "lightgbm": "lightgbm", "xgboost": "xgboost", "catboost": "catboost",
            "matplotlib": "matplotlib", "cv2": "opencv-python-headless"}
_missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Memasang:", _missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import numpy as np
import pandas as pd
import scipy
from scipy.signal import savgol_filter
import sklearn
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
import xgboost as xgb
import catboost
from catboost import CatBoostClassifier
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from matplotlib.backends.backend_pdf import PdfPages

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(CONFIG["seed"])

In [ ]:
import tempfile, errno, contextlib, traceback


def disk_free_gb(path):
    """Ruang kosong (GB) pada disk tempat path berada; None bila tidak bisa dibaca."""
    p = os.path.abspath(path)
    while p and not os.path.exists(p) and os.path.dirname(p) != p:
        p = os.path.dirname(p)
    try:
        return shutil.disk_usage(p).free / 1e9
    except OSError:
        return None


class PipelineError(RuntimeError):
    """Error dengan penjelasan yang bisa langsung ditindaklanjuti."""


def explain_error(stage, exc):
    msg = "{}: {}".format(type(exc).__name__, exc)
    low = msg.lower()
    winerr = getattr(exc, "winerror", None)
    if "bobot yolo" in low or ("download" in low and ".pt" in low):
        cause = "Bobot awal YOLO tidak bisa dimuat/diunduh."
        fix = ["Pastikan komputer terhubung ke internet saat run pertama (bobot diunduh otomatis dari GitHub Ultralytics).",
               "Periksa nama di CONFIG['yolo']['weights'] (mis. 'yolo26s.pt', 'yolo11s.pt').",
               "Atau unduh manual file .pt dan isi path lengkapnya di CONFIG['yolo']['weights']."]
    elif isinstance(exc, (ImportError, ModuleNotFoundError)):
        cause = "Pustaka Python belum terpasang di environment yang dipakai Jupyter."
        fix = ["Pasang pustaka yang disebut pada pesan asli (pip install ...) di environment yang sama, lalu restart kernel."]
    elif isinstance(exc, MemoryError):
        cause = "RAM komputer tidak cukup."
        fix = ["Tutup aplikasi lain lalu jalankan ulang.", "Kurangi CONFIG['gap_max_train'] atau jumlah skenario."]
    elif "out of memory" in low:
        cause = "Memori GPU tidak cukup."
        fix = ["Turunkan CONFIG['yolo']['batch'] (mis. 4) dan/atau CONFIG['yolo']['infer_batch'] (mis. 2).",
               "Tutup aplikasi lain yang memakai GPU (browser, game, dll.)."]
    elif getattr(exc, "errno", None) == errno.ENOSPC or winerr == 112 or "no space left" in low or "not enough space" in low:
        cause = "Disk penuh saat menulis file."
        fix = ["Kosongkan ruang di disk sistem (C:) — Google Drive for Desktop juga memakai C: untuk streaming.",
               "Bila dataset YOLO ditaruh di temp lokal, set CONFIG['yolo']['dataset_location'] = 'work_dir'."]
    elif isinstance(exc, PermissionError) or winerr in (5, 32, 33):
        cause = "File sedang dikunci (biasanya oleh sinkronisasi Google Drive, antivirus, atau jendela Explorer yang terbuka)."
        fix = ["Tunggu sinkronisasi Drive selesai (ikon Drive di taskbar), tutup Explorer yang membuka folder kerja.",
               "Jalankan ulang 'Run all' — tahap yang sudah selesai dimuat dari checkpoint."]
    elif "image not found" in low or "gagal membaca frame video" in low or isinstance(exc, (FileNotFoundError, OSError)):
        cause = ("File belum tersedia/terbaca dari Google Drive for Desktop (sedang diunduh, diunggah, atau dikeluarkan "
                 "dari cache lokal karena disk sistem hampir penuh).")
        fix = ["Pastikan Google Drive for Desktop berjalan dan online.",
               "Kosongkan ruang di disk sistem (minimal {} GB).".format(CONFIG["min_free_gb_local"]),
               "Klik kanan folder data di Drive -> 'Available offline' bila ruang memungkinkan.",
               "Jalankan ulang 'Run all': file rusak/hilang dideteksi dan dibuat ulang otomatis."]
    else:
        cause = "Error tidak terduga."
        fix = ["Kirim 3 baris terakhir pesan ini (tanpa data/gambar) untuk dianalisis."]
    lines = ["", "=" * 78, "  ERROR pada tahap: {}".format(stage), "-" * 78, "  Pesan asli : {}".format(msg[:600]),
             "  Penyebab   : {}".format(cause), "  Perbaikan  :"] + ["    {}. {}".format(i + 1, f) for i, f in enumerate(fix)]
    lines += ["  Progres    : tahap yang selesai sudah tersimpan sebagai checkpoint; 'Run all' melanjutkan dari sana.",
              "=" * 78]
    return "\n".join(lines)


@contextlib.contextmanager
def stage(name):
    """Bungkus satu tahap: bila gagal, tampilkan penjelasan terstruktur lalu hentikan dengan PipelineError."""
    try:
        yield
    except PipelineError:
        raise
    except KeyboardInterrupt:
        print("\nDihentikan manual pada tahap: {} (progres yang tersimpan tetap aman).".format(name))
        raise
    except Exception as exc:
        text = explain_error(name, exc)
        print(text)
        err = PipelineError(text)
        err.short = "{} -> {}: {}".format(name, type(exc).__name__, str(exc).strip()[:160])
        raise err from exc


def retry_io(fn, what, retries=None, wait=None):
    """Jalankan operasi file dengan percobaan ulang (untuk Google Drive yang sedang sinkron)."""
    retries = CONFIG["io_retries"] if retries is None else retries
    wait = CONFIG["io_retry_wait_s"] if wait is None else wait
    for attempt in range(retries + 1):
        try:
            return fn()
        except (OSError, EOFError, pickle.UnpicklingError) as exc:
            if attempt == retries:
                raise
            print("  [I/O] {} gagal ({}), mencoba ulang {}/{} ...".format(what, type(exc).__name__, attempt + 1, retries))
            time.sleep(wait * (attempt + 1))


TACTICAL_FILES = {"atd_raw": "BAR-ATH_tactical_atd.txt", "atd_meta": "BAR-ATH_tactical_atd.xml",
                  "smart_tagging": "BAR-ATH_tactical_smart_tagging.xml",
                  "homography": "BAR-ATH_tactical_homography.csv", "video": "BAR-ATH_tactical.mp4"}
REQUIRED_ASSETS = ("atd_raw", "atd_meta", "smart_tagging", "homography")


def find_assets(roots, max_depth=5):
    found = {}
    for root in roots:
        if not root or not os.path.isdir(root):
            continue
        base_depth = root.rstrip("/\\").count(os.sep)
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count(os.sep) - base_depth >= max_depth:
                dirnames[:] = []
            dirnames[:] = [d for d in dirnames if d.lower() != "broadcast" and not d.startswith(".")]
            for key, name in TACTICAL_FILES.items():
                if key not in found and name in filenames:
                    found[key] = os.path.join(dirpath, name)
            if all(k in found for k in TACTICAL_FILES):
                return found
        if all(k in found for k in REQUIRED_ASSETS):
            return found
    return found


def check_data_dir(path):
    """Diagnosis path data_dir sebelum pencarian file; menghasilkan pesan yang bisa langsung ditindaklanjuti."""
    if not path:
        return None
    bad = [c for c in path if ord(c) < 32]
    if bad:
        raise ValueError(
            "CONFIG['data_dir'] berisi karakter kontrol {} -- tanda path Windows ditulis tanpa raw string, sehingga "
            "'\\1', '\\n', '\\t', dll. diubah Python.\nPath yang terbaca: {!r}\nPerbaikan: tulis r\"H:\\...\" "
            "atau ganti semua '\\' dengan '/'.".format([hex(ord(c)) for c in bad], path))
    path = os.path.normpath(os.path.expanduser(path))
    if not os.path.isdir(path):
        parent = path
        while parent and not os.path.isdir(parent) and os.path.dirname(parent) != parent:
            parent = os.path.dirname(parent)
        hint = ""
        if os.path.isdir(parent):
            hint = "\nFolder terdalam yang ada: {}\nIsinya: {}".format(parent, sorted(os.listdir(parent))[:30])
        raise FileNotFoundError("Folder data_dir tidak ditemukan: {}{}\nPastikan Google Drive for Desktop berjalan "
                                "dan huruf drive (H:, G:, ...) benar.".format(path, hint))
    return path


def extract_zips(path, dest):
    """Ekstrak arsip .zip di data_dir (maks. 2 level) yang berisi file BAR-ATH_tactical_*."""
    done = []
    for zp in glob.glob(os.path.join(path, "*.zip")) + glob.glob(os.path.join(path, "*", "*.zip")):
        try:
            with zipfile.ZipFile(zp) as zf:
                names = zf.namelist()
                if any(os.path.basename(n) in TACTICAL_FILES.values() for n in names):
                    out = os.path.join(dest, os.path.splitext(os.path.basename(zp))[0])
                    if not os.path.isdir(out):
                        print("Mengekstrak", os.path.basename(zp), "->", out)
                        zf.extractall(out)
                    done.append(out)
        except zipfile.BadZipFile:
            print("Lewati arsip rusak:", zp)
    return done


CLOUD_MARKERS = ("/my drive", "/drive saya", "/shared drives", "/drive bersama", ".shortcut-targets-by-id",
                 "googledrive", "google drive", "/cloudstorage/", "onedrive", "dropbox", "icloud")


def is_cloud_synced(path):
    p = os.path.abspath(path).lower().replace("\\", "/")
    return any(m in p for m in CLOUD_MARKERS)


def default_work_dir():
    """Lokasi kerja yang TIDAK bergantung pada folder tempat Jupyter dijalankan (agar cache/checkpoint ditemukan lagi)."""
    if CONFIG["work_dir"]:
        return os.path.normpath(os.path.expanduser(CONFIG["work_dir"]))
    if ENV == "colab":
        return os.path.join(CONFIG["drive_root"], "work")
    return os.path.join(os.path.expanduser("~"), "MTAH2027_work")


CONFIG["data_dir"] = check_data_dir(CONFIG["data_dir"])
_roots = [CONFIG["data_dir"], os.environ.get("MTAH_DATA_DIR")]
if ENV == "colab":
    _roots += [os.path.join(CONFIG["drive_root"], "data"), CONFIG["drive_root"], "/content/drive/MyDrive"]
else:
    _home = os.path.expanduser("~")
    _roots += [os.path.abspath("data")]
    _roots += glob.glob("[A-Z]:/My Drive/MTAH2027") + glob.glob("[A-Z]:/Drive Saya/MTAH2027")
    _roots += glob.glob(os.path.join(_home, "Library", "CloudStorage", "GoogleDrive-*", "My Drive", "MTAH2027"))
    _roots += glob.glob(os.path.join(_home, "Google Drive", "My Drive", "MTAH2027"))
ASSETS = find_assets([r for r in _roots if r])
_absent = [k for k in REQUIRED_ASSETS if k not in ASSETS]
if _absent and CONFIG["data_dir"] and CONFIG["auto_extract_zip"]:
    _xbase = default_work_dir()
    if ENV == "local" and is_cloud_synced(_xbase):
        _xbase = os.path.join(os.path.expanduser("~"), "MTAH2027_work")
    _xdir = os.path.join(_xbase, "data_extracted")
    _ex = extract_zips(CONFIG["data_dir"], _xdir)
    if _ex:
        ASSETS = find_assets(_ex + [r for r in _roots if r])
        _absent = [k for k in REQUIRED_ASSETS if k not in ASSETS]
if _absent:
    _listing = ""
    if CONFIG["data_dir"]:
        _items = []
        for _dp, _dn, _fn in os.walk(CONFIG["data_dir"]):
            _depth = os.path.relpath(_dp, CONFIG["data_dir"]).count(os.sep)
            if _depth > 2:
                continue
            _items += [os.path.relpath(os.path.join(_dp, f), CONFIG["data_dir"]) for f in _fn]
            if len(_items) > 40:
                break
        _listing = "\nFile yang terlihat di data_dir (maks. 40): {}".format(_items[:40])
    raise FileNotFoundError("File data tidak ditemukan: {}.\nIsi CONFIG['data_dir'] dengan folder yang berisi "
                            "'tactical/'.{}".format([TACTICAL_FILES[k] for k in _absent], _listing))
print("Data ditemukan di:", os.path.dirname(ASSETS["atd_raw"]))

WORK_DIR = default_work_dir()
RESULTS_MIRROR = CONFIG["results_mirror_dir"]
STORAGE_MODE = CONFIG["storage_mode"]
if STORAGE_MODE == "auto":
    STORAGE_MODE = "drive" if (ENV == "local" and is_cloud_synced(WORK_DIR)) else "local"
with stage("menyiapkan folder kerja"):
    os.makedirs(WORK_DIR, exist_ok=True)
    _probe = os.path.join(WORK_DIR, ".write_test")
    retry_io(lambda: open(_probe, "w").write("ok"), "uji tulis work_dir")
    retry_io(lambda: os.remove(_probe), "hapus file uji")

SYS_DISK = os.path.expanduser("~")
STORAGE = {"mode": STORAGE_MODE, "work_dir": WORK_DIR, "free_work_gb": disk_free_gb(WORK_DIR),
           "free_system_gb": disk_free_gb(SYS_DISK), "free_temp_gb": disk_free_gb(tempfile.gettempdir()),
           "temp_dir": tempfile.gettempdir()}
print("\n=== Penyimpanan ===")
print("  mode           {}".format(STORAGE_MODE))
print("  work_dir       {}  (ruang kosong {})".format(WORK_DIR, "{:.1f} GB".format(STORAGE["free_work_gb"])
                                                       if STORAGE["free_work_gb"] is not None else "tidak terbaca"))
print("  disk sistem    {}  (ruang kosong {:.1f} GB)".format(SYS_DISK, STORAGE["free_system_gb"] or 0))
print("  folder temp    {}  (ruang kosong {:.1f} GB)".format(STORAGE["temp_dir"], STORAGE["free_temp_gb"] or 0))
if STORAGE["free_system_gb"] is not None and STORAGE["free_system_gb"] < CONFIG["min_free_gb_local"]:
    print("\n  PERINGATAN: ruang kosong disk sistem hanya {:.1f} GB (< {} GB). Google Drive for Desktop memakai disk ini "
          "untuk streaming file;\n  bila terlalu sempit, file bisa gagal terbaca di tengah proses. Kosongkan ruang bila "
          "memungkinkan.".format(STORAGE["free_system_gb"], CONFIG["min_free_gb_local"]))
if STORAGE_MODE == "drive":
    print("  Mode drive: data dibaca langsung dari Drive, I/O dicoba ulang otomatis, disk lokal hanya untuk dataset YOLO "
          "sementara (bila muat).")
    _old_cache = os.path.join(WORK_DIR, "data_cache")
    if os.path.isdir(_old_cache):
        print("  Catatan: '{}' dari run lama tidak dipakai di mode drive; boleh dihapus untuk menghemat kuota Drive."
              .format(_old_cache))
if RESULTS_MIRROR:
    print("  salinan hasil  {}".format(RESULTS_MIRROR))

_cache = CONFIG["local_cache_dir"]
if _cache == "auto":
    _cache = ("/content/mtah_data_local" if ENV == "colab" else
              (None if STORAGE_MODE == "drive" else os.path.join(WORK_DIR, "data_cache")))
if _cache:
    with stage("menyalin data ke cache"):
        os.makedirs(_cache, exist_ok=True)
        for key in list(ASSETS):
            if key == "video" and not CONFIG["yolo"]["enabled"]:
                continue
            dst = os.path.join(_cache, os.path.basename(ASSETS[key]))
            if os.path.abspath(dst) == os.path.abspath(ASSETS[key]):
                continue
            src_st = os.stat(ASSETS[key])
            fresh = (os.path.exists(dst) and os.path.getsize(dst) == src_st.st_size
                     and abs(os.path.getmtime(dst) - src_st.st_mtime) < 2)
            if fresh:
                print("Cache dipakai (tidak disalin ulang):", os.path.basename(dst))
            else:
                free = disk_free_gb(_cache)
                if free is not None and free < src_st.st_size / 1e9 + 1:
                    raise OSError(errno.ENOSPC, "ruang {:.1f} GB tidak cukup untuk menyalin {} ({:.1f} GB) ke cache; "
                                  "set local_cache_dir=None".format(free, os.path.basename(dst), src_st.st_size / 1e9))
                t0 = time.time()
                print("Menyalin ke cache ({:.0f} MB): {} ...".format(src_st.st_size / 1e6, os.path.basename(dst)))
                shutil.copy2(ASSETS[key], dst + ".part")      # file .part: salinan yang terputus tidak dianggap lengkap
                os.replace(dst + ".part", dst)
                print("  selesai dalam {:.0f} s".format(time.time() - t0))
            ASSETS[key] = dst

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
FAST_DIR = "/content/mtah_fast" if ENV == "colab" else WORK_DIR   # lokasi YOLO dipilih ulang di Bagian 12 (mode drive)
DIRS = {"checkpoints": os.path.join(WORK_DIR, CONFIG["checkpoint_dirname"]), "vendor": os.path.join(WORK_DIR, "vendor"),
        "yolo": os.path.join(FAST_DIR, "yolo"), "run": os.path.join(WORK_DIR, CONFIG["results_dirname"], RUN_ID)}
DIRS.update({"figures": os.path.join(DIRS["run"], "figures"), "reports": os.path.join(DIRS["run"], "reports"),
             "models": os.path.join(DIRS["run"], "models"), "scenarios": os.path.join(DIRS["run"], "scenarios"),
             "comparison": os.path.join(DIRS["run"], "comparison")})
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)


def _gpu_name():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=10)
        return out.stdout.strip() or "tidak ada"
    except Exception:
        return "tidak ada"


ENV_INFO = {"env": ENV, "python": platform.python_version(), "platform": platform.platform(),
            "gpu": _gpu_name(), "cpu_count": os.cpu_count(), "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "lightgbm": lgb.__version__,
            "xgboost": xgb.__version__, "catboost": catboost.__version__, "opencv": cv2.__version__,
            "matplotlib": matplotlib.__version__}
print("\n=== Environment ===")
for k, v in ENV_INFO.items():
    print("  {:<13} {}".format(k, v))
SHARED_CKPT_DIRS = [os.path.join(WORK_DIR, d) for d in (CONFIG["shared_checkpoint_dirs"] or [])]
print("\n=== Lokasi ===\n  WORK_DIR      {}\n  RUN_ID        {}".format(WORK_DIR, RUN_ID))
print("  hasil run     {}".format(DIRS["run"]))
print("  checkpoint    {}  (tulis)".format(DIRS["checkpoints"]))
for _d in SHARED_CKPT_DIRS:
    print("  cache bersama {}  ({})".format(_d, "baca saja" if os.path.isdir(_d) else "tidak ada"))
for k, v in ASSETS.items():
    print("  {:<13} {}  ({:.1f} MB)".format(k, v, os.path.getsize(v) / 1e6))
if "video" not in ASSETS:
    print("\nPERINGATAN: video taktis tidak ditemukan -> skenario berbasis YOLO akan dilewati.")

## 3. Sistem checkpoint dan penyimpanan visualisasi

`CKPT.run(nama, fungsi, parameter, dependensi)`: kunci = hash(nama + parameter + kunci dependensi).
Run ulang memuat tahap yang sudah selesai; mengubah parameter suatu tahap hanya menghitung ulang tahap itu
dan turunannya. **v3:** checkpoint baru ditulis ke `checkpoints_v3/`; tahap yang kuncinya sama dengan v2 dimuat dari
`checkpoints/` (log: `DIMUAT (cache bersama: checkpoints)`) tanpa disalin, sehingga folder v2 tetap utuh.
`save_fig()` menyimpan PNG bernomor (per kelompok: umum, skenario, perbandingan) dan mencatatnya
di manifest; di akhir dibuat PDF gabungan dan galeri HTML.

In [ ]:
def load_pickle(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)


def write_text(path, text):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)


def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=str, ensure_ascii=False)


class Checkpointer:
    """Checkpoint per tahap. v3: checkpoint baru ditulis ke `root`; bila belum ada, file dengan nama & kunci yang SAMA
    dicari di `shared` (checkpoint v2) dan dimuat dari sana TANPA disalin atau diubah (cache sumber bersama)."""

    def __init__(self, root, force=(), shared=()):
        self.root = root
        self.force = set(force)
        self.shared = [d for d in shared if d and os.path.isdir(d)
                       and os.path.normcase(os.path.abspath(d)) != os.path.normcase(os.path.abspath(root))]
        self.registry_path = os.path.join(root, "registry.json")
        self.registry = json.load(open(self.registry_path)) if os.path.exists(self.registry_path) else {}
        self.keys = {}
        self.loaded_from_shared = {}

    @staticmethod
    def make_key(name, params, deps):
        blob = json.dumps({"stage": name, "params": params, "deps": list(deps)}, sort_keys=True, default=str)
        return hashlib.sha1(blob.encode()).hexdigest()[:12]

    def key_for(self, name, params, deps=()):
        return self.make_key(name, params, [self.keys[d] for d in deps])

    def locate(self, name, key):
        """(path_baca, dari_cache_bersama). Bila tidak ada di mana pun: path tulis di root."""
        fname = "{}_{}.pkl".format(name, key)
        own = os.path.join(self.root, fname)
        if os.path.exists(own):
            return own, False
        for d in self.shared:
            p = os.path.join(d, fname)
            if os.path.exists(p):
                return p, True
        return own, False

    def _label(self, shared_hit, path):
        return "DIMUAT (cache bersama: {})".format(os.path.basename(os.path.dirname(path))) if shared_hit else "DIMUAT  "

    def peek(self, name, params, deps=(), validate=None):
        """Muat checkpoint bila ada & valid TANPA menjalankan apa pun; daftarkan kuncinya. None bila tidak ada."""
        key = self.key_for(name, params, deps)
        path, shared_hit = self.locate(name, key)
        if name in self.force or not os.path.exists(path):
            return None
        try:
            result = retry_io(lambda: load_pickle(path), "memuat checkpoint " + name)
        except Exception:
            return None
        if validate is not None and not validate(result):
            return None
        self.keys[name] = key
        if shared_hit:
            self.loaded_from_shared[name] = path
        print("[checkpoint] {:<18} {} key={}".format(name, self._label(shared_hit, path), key))
        return result

    def run(self, name, fn, params, deps=(), validate=None):
        """validate(result) -> True/False: checkpoint yang menunjuk ke file eksternal (dataset, bobot) diperiksa dulu."""
        key = self.key_for(name, params, deps)
        read_path, shared_hit = self.locate(name, key)
        path = os.path.join(self.root, "{}_{}.pkl".format(name, key))      # hasil baru SELALU ditulis ke root
        result, loaded = None, False
        if os.path.exists(read_path) and name not in self.force:
            t0 = time.time()
            try:
                result = retry_io(lambda: load_pickle(read_path), "memuat checkpoint " + name)
                loaded = validate is None or bool(validate(result))
            except Exception as exc:
                print("[checkpoint] {:<18} file rusak ({}) -> dihitung ulang".format(name, type(exc).__name__))
                loaded, validate = False, None
            if loaded:
                if shared_hit:
                    self.loaded_from_shared[name] = read_path
                print("[checkpoint] {:<18} {} key={}  ({:.1f} s)".format(name, self._label(shared_hit, read_path), key,
                                                                         time.time() - t0))
            elif validate is not None:
                print("[checkpoint] {:<18} TIDAK VALID (file eksternal hilang/rusak) -> dihitung ulang".format(name))
        if not loaded:
            t0 = time.time()
            with stage(name):
                result = fn()
            tmp = path + ".tmp"

            def _save():
                with open(tmp, "wb") as fh:
                    pickle.dump(result, fh, protocol=pickle.HIGHEST_PROTOCOL)
                    fh.flush()
                    os.fsync(fh.fileno())
                os.replace(tmp, path)
            with stage("menyimpan checkpoint " + name):
                retry_io(_save, "menyimpan checkpoint " + name)
            self.registry[name] = {"key": key, "path": path, "seconds": round(time.time() - t0, 1),
                                   "created": time.strftime("%Y-%m-%d %H:%M:%S")}
            try:
                retry_io(lambda: write_json(self.registry_path, self.registry), "menulis registry")
            except OSError:
                print("  (registry.json tidak tertulis; tidak mempengaruhi checkpoint)")
            print("[checkpoint] {:<18} DIHITUNG key={}  ({:.1f} s)".format(name, key, time.time() - t0))
        self.keys[name] = key
        return result


CKPT = Checkpointer(DIRS["checkpoints"], CONFIG["force_stages"], SHARED_CKPT_DIRS)
print("Checkpoint baru ->", DIRS["checkpoints"])
print("Cache bersama (baca saja) ->", CKPT.shared or "tidak ada")
FIG_MANIFEST = {}


def save_fig(fig, name, caption="", folder=None, group="umum"):
    folder = folder or DIRS["figures"]
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, name + ".png")
    retry_io(lambda: fig.savefig(path, dpi=CONFIG["fig_dpi"], bbox_inches="tight"), "menyimpan gambar " + name)
    FIG_MANIFEST["{}/{}".format(group, name)] = {"path": path, "caption": caption, "group": group}
    with open(os.path.join(DIRS["figures"], "figures_manifest.json"), "w") as fh:
        json.dump(FIG_MANIFEST, fh, indent=2, ensure_ascii=False)
    if CONFIG["show_figures"]:
        plt.show()
    plt.close(fig)
    print("  gambar disimpan:", path)


def save_table(df, name, folder=None):
    path = os.path.join(folder or DIRS["reports"], name + ".csv")
    retry_io(lambda: df.to_csv(path, index=True), "menyimpan tabel " + name)
    print("  tabel disimpan:", path)
    return path


L_PITCH, W_PITCH = CONFIG["pitch"]
FPS = CONFIG["fps"]
COLORS = {"H": "#a50044", "A": "#1f5fa8", "D": "#9e9e9e", "X": "#ffffff"}
SC_COLORS = ["#4c72b0", "#dd8452", "#55a868", "#c44e52", "#8172b3", "#937860", "#da8bc3", "#8c8c8c"]


def draw_pitch(ax, color="#3a3a3a"):
    L, W = L_PITCH, W_PITCH
    ax.add_patch(Rectangle((0, 0), L, W, fill=False, ec=color, lw=1))
    ax.plot([L / 2, L / 2], [0, W], color=color, lw=1)
    ax.add_patch(Circle((L / 2, W / 2), 9.15, fill=False, ec=color, lw=1))
    for x0, sgn in ((0, 1), (L, -1)):
        ax.add_patch(Rectangle((x0 if sgn > 0 else x0 - 16.5, W / 2 - 20.16), 16.5, 40.32, fill=False, ec=color, lw=1))
        ax.add_patch(Rectangle((x0 if sgn > 0 else x0 - 5.5, W / 2 - 9.16), 5.5, 18.32, fill=False, ec=color, lw=1))
    ax.set_xlim(-4, L + 4)
    ax.set_ylim(-4, W + 4)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

## 4. Repo resmi: scorer, validator, dan ground truth publik
Diunduh pada **commit yang dikunci** (via `git`, atau arsip ZIP bila `git` tidak terpasang — umum di Windows).

In [ ]:
OFFICIAL_DIR = os.path.join(DIRS["vendor"], "mtah-official-" + CONFIG["official_repo_commit"][:7])
if not os.path.exists(os.path.join(OFFICIAL_DIR, "games.py")):
    os.makedirs(DIRS["vendor"], exist_ok=True)
    if shutil.which("git"):
        subprocess.check_call(["git", "clone", "-q", CONFIG["official_repo_url"] + ".git", OFFICIAL_DIR])
        subprocess.check_call(["git", "-C", OFFICIAL_DIR, "checkout", "-q", CONFIG["official_repo_commit"]])
    else:
        url = "{}/archive/{}.zip".format(CONFIG["official_repo_url"], CONFIG["official_repo_commit"])
        zpath = os.path.join(DIRS["vendor"], "official.zip")
        urllib.request.urlretrieve(url, zpath)
        with zipfile.ZipFile(zpath) as zf:
            top = zf.namelist()[0].split("/")[0]
            zf.extractall(DIRS["vendor"])
        shutil.move(os.path.join(DIRS["vendor"], top), OFFICIAL_DIR)
OFFICIAL_COMMIT = CONFIG["official_repo_commit"]
if os.path.isdir(os.path.join(OFFICIAL_DIR, ".git")):
    OFFICIAL_COMMIT = subprocess.run(["git", "-C", OFFICIAL_DIR, "rev-parse", "HEAD"],
                                     capture_output=True, text=True).stdout.strip()
print("Repo resmi pada commit:", OFFICIAL_COMMIT)

sys.path.insert(0, os.path.join(OFFICIAL_DIR, "evaluation"))
import possession_metrics as pm  # noqa: E402

_spec = importlib.util.spec_from_file_location("official_games", os.path.join(OFFICIAL_DIR, "games.py"))
official_games = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(official_games)

N_FRAMES = official_games.source_frames("tactical")
HALF_BLOCKS = official_games.half_video_blocks("BAR-ATH")
KICKOFF_2ND = 113742                                          # dipublikasikan di FAQ.md / DATA.md
GT_PUBLIC = pm.label_codes(pm.read_label_csv(os.path.join(OFFICIAL_DIR, "evaluation", "gt",
                                                          "BAR-ATH_possession_gt_public.csv")))
GT_PUB_LAB = np.array(pm.LABELS)[GT_PUBLIC]
VALIDATOR = os.path.join(OFFICIAL_DIR, "evaluation", "validate_submission.py")
HALF_ID = np.zeros(N_FRAMES, np.int8)
for half, first, last in HALF_BLOCKS:
    HALF_ID[first:last + 1] = half
GRADED = HALF_ID > 0
GRADED_FRAMES = np.flatnonzero(GRADED)
print("Frame total {:,} | dinilai {:,} | blok babak {}".format(N_FRAMES, int(GRADED.sum()), HALF_BLOCKS))

## 5. Memuat data
### 5.1 Metadata tim, 5.2 tracking (checkpoint `atd`), 5.3 Smart Tagging, 5.4 homography
Slot track **bukan pemain**; yang dipercaya hanya tim pemilik slot. Penyelarasan: `video_frame = frame_id - 1`.

In [ ]:
import xml.etree.ElementTree as ET


def read_team_metadata(meta_path):
    root = ET.parse(meta_path).getroot()
    teams = [(t.get("id"), (t.findtext("Name") or "").strip()) for t in root.iter("Team") if t.get("id")]
    if len(teams) != 2:
        raise ValueError("Metadata harus berisi 2 tim, ditemukan {}".format(len(teams)))
    home_id = teams[0][0]
    for score in root.iter("Score"):
        if score.get("idLocalTeam"):
            home_id = score.get("idLocalTeam")
    slots = [p.get("teamId") for p in root.iter("Player") if p.get("teamId")]
    team_of_slot = np.array([0 if t == home_id else 1 for t in slots], dtype=np.int8)
    names = {0: dict(teams)[home_id], 1: [n for i, n in teams if i != home_id][0]}
    return team_of_slot, names


TEAM_OF_SLOT, TEAM_NAMES = read_team_metadata(ASSETS["atd_meta"])
N_SLOTS = len(TEAM_OF_SLOT)
print("Tim H:", TEAM_NAMES[0], "| Tim A:", TEAM_NAMES[1], "| slot", N_SLOTS)
if "barc" not in TEAM_NAMES[0].lower():
    print("PERINGATAN: tim H bukan Barcelona menurut metadata. Periksa urutan tim!")


def _parse_atd_slow(raw_path, n_slots, kp, kb):
    rows = []
    with open(raw_path) as fh:
        for line in fh:
            parts = line.rstrip("\n").split(":")
            if len(parts) != 3:
                continue
            vals = [float(parts[0])]
            groups = parts[1].split(";")
            for g in range(n_slots):
                items = (groups[g].split(",") if g < len(groups) else []) + ["nan"] * kp
                vals += [float(v) if v not in ("", "NaN") else np.nan for v in items[:kp]]
            items = parts[2].split(",") + ["nan"] * kb
            vals += [float(v) if v not in ("", "NaN") else np.nan for v in items[:kb]]
            rows.append(vals)
    return np.array(rows, dtype=np.float64)


def parse_atd(raw_path, n_slots, n_frames, L, W, chunk=20000):
    with open(raw_path) as fh:
        first = fh.readline().rstrip("\n")
    parts = first.split(":")
    if len(parts) != 3:
        raise ValueError("Format baris ATD tak terduga")
    groups = parts[1].split(";")
    if len(groups) != n_slots:
        raise ValueError("Jumlah grup pemain {} != slot metadata {}".format(len(groups), n_slots))
    kp, kb = len(groups[0].split(",")), len(parts[2].split(","))
    ncol = 1 + n_slots * kp + kb
    blocks = []
    try:
        with open(raw_path) as fh:
            while True:
                lines = list(itertools.islice(fh, chunk))
                if not lines:
                    break
                text = "".join(lines).replace(":", ",").replace(";", ",")
                block = pd.read_csv(io.StringIO(text), header=None, dtype=np.float64,
                                    na_values=["NaN", "nan", ""]).to_numpy()
                if block.shape[1] != ncol:
                    raise ValueError("kolom {} != {}".format(block.shape[1], ncol))
                blocks.append(block)
        M = np.vstack(blocks)
    except Exception as exc:
        print("Parser cepat gagal ({}), beralih ke parser baris-per-baris...".format(exc))
        M = _parse_atd_slow(raw_path, n_slots, kp, kb)
    fid = M[:, 0].astype(np.int64)
    vf = fid - 1
    ok = (vf >= 0) & (vf < n_frames)
    P = np.full((n_frames, n_slots, 2), np.nan, np.float32)
    SPD = np.full((n_frames, n_slots), np.nan, np.float32)
    BALL = np.full((n_frames, 2), np.nan, np.float32)
    pl = M[ok, 1:1 + n_slots * kp].reshape(-1, n_slots, kp)
    P[vf[ok], :, 0], P[vf[ok], :, 1] = pl[:, :, 0] * L, pl[:, :, 1] * W
    if kp >= 3:
        SPD[vf[ok]] = pl[:, :, 2]
    else:
        SPD[1:] = np.linalg.norm(P[1:] - P[:-1], axis=2) * CONFIG["fps"]
    bl = M[ok, 1 + n_slots * kp:]
    BALL[vf[ok], 0], BALL[vf[ok], 1] = bl[:, 0] * L, bl[:, 1] * W
    info = {"n_records": int(len(fid)), "frame_id_min": int(fid.min()), "frame_id_max": int(fid.max()),
            "contiguous": bool(np.array_equal(np.sort(fid), np.arange(fid.min(), fid.min() + len(fid)))),
            "values_per_player": kp, "values_per_ball": kb}
    return {"P": P, "SPD": SPD, "BALL_RAW": BALL, "info": info}


ATD = CKPT.run("atd", lambda: parse_atd(ASSETS["atd_raw"], N_SLOTS, N_FRAMES, L_PITCH, W_PITCH),
               params={"file": os.path.basename(ASSETS["atd_raw"]), "size": os.path.getsize(ASSETS["atd_raw"]),
                       "pitch": CONFIG["pitch"]})
P, SPD, BALL_RAW = ATD["P"], ATD["SPD"], ATD["BALL_RAW"]
print(json.dumps(ATD["info"]))
assert ATD["info"]["n_records"] == N_FRAMES and ATD["info"]["frame_id_min"] == 1, "Penyelarasan frame gagal!"

In [ ]:
def team_side(text):
    t = (text or "").strip().lower()
    if not t or t in ("n/a", "na", "none"):
        return -1
    if "barc" in t:
        return 0
    if "athl" in t or "atl" in t or "bilbao" in t:
        return 1
    for side, name in TEAM_NAMES.items():
        if name and (t in name.lower() or name.lower() in t):
            return side
    return -1


def load_smart_tagging(path):
    rows = []
    for inst in ET.parse(path).getroot().iter("instance"):
        tags = {lab.findtext("group"): lab.findtext("text") for lab in inst.findall("label")}
        s0, s1 = float(inst.findtext("start")), float(inst.findtext("end"))
        rows.append({"code": (inst.findtext("code") or "").strip().upper(), "start": int(round(s0 * FPS)),
                     "end": int(round(s1 * FPS)), "team_text": tags.get("Team"), "side": team_side(tags.get("Team")),
                     "half": tags.get("Half"), "type": tags.get("Type")})
    return pd.DataFrame(rows)


ST = load_smart_tagging(ASSETS["smart_tagging"])
POSSESSION_CODES = {"BUILD UP", "PROGRESSION", "CREATING CHANCES", "BALL IN FINAL THIRD", "BALL IN THE BOX",
                    "PLAYERS IN THE BOX", "ATTACKING TRANSITION", "LONG BALL"}
DEFENDING_CODES = {"DEFENDING IN DEFENSIVE THIRD", "DEFENDING IN MIDDLE THIRD", "DEFENDING IN ATTACKING THIRD",
                   "DEFENSIVE TRANSITION"}
print("Smart Tagging:", len(ST), "instance |", ST.groupby(["team_text", "side"]).size().to_dict())

HOM_DF = pd.read_csv(ASSETS["homography"])
_hcols = ["h{}{}".format(r, c) for r in range(3) for c in range(3)]
HOM = np.full((N_FRAMES, 3, 3), np.nan)
_fr = HOM_DF["frame"].values.astype(int)
_okf = (_fr >= 0) & (_fr < N_FRAMES)
HOM[_fr[_okf]] = HOM_DF[_hcols].values[_okf].reshape(-1, 3, 3)
HOM_VALID = np.zeros(N_FRAMES, bool)
HOM_VALID[_fr[_okf]] = HOM_DF["valid"].values[_okf].astype(bool)
HOM_VALID &= np.isfinite(HOM).all(axis=(1, 2))
print("Homography valid: {:.1%}".format(HOM_VALID.mean()))

### 5.5 Uji kebenaran parser: reproduksi baseline resmi
Bila parser benar, label "pemain terdekat + voting" kita harus **identik** dengan 4 file baseline di repo resmi.

In [ ]:
def nearest_team_codes(P, team_of_slot, ball):
    out = np.full(len(ball), -1, np.int8)
    dist = np.full(len(ball), np.nan, np.float32)
    idx = np.flatnonzero(~np.isnan(ball[:, 0]))
    for s in range(0, len(idx), 50000):
        ii = idx[s:s + 50000]
        d = np.linalg.norm(P[ii] - ball[ii, None, :], axis=2)
        d = np.where(np.isnan(d), np.inf, d)
        j = np.argmin(d, axis=1)
        dmin = d[np.arange(len(ii)), j]
        ok = np.isfinite(dmin)
        out[ii[ok]] = team_of_slot[j[ok]]
        dist[ii[ok]] = dmin[ok]
    return out, dist


def baseline_smooth(codes, window):
    if window <= 0:
        return np.where(codes == 0, "H", np.where(codes == 1, "A", "D"))
    hc = np.concatenate(([0], np.cumsum(codes == 0)))
    ac = np.concatenate(([0], np.cumsum(codes == 1)))
    i = np.arange(len(codes))
    lo, hi = np.maximum(0, i - window), np.minimum(len(codes), i + window + 1)
    h, a = hc[hi] - hc[lo], ac[hi] - ac[lo]
    lab = np.full(len(codes), "D", dtype="<U1")
    seen = (h + a) > 0
    lab[seen] = np.where(h[seen] >= a[seen], "H", "A")
    return lab


RAW_NEAR_TEAM, RAW_NEAR_DIST = nearest_team_codes(P, TEAM_OF_SLOT, BALL_RAW)
BASELINES = {n: baseline_smooth(RAW_NEAR_TEAM, w) for n, w in
             [("baseline_raw", 0), ("baseline_smooth25", 25), ("baseline_smooth50", 50), ("baseline_smooth125", 125)]}
PARSER_CHECK = {}
for name, labels in BASELINES.items():
    official = pd.read_csv(os.path.join(OFFICIAL_DIR, "evaluation", "results", name + ".csv"))["label"].values
    PARSER_CHECK[name] = float(np.mean(official.astype(str) == labels))
    print("  {:<20} kecocokan dengan file resmi: {:.4%}".format(name, PARSER_CHECK[name]))
print("LULUS: parser identik dengan baseline resmi." if min(PARSER_CHECK.values()) > 0.9999 else
      "PERINGATAN: tidak identik dengan baseline resmi (wajar hanya bila memakai data sintetis).")

## 6. Eksplorasi data (EDA)

In [ ]:
def run_lengths(mask):
    m = np.concatenate(([False], mask, [False]))
    d = np.diff(m.astype(np.int8))
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1) - 1


ball_seen = ~np.isnan(BALL_RAW[:, 0])
per_min = ball_seen[:N_FRAMES // 1500 * 1500].reshape(-1, 1500).mean(axis=1)
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.fill_between(np.arange(len(per_min)), per_min * 100, color="#1f77b4", alpha=.8, step="mid")
for half, a, b in HALF_BLOCKS:
    ax.axvspan(a / 1500, b / 1500, color="#2ca02c", alpha=.08)
ax.axhline(100 * ball_seen[GRADED].mean(), color="#d62728", ls="--",
           label="rata-rata frame dinilai: {:.1%}".format(ball_seen[GRADED].mean()))
ax.set_xlabel("menit video")
ax.set_ylabel("% frame dengan bola")
ax.set_ylim(0, 100)
ax.legend(loc="lower left")
ax.set_title("Cakupan bola pada tracking (area hijau = babak dinilai)")
save_fig(fig, "01_cakupan_bola_per_menit", "Persentase frame dengan posisi bola per menit")

gs, ge = run_lengths(~ball_seen & GRADED)
gap_len = ge - gs + 1
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].hist(gap_len / FPS, bins=np.logspace(np.log10(0.04), np.log10(max(gap_len.max() / FPS, 1)), 50), color="#555")
axes[0].set_xscale("log")
axes[0].set_xlabel("durasi gap bola (s, log)")
axes[0].set_title("Distribusi durasi bola hilang")
order = np.sort(gap_len)[::-1]
axes[1].plot(np.arange(1, len(order) + 1), np.cumsum(order) / order.sum() * 100)
axes[1].set_xscale("log")
axes[1].set_xlabel("jumlah gap terpanjang (log)")
axes[1].set_ylabel("% frame hilang tercakup")
axes[1].grid(alpha=.3)
axes[1].set_title("Konsentrasi frame hilang pada gap panjang")
save_fig(fig, "02_distribusi_gap_bola", "Durasi gap bola dan konsentrasinya")
print("Gap bola (dinilai): {:,} gap | median {:.2f} s | p90 {:.1f} s | maks {:.1f} s".format(
    len(gap_len), np.median(gap_len) / FPS, np.percentile(gap_len, 90) / FPS, gap_len.max() / FPS))

n_tracked = (~np.isnan(P[:, :, 0])).sum(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [2.2, 1]})
x = np.arange(0, N_FRAMES, 250)
for k, lab in ((None, "total"), (0, "H"), (1, "A")):
    cnt = n_tracked if k is None else (~np.isnan(P[:, TEAM_OF_SLOT == k, 0])).sum(axis=1)
    axes[0].plot(x / 1500, pd.Series(cnt).rolling(250, min_periods=1).mean().values[x], lw=.8,
                 color=None if k is None else COLORS["HA"[k]], label=lab)
axes[0].set_xlabel("menit video")
axes[0].set_ylabel("objek terlacak (rata-rata 10 s)")
axes[0].legend()
axes[1].hist(n_tracked[GRADED], bins=np.arange(0, 30) - .5, color="#555")
axes[1].set_title("Rata-rata {:.1f} objek/frame".format(n_tracked[GRADED].mean()))
save_fig(fig, "03_jumlah_pemain_terlacak", "Jumlah objek pemain terlacak per frame")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
bxy = BALL_RAW[GRADED & ball_seen]
h = axes[0].hist2d(bxy[:, 0], bxy[:, 1], bins=[70, 45], range=[[-5, L_PITCH + 5], [-5, W_PITCH + 5]], cmap="magma")
draw_pitch(axes[0], color="white")
axes[0].set_title("Heatmap posisi bola (frame dinilai)")
fig.colorbar(h[3], ax=axes[0], fraction=.03)
f0 = HALF_BLOCKS[0][1]
draw_pitch(axes[1])
for k in (0, 1):
    pts = P[f0, TEAM_OF_SLOT == k]
    axes[1].scatter(pts[:, 0], pts[:, 1], s=40, color=COLORS["HA"[k]], label="{} ({})".format("HA"[k], TEAM_NAMES[k]))
if ball_seen[f0]:
    axes[1].scatter(*BALL_RAW[f0], s=60, color="black", label="bola")
axes[1].legend(loc="upper center", ncol=3, fontsize=8, bbox_to_anchor=(.5, -.02))
axes[1].set_title("Kick-off babak 1 (frame {})".format(f0))
save_fig(fig, "04_heatmap_bola_dan_kickoff", "Heatmap bola dan posisi kick-off")

codes_order = ST["code"].value_counts().index.tolist()
fig, ax = plt.subplots(figsize=(13, 5.5))
for i, code in enumerate(codes_order):
    for _, r in ST[ST["code"] == code].iterrows():
        c = COLORS["H"] if r["side"] == 0 else COLORS["A"] if r["side"] == 1 else "#888"
        ax.plot([r["start"] / 1500, r["end"] / 1500], [i, i], color=c, lw=4, solid_capstyle="butt")
ax.set_yticks(range(len(codes_order)))
ax.set_yticklabels(codes_order, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("menit video")
ax.set_title("Smart Tagging (merah = H, biru = A, abu = N/A)")
save_fig(fig, "05_timeline_smart_tagging", "Timeline Smart Tagging")

## 7. Fungsi tahap A–D (dipakai ulang untuk setiap sumber bola)

Bagian ini **hanya mendefinisikan fungsi**. Eksekusinya ada di Bagian 9 (sumber ATD) dan Bagian 12 (sumber ATD+YOLO).

**A. Pra-pemrosesan**: buang posisi terlalu jauh di luar lapangan, lonjakan, dan kedipan; haluskan (Savitzky-Golay).

**B. M1 — masked-gap self-supervised**: di frame yang bolanya terlihat, dibuat gap buatan; model memprediksi
*tim mana yang paling dekat dengan bola* dari posisi bola di ujung gap dan konfigurasi pemain. Labelnya dari data itu
sendiri, sehingga bisa divalidasi di **seluruh pertandingan** (GroupKFold per blok 2 menit). Koreksi posisi bola
(regresi LightGBM) hanya dipakai bila terbukti lebih baik dari interpolasi pada data out-of-fold.

**C. Kontak & kontrol**: Jalur A (kontak ≥ `t_dwell`) atau Jalur B (sentuhan berikutnya oleh rekan setim dalam
`pass_window`); duel → status quo.

**D. M2 — bola mati weakly-supervised**: label lemah dari Smart Tagging; fitur mengikuti Law 8/9/13–17 IFAB.

In [ ]:
def preprocess_ball(ball_raw):
    B = ball_raw.copy()
    obs = ~np.isnan(B[:, 0])
    m = CONFIG["ball_max_outside"]
    far = obs & ((B[:, 0] < -m) | (B[:, 0] > L_PITCH + m) | (B[:, 1] < -m) | (B[:, 1] > W_PITCH + m))
    B[far] = np.nan
    vmax = CONFIG["ball_max_speed"] / FPS
    for _ in range(2):
        obs = ~np.isnan(B[:, 0])
        prev_ok = np.r_[False, obs[:-1]] & obs
        next_ok = np.r_[obs[1:], False] & obs
        dprev = np.full(len(B), np.nan)
        dprev[1:] = np.linalg.norm(B[1:] - B[:-1], axis=1)
        dnext = np.r_[dprev[1:], np.nan]
        B[obs & prev_ok & next_ok & (dprev > vmax) & (dnext > vmax)] = np.nan
    obs = ~np.isnan(B[:, 0])
    for a, b in zip(*run_lengths(obs)):
        if b - a + 1 < CONFIG["ball_min_run"]:
            B[a:b + 1] = np.nan
    obs = ~np.isnan(B[:, 0])
    Bs, V = B.copy(), np.full_like(B, np.nan)
    w, o = CONFIG["savgol_window"], CONFIG["savgol_order"]
    for a, b in zip(*run_lengths(obs)):
        seg = B[a:b + 1]
        if b - a + 1 >= w:
            Bs[a:b + 1] = savgol_filter(seg, w, o, axis=0)
            V[a:b + 1] = savgol_filter(seg, w, o, deriv=1, axis=0) * FPS
        elif b - a + 1 >= 2:
            V[a:b + 1] = np.gradient(seg, axis=0) * FPS
    return {"B": Bs.astype(np.float32), "obs": obs, "speed": np.linalg.norm(V, axis=1).astype(np.float32),
            "removed": {"terlalu_jauh": int(far.sum()),
                        "total_dibuang": int((~np.isnan(ball_raw[:, 0])).sum() - obs.sum())}}


def prev_next_observed(obs, half_id):
    n = len(obs)
    idx = np.arange(n)
    prev = np.maximum.accumulate(np.where(obs, idx, -1))
    nxt = np.minimum.accumulate(np.where(obs, idx, n)[::-1])[::-1]
    prev_ok = prev >= 0
    prev_ok[prev_ok] &= half_id[prev[prev_ok]] == half_id[np.flatnonzero(prev_ok)]
    nxt_ok = nxt < n
    nxt_ok[nxt_ok] &= half_id[nxt[nxt_ok]] == half_id[np.flatnonzero(nxt_ok)]
    return prev, nxt, prev_ok, nxt_ok


def team_player_features(frames):
    feats = {}
    Pf, Sf = P[frames], SPD[frames]
    for k in (0, 1):
        pk, sk, t = Pf[:, TEAM_OF_SLOT == k], Sf[:, TEAM_OF_SLOT == k], "HA"[k]
        feats["cx_" + t], feats["cy_" + t] = np.nanmean(pk[:, :, 0], axis=1), np.nanmean(pk[:, :, 1], axis=1)
        feats["sx_" + t], feats["sy_" + t] = np.nanstd(pk[:, :, 0], axis=1), np.nanstd(pk[:, :, 1], axis=1)
        feats["n_" + t] = (~np.isnan(pk[:, :, 0])).sum(axis=1).astype(np.float32)
        feats["v_" + t] = np.nanmean(sk, axis=1)
        feats["xmin_" + t], feats["xmax_" + t] = np.nanmin(pk[:, :, 0], axis=1), np.nanmax(pk[:, :, 0], axis=1)
        j = np.argmax(np.where(np.isnan(sk), -1, sk), axis=1)
        feats["fx_" + t], feats["fy_" + t] = pk[np.arange(len(frames)), j, 0], pk[np.arange(len(frames)), j, 1]
        feats["vmax_" + t] = np.nanmax(sk, axis=1)
    hx = np.full(len(frames), np.nan, np.float32)
    hy = np.full(len(frames), np.nan, np.float32)
    for s in range(0, len(frames), 5000):
        p = Pf[s:s + 5000]
        d = np.sqrt(((p[:, :, None, :] - p[:, None, :, :]) ** 2).sum(-1))
        cnt = np.where(np.isnan(d), 0, d < 6.0).sum(-1).astype(np.float32)
        cnt[np.isnan(p[:, :, 0])] = -1
        j = np.argmax(cnt, axis=1)
        hx[s:s + 5000], hy[s:s + 5000] = p[np.arange(len(p)), j, 0], p[np.arange(len(p)), j, 1]
    feats["hot_x"], feats["hot_y"] = hx, hy
    return pd.DataFrame({k: np.asarray(v, np.float32) for k, v in feats.items()})


PF_INDEX = np.full(N_FRAMES, -1, np.int64)
PF_INDEX[GRADED_FRAMES] = np.arange(len(GRADED_FRAMES))
PF = CKPT.run("player_feats", lambda: team_player_features(GRADED_FRAMES), params={"v": 1}, deps=["atd"])


def nearest_by_team(frames, pts):
    dH = np.full(len(frames), np.nan, np.float32)
    dA = np.full(len(frames), np.nan, np.float32)
    for s in range(0, len(frames), 50000):
        f, q = frames[s:s + 50000], pts[s:s + 50000]
        d = np.linalg.norm(P[f] - q[:, None, :], axis=2)
        d = np.where(np.isnan(d), np.inf, d)
        dh, da = d[:, TEAM_OF_SLOT == 0].min(axis=1), d[:, TEAM_OF_SLOT == 1].min(axis=1)
        dH[s:s + 50000] = np.where(np.isfinite(dh), dh, np.nan)
        dA[s:s + 50000] = np.where(np.isfinite(da), da, np.nan)
    return dH, dA


def context_features(t, p, n, B, near_team, near_dist):
    t, p, n = np.asarray(t), np.asarray(p), np.asarray(n)
    bp, bn = B[p], B[n]
    frac = ((t - p) / np.maximum(n - p, 1)).astype(np.float32)
    lerp = bp + (bn - bp) * frac[:, None]
    X = pd.DataFrame({"bp_x": bp[:, 0], "bp_y": bp[:, 1], "bn_x": bn[:, 0], "bn_y": bn[:, 1],
                      "dt_p": (t - p).astype(np.float32), "dt_n": (n - t).astype(np.float32),
                      "gap_len": (n - p - 1).astype(np.float32), "frac": frac,
                      "lerp_x": lerp[:, 0], "lerp_y": lerp[:, 1],
                      "dist_pn": np.linalg.norm(bn - bp, axis=1).astype(np.float32),
                      "team_p": near_team[p].astype(np.float32), "dist_p": near_dist[p],
                      "team_n": near_team[n].astype(np.float32), "dist_n": near_dist[n]})
    for name, pts in (("lerp", lerp), ("bp", bp), ("bn", bn)):
        dH, dA = nearest_by_team(t, pts.astype(np.float32))
        X["dH_" + name], X["dA_" + name], X["dd_" + name] = dH, dA, dH - dA
    pf = PF.iloc[PF_INDEX[t]].reset_index(drop=True)
    for k in "HA":
        X["dcen_" + k] = np.hypot(pf["cx_" + k].values - lerp[:, 0], pf["cy_" + k].values - lerp[:, 1])
    return pd.concat([X, pf], axis=1).astype(np.float32), lerp


def lgb_params(balanced=False):
    p = dict(CONFIG["lgbm"], random_state=CONFIG["seed"], n_jobs=CONFIG["n_threads"], deterministic=True,
             force_row_wise=True, verbose=-1)
    if balanced:
        p["class_weight"] = "balanced"
    return p


def build_m1_data(prep):
    """Sampel gap buatan (latih) + fitur untuk gap asli (inferensi) + estimasi posisi bola."""
    rng = np.random.default_rng(CONFIG["seed"])
    B_OBS, obs = prep["B"], prep["obs"] & GRADED
    near_team, near_dist = nearest_team_codes(P, TEAM_OF_SLOT, np.where(obs[:, None], B_OBS, np.nan))
    prev, nxt, prev_ok, nxt_ok = prev_next_observed(obs, HALF_ID)
    gs, ge = run_lengths(~obs & GRADED)
    glen = ge - gs + 1
    long_gaps = glen[glen > CONFIG["short_gap_max"]]
    if len(long_gaps) == 0:
        long_gaps = np.array([CONFIG["short_gap_max"] + 1])
    train_ok = obs & (near_team >= 0)
    weak = prep.get("weak")                       # v3: frame yang posisinya berasal dari YOLO (hanya sumber 'yolo')
    if weak is not None and CONFIG["yolo_fusion"]["m1_train_on_atd_only"]:
        train_ok &= ~weak                         # target latih hanya bola ATD; YOLO tetap memperpendek gap (jangkar)
    t = np.repeat(np.flatnonzero(train_ok), CONFIG["gap_aug_per_frame"])
    Lg = rng.choice(long_gaps, size=len(t))
    off = (rng.random(len(t)) * Lg).astype(np.int64) + 1
    p0, n0 = t - off, t - off + Lg + 1
    ok = (p0 >= 0) & (n0 < N_FRAMES)
    t, p0, n0 = t[ok], p0[ok], n0[ok]
    p, n = prev[p0], nxt[n0]
    ok = (p >= 0) & (n < N_FRAMES)
    t, p, n = t[ok], p[ok], n[ok]
    ok = (HALF_ID[p] == HALF_ID[t]) & (HALF_ID[n] == HALF_ID[t])
    t, p, n = t[ok], p[ok], n[ok]
    if len(t) > CONFIG["gap_max_train"]:
        sel = np.sort(rng.choice(len(t), CONFIG["gap_max_train"], replace=False))
        t, p, n = t[sel], p[sel], n[sel]
    X, lerp = context_features(t, p, n, B_OBS, near_team, near_dist)
    y_cls = np.where(near_dist[t] <= CONFIG["gap_cls_max_dist"], near_team[t], -1).astype(np.int8)
    y_dx, y_dy = B_OBS[t, 0] - lerp[:, 0], B_OBS[t, 1] - lerp[:, 1]
    groups = t // (FPS * CONFIG["cv_block_s"])
    folds = list(GroupKFold(n_splits=CONFIG["cv_folds"]).split(np.zeros((len(t), 1)), groups=groups))

    # regresi posisi (LightGBM), dipakai hanya bila lebih baik dari interpolasi (out-of-fold)
    oof_dx, oof_dy = np.full(len(t), np.nan), np.full(len(t), np.nan)
    for tr, va in folds:
        oof_dx[va] = lgb.LGBMRegressor(**lgb_params()).fit(X.iloc[tr], y_dx[tr]).predict(X.iloc[va])
        oof_dy[va] = lgb.LGBMRegressor(**lgb_params()).fit(X.iloc[tr], y_dy[tr]).predict(X.iloc[va])
    err_model, err_lerp = np.hypot(oof_dx - y_dx, oof_dy - y_dy), np.hypot(y_dx, y_dy)
    reg_min_gap = None
    for lo in (10, 25, 50, 100, 250, 500):
        sel = X["gap_len"].values > lo
        if sel.sum() > 200 and np.median(err_model[sel]) < np.median(err_lerp[sel]):
            reg_min_gap = lo
            break
    rx = lgb.LGBMRegressor(**lgb_params()).fit(X, y_dx)
    ry = lgb.LGBMRegressor(**lgb_params()).fit(X, y_dy)

    # gap asli
    miss = np.flatnonzero(~obs & GRADED)
    glen_of = np.zeros(N_FRAMES, np.int64)
    for a, b in zip(gs, ge):
        glen_of[a:b + 1] = b - a + 1
    short, longm = miss[glen_of[miss] <= CONFIG["short_gap_max"]], miss[glen_of[miss] > CONFIG["short_gap_max"]]
    B_est = np.where(obs[:, None], B_OBS, np.nan).astype(np.float32)
    src = np.zeros(N_FRAMES, np.int8)
    src[obs] = 1
    pp, nn = prev[short], nxt[short]
    both = (pp >= 0) & (nn < N_FRAMES) & prev_ok[short] & nxt_ok[short]
    s2, pp, nn = short[both], pp[both], nn[both]
    B_est[s2] = B_OBS[pp] + (B_OBS[nn] - B_OBS[pp]) * ((s2 - pp) / (nn - pp))[:, None]
    src[s2] = 2
    pl, nl = prev[longm].copy(), nxt[longm].copy()
    pv, nv = (pl >= 0) & prev_ok[longm], (nl < N_FRAMES) & nxt_ok[longm]
    pl = np.where(pv, pl, np.where(nv, nl, -1))
    nl = np.where(nv, nl, pl)
    good = pl >= 0
    longm, pl, nl = longm[good], pl[good], nl[good]
    Xg, lerp_g = context_features(longm, pl, nl, B_OBS, near_team, near_dist) if len(longm) else (X.iloc[:0], np.zeros((0, 2)))
    if len(longm):
        use = (Xg["gap_len"].values > reg_min_gap) if reg_min_gap is not None else np.zeros(len(Xg), bool)
        B_est[longm, 0] = lerp_g[:, 0] + np.where(use, rx.predict(Xg), 0)
        B_est[longm, 1] = lerp_g[:, 1] + np.where(use, ry.predict(Xg), 0)
        src[longm] = 3
    return {"X": X, "y": y_cls, "gap_len": X["gap_len"].values, "folds": folds, "Xg": Xg, "longm": longm,
            "base_lerp": np.where(X["dd_lerp"].values <= 0, 0, 1), "base_prev": X["team_p"].values,
            "err_model": err_model, "err_lerp": err_lerp, "reg_min_gap": reg_min_gap,
            "B_est": B_est, "src": src, "near_team": near_team, "near_dist": near_dist,
            "n_short_filled": int(len(s2))}


def build_episodes(prep, m1d):
    B_EST, SRC = m1d["B_est"], m1d["src"]
    reliable = (SRC == 1) | (SRC == 2)
    fr = np.flatnonzero(reliable)
    dH = np.full(N_FRAMES, np.nan, np.float32)
    dA = np.full(N_FRAMES, np.nan, np.float32)
    dH[fr], dA[fr] = nearest_by_team(fr, B_EST[fr])
    code = np.full(N_FRAMES, -1, np.int8)
    dmin = np.fmin(dH, dA)
    m = reliable & (dmin <= CONFIG["r_pz"])
    code[m] = np.where(dH[m] <= dA[m], 0, 1)
    code[m & (dH <= CONFIG["r_dz"]) & (dA <= CONFIG["r_dz"])] = 2
    eps = []
    for half, a, b in HALF_BLOCKS:
        seg = code[a:b + 1]
        ch = np.flatnonzero(np.diff(seg) != 0) + 1
        for s, e in zip(np.r_[0, ch], np.r_[ch - 1, len(seg) - 1]):
            c = seg[s]
            if c < 0:
                continue
            if eps and eps[-1][3] == half and eps[-1][2] == c and (a + s) - eps[-1][1] - 1 <= CONFIG["contact_merge_gap"]:
                eps[-1][1] = a + e
            else:
                eps.append([a + s, a + e, int(c), half])
    E = pd.DataFrame(eps, columns=["start", "end", "code", "half"])
    E["dur"] = E["end"] - E["start"] + 1
    E["dwell"] = (E["code"] < 2) & (E["dur"] >= CONFIG["t_dwell"])
    weak = prep.get("weak")                       # v3: hanya sumber 'yolo'; sumber ATD tidak berubah
    if weak is not None and len(E):
        cw = np.r_[0, np.cumsum(weak.astype(np.int64))]
        E["weak_frac"] = (cw[E["end"].values + 1] - cw[E["start"].values]) / E["dur"].values
        mostly = E["weak_frac"].values > 0.5
        E["dwell"] = (E["code"] < 2) & np.where(mostly, E["dur"] >= CONFIG["yolo_fusion"]["t_dwell_weak"],
                                                E["dur"] >= CONFIG["t_dwell"])
    ti = np.flatnonzero(E["code"].values < 2)
    chain = np.zeros(len(E), bool)
    st, en, cd, hf = E["start"].values, E["end"].values, E["code"].values, E["half"].values
    for i, j in zip(ti[:-1], ti[1:]):
        if hf[i] == hf[j] and st[j] - en[i] <= CONFIG["pass_window"] and cd[i] == cd[j]:
            chain[i] = True
    E["chain"] = chain
    E["confirmed"] = E["dwell"] | E["chain"]
    return {"E": E, "dH": dH, "dA": dA}


def rolling(x, w, fn="mean"):
    r = pd.Series(x).rolling(w, min_periods=1, center=True)
    return (r.mean() if fn == "mean" else r.median() if fn == "median" else r.max()).values.astype(np.float32)


def attack_direction():
    out = {}
    for half, a, b in HALF_BLOCKS:
        cx = {k: np.nanmean(P[a:b + 1, TEAM_OF_SLOT == k, 0]) for k in (0, 1)}
        out[(half, 0)] = 1 if cx[0] < cx[1] else -1
        out[(half, 1)] = -out[(half, 0)]
    return out


def shift(x, k):
    """k > 0: nilai masa depan (lead); k < 0: nilai masa lalu (lag)."""
    if k > 0:
        return np.r_[x[k:], np.full(k, np.nan, np.float32)]
    return np.r_[np.full(-k, np.nan, np.float32), x[:k]]


def build_dead_features(prep, m1d, epi):
    lead = max(1, int(round(CONFIG["dead_lead_s"] * FPS)))
    B_OBS, OBS, B_SPEED = prep["B"], prep["obs"], prep["speed"]
    B_EST, SRC = m1d["B_est"], m1d["src"]
    sp = SPD.copy()
    sp[(sp > 12) | (sp < 0)] = np.nan
    team_v = np.nanmean(sp, axis=1).astype(np.float32)
    frac_run = np.nanmean(np.where(np.isnan(sp), np.nan, sp > 4.0), axis=1).astype(np.float32)
    F = pd.DataFrame(index=np.arange(N_FRAMES))
    for w in (25, 75):
        r = rolling(team_v, w)
        F["v_team_{}".format(w)], F["v_team_{}_lead".format(w)], F["v_team_{}_lag".format(w)] = r, shift(r, lead), shift(r, -lead)
    F["frac_run_25"] = rolling(frac_run, 25)
    F["frac_run_25_lead"] = shift(F["frac_run_25"].values, lead)
    F["spread_x"], F["spread_y"] = np.nanstd(P[:, :, 0], axis=1), np.nanstd(P[:, :, 1], axis=1)
    F["ball_obs_25"], F["ball_obs_125"] = rolling(OBS.astype(np.float32), 25), rolling(OBS.astype(np.float32), 125)
    prev, nxt, _, _ = prev_next_observed(OBS, HALF_ID)
    idx = np.arange(N_FRAMES)
    F["since_seen"] = np.clip(np.where(prev >= 0, idx - prev, 9999), 0, 1500).astype(np.float32)
    F["until_seen"] = np.clip(np.where(nxt < N_FRAMES, nxt - idx, 9999), 0, 1500).astype(np.float32)
    bx, by = B_EST[:, 0], B_EST[:, 1]
    rel = (SRC == 1) | (SRC == 2)
    F["ball_out_dist"] = np.where(rel, np.maximum.reduce([-bx, bx - L_PITCH, -by, by - W_PITCH]), np.nan)
    pc = np.clip(prev, 0, None)
    lastx, lasty = np.where(prev >= 0, B_OBS[pc, 0], np.nan), np.where(prev >= 0, B_OBS[pc, 1], np.nan)
    F["last_out_dist"] = np.maximum.reduce([-lastx, lastx - L_PITCH, -lasty, lasty - W_PITCH])
    F["last_line_dist"] = np.minimum.reduce([np.abs(lastx), np.abs(L_PITCH - lastx), np.abs(lasty), np.abs(W_PITCH - lasty)])
    F["last_ball_speed"] = np.where(prev >= 0, B_SPEED[pc], np.nan)
    bs = np.where(rel, B_SPEED, np.nan)
    F["ball_speed"], F["ball_speed_med25"] = bs, rolling(bs, 25, "median")
    F["ball_speed_max25_lead"] = shift(rolling(bs, 25, "max"), lead)
    F["ball_cx_dist"] = np.hypot(bx - L_PITCH / 2, by - W_PITCH / 2)
    F["near_dist"] = np.fmin(epi["dH"], epi["dA"])
    px, py = P[:, :, 0], P[:, :, 1]
    F["n_out_sideline"] = ((py < -0.3) | (py > W_PITCH + 0.3)).sum(axis=1).astype(np.float32)
    F["n_out_goalline"] = ((px < -0.3) | (px > L_PITCH + 0.3)).sum(axis=1).astype(np.float32)
    corners = [(0, 0), (0, W_PITCH), (L_PITCH, 0), (L_PITCH, W_PITCH)]
    F["min_corner_dist"] = np.nanmin(np.stack([np.nanmin(np.hypot(px - cx, py - cy), axis=1) for cx, cy in corners]), axis=0)
    F["min_center_dist"] = np.nanmin(np.hypot(px - L_PITCH / 2, py - W_PITCH / 2), axis=1)
    AD = attack_direction()
    own = np.full(N_FRAMES, np.nan, np.float32)
    for half, a, b in HALF_BLOCKS:
        tot, cnt = 0, 0
        for k in (0, 1):
            xs = px[a:b + 1, TEAM_OF_SLOT == k]
            inown = (xs < L_PITCH / 2 + 1) if AD[(half, k)] > 0 else (xs > L_PITCH / 2 - 1)
            tot = tot + np.where(np.isnan(xs), 0, inown).sum(axis=1)
            cnt = cnt + (~np.isnan(xs)).sum(axis=1)
        own[a:b + 1] = tot / np.maximum(cnt, 1)
    F["frac_own_half"], F["frac_own_half_75"] = own, rolling(own, 75)
    for c in ("n_out_sideline", "n_out_goalline"):
        F[c + "_75"] = rolling(F[c].values, 75)
    if CONFIG["m2_version"] >= 2:
        # --- v2: bola diam (bola diletakkan sebelum restart) ---
        static = np.where(rel, (B_SPEED < 0.5).astype(np.float32), np.nan)
        F["ball_static_25"], F["ball_static_75"] = rolling(static, 25), rolling(static, 75)
        F["ball_static_75_lag"] = shift(F["ball_static_75"].values, -lead)
        # --- v2: waktu sejak bola terakhir terlihat di luar lapangan ---
        out_now = np.nan_to_num(F["ball_out_dist"].values, nan=-1) > 0.3
        last_out = np.maximum.accumulate(np.where(out_now, idx, -1))
        F["since_out"] = np.clip(np.where(last_out >= 0, idx - last_out, 9999), 0, 1500).astype(np.float32)
        # --- v2: jarak posisi bola terakhir ke titik restart (Law 8, 13-17) ---
        corner_d = np.min(np.stack([np.hypot(lastx - cx, lasty - cy) for cx, cy in corners]), axis=0)
        F["last_corner_dist"] = corner_d
        F["last_center_dist"] = np.hypot(lastx - L_PITCH / 2, lasty - W_PITCH / 2)
        F["last_touchline_dist"] = np.minimum(np.abs(lasty), np.abs(W_PITCH - lasty))
        ga_dx = np.minimum(np.abs(lastx), np.abs(L_PITCH - lastx))
        F["last_goalarea_dist"] = np.hypot(np.maximum(ga_dx - 5.5, 0), np.maximum(np.abs(lasty - W_PITCH / 2) - 9.16, 0))
        # --- v2: pemain berjalan/berdiri & pemain tercepat ---
        walk = np.nanmean(np.where(np.isnan(sp), np.nan, sp < 1.5), axis=1).astype(np.float32)
        F["frac_walk_25"], F["frac_walk_75"] = rolling(walk, 25), rolling(walk, 75)
        F["frac_walk_75_lead"] = shift(F["frac_walk_75"].values, lead)
        vmax = np.nanmax(sp, axis=1).astype(np.float32)
        F["v_max_25"], F["v_max_75"] = rolling(vmax, 25, "median"), rolling(vmax, 75, "median")
        F["near_dist_75"] = rolling(F["near_dist"].values, 75, "median")
    return F.astype(np.float32)


def weak_dead_labels():
    y = np.full(N_FRAMES, -1, np.int8)
    m = int(round(CONFIG["alive_weak_margin_s"] * FPS))
    for _, r in ST[ST["code"].isin(POSSESSION_CODES | DEFENDING_CODES)].iterrows():
        if r["end"] - r["start"] > 2 * m + 25:
            y[r["start"] + m:r["end"] - m] = 0
    sp = ST[ST["code"] == "SET PIECES"]
    blk = np.zeros(N_FRAMES, bool)
    for _, r in sp.iterrows():
        blk[max(r["start"] - 3 * FPS, 0):r["end"] + 3 * FPS] = True
    y[blk & (y == 0)] = -1
    ln = int(round(CONFIG["dead_weak_len_s"] * FPS))
    for _, r in sp.iterrows():
        y[r["start"]:r["start"] + ln] = 1
    y[HALF_BLOCKS[1][1]:KICKOFF_2ND - 12] = 1
    if CONFIG["m2_version"] >= 2:
        # v2: setelah gol, bola mati sampai kick-off berikutnya (tag 'Kick Off (After Goal)')
        kos = sp[sp["type"].fillna("").str.lower().str.contains("kick off")]
        cap = int(CONFIG["goal_dead_cap_s"] * FPS)
        for _, g in ST[ST["code"] == "GOALS"].iterrows():
            nxt = kos[kos["start"] > g["end"]]
            if len(nxt) == 0:
                continue
            ko = int(nxt["start"].iloc[0]) + ln          # tag kick-off membuka ~3,5 s sebelum bola ditendang
            a0 = int(g["end"]) + FPS
            if 0 < ko - a0 <= cap:
                y[a0:ko] = 1
    y[~GRADED] = -1
    return y


def build_m2_data(prep, m1d, epi):
    F = build_dead_features(prep, m1d, epi)
    y = weak_dead_labels()
    lab = np.flatnonzero(y >= 0)
    blocks = np.arange(N_FRAMES) // (FPS * CONFIG["cv_block_s"])
    K = CONFIG["cv_folds"]
    fold_of_block = {}
    for k, (_, va) in enumerate(GroupKFold(n_splits=K).split(np.zeros((len(lab), 1)), groups=blocks[lab])):
        for b in np.unique(blocks[lab[va]]):
            fold_of_block[b] = k
    fold = np.array([fold_of_block.get(b, b % K) for b in blocks])
    return {"F": F, "y": y, "lab": lab, "fold": fold}

## 8. Model zoo dan ensemble (stacking)

| Kode | Model | Penanganan NaN | Catatan |
|---|---|---|---|
| `lgbm` | LightGBM | bawaan | `deterministic=True` |
| `xgb` | XGBoost (`hist`) | bawaan | GPU opsional via `gbdt_device` |
| `cat` | CatBoost | bawaan | GPU opsional via `gbdt_device` |
| `rf` | Random Forest | imputasi median | `max_samples=0.5` agar cepat |
| `ensemble` | Stacking / rata-rata | — | regresi logistik di atas **logit** probabilitas OOF, atau rata-rata logit; dipilih otomatis via log-loss OOF |

**Ensemble digabung di level probabilitas, sebelum Viterbi.** Menggabungkan label akhir H/A/D (voting) akan
memunculkan kedipan dan merusak `spellF1`. Metrik ensemble dihitung *cross-fitted* (meta-model dilatih ulang per fold),
sehingga tidak optimistis.

In [ ]:
FAMILY_NAMES = {"lgbm": "LightGBM", "xgb": "XGBoost", "cat": "CatBoost", "rf": "RandomForest", "ensemble": "Ensemble"}


def make_classifier(family, balanced, n_pos=1, n_neg=1):
    seed, nt = CONFIG["seed"], CONFIG["n_threads"]
    if family == "lgbm":
        return lgb.LGBMClassifier(**lgb_params(balanced))
    if family == "xgb":
        p = dict(CONFIG["xgb"], random_state=seed, n_jobs=nt, tree_method="hist", device=CONFIG["gbdt_device"],
                 eval_metric="logloss", verbosity=0)
        if balanced:
            p["scale_pos_weight"] = float(n_neg) / max(n_pos, 1)
        return xgb.XGBClassifier(**p)
    if family == "cat":
        p = dict(CONFIG["cat"], random_seed=seed, thread_count=nt, verbose=0, allow_writing_files=False,
                 task_type="GPU" if CONFIG["gbdt_device"] == "cuda" else "CPU")
        if balanced:
            p["auto_class_weights"] = "Balanced"
        return CatBoostClassifier(**p)
    if family == "rf":
        return make_pipeline(SimpleImputer(strategy="median"),
                             RandomForestClassifier(**CONFIG["rf"], random_state=seed, n_jobs=nt,
                                                    class_weight="balanced_subsample" if balanced else None))
    raise ValueError(family)


def proba_of(model, X, cls):
    classes = list(model.classes_)
    return model.predict_proba(X)[:, classes.index(cls)]


def feature_importance(model, columns):
    try:
        est = model[-1] if hasattr(model, "steps") else model
        if isinstance(est, CatBoostClassifier):
            vals = est.get_feature_importance()
        else:
            vals = est.feature_importances_
        if len(vals) == len(columns):
            return pd.Series(vals, index=columns).sort_values(ascending=False)
    except Exception:
        pass
    return pd.Series(dtype=float)


def binary_metrics(y01, p):
    m = np.isfinite(p)
    y01, p = y01[m], np.clip(p[m], 1e-4, 1 - 1e-4)
    if len(np.unique(y01)) < 2:
        return {"acc": float("nan"), "auc": float("nan"), "logloss": float("nan"), "n": int(len(y01))}
    return {"acc": float(np.mean((p >= .5) == y01)), "auc": float(roc_auc_score(y01, p)),
            "logloss": float(log_loss(y01, p)), "n": int(len(y01))}


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def fit_m1_family(m1d, family):
    X, y, folds = m1d["X"], m1d["y"], m1d["folds"]
    cm = y >= 0
    oof = np.full(len(y), np.nan)
    for tr, va in folds:
        trc, vac = tr[cm[tr]], va[cm[va]]
        oof[vac] = proba_of(make_classifier(family, False).fit(X.iloc[trc], y[trc]), X.iloc[vac], 0)
    model = make_classifier(family, False).fit(X[cm], y[cm])
    p_inf = proba_of(model, m1d["Xg"], 0) if len(m1d["Xg"]) else np.zeros(0)
    return {"oof": oof, "p_inf": p_inf, "metrics": binary_metrics((y[cm] == 0).astype(int), oof[cm]),
            "importance": feature_importance(model, X.columns)}


def fit_m2_family(m2d, family):
    F, y, lab, fold = m2d["F"], m2d["y"], m2d["lab"], m2d["fold"]
    p_dead = np.full(N_FRAMES, np.nan, np.float32)
    imp = None
    for k in range(CONFIG["cv_folds"]):
        tr = lab[fold[lab] != k]
        n_pos, n_neg = int((y[tr] == 1).sum()), int((y[tr] == 0).sum())
        model = make_classifier(family, True, n_pos, n_neg).fit(F.iloc[tr], y[tr])
        target = GRADED_FRAMES[fold[GRADED_FRAMES] == k]
        p_dead[target] = proba_of(model, F.iloc[target], 1)
        if imp is None:
            imp = feature_importance(model, F.columns)
    return {"p_dead": p_dead, "metrics": binary_metrics((y[lab] == 1).astype(int), p_dead[lab]), "importance": imp}


def choose_ensemble(y01, oof_stack, oof_mean):
    """Pilih stacking vs rata-rata logit berdasarkan log-loss out-of-fold (atau paksa lewat CONFIG)."""
    method = CONFIG["ensemble_method"]
    ms, mm = binary_metrics(y01, oof_stack), binary_metrics(y01, oof_mean)
    if method == "auto":
        method = "stack" if ms["logloss"] <= mm["logloss"] else "mean"
    return method, (ms if method == "stack" else mm), {"stack": ms, "mean": mm}


def stack_m1(m1d, members):
    y, folds = m1d["y"], m1d["folds"]
    cm = (y >= 0) & np.all([np.isfinite(members[f]["oof"]) for f in members], axis=0)
    Z = np.column_stack([logit(members[f]["oof"]) for f in members])
    yb = (y == 0).astype(int)
    oof = np.full(len(y), np.nan)
    for tr, va in folds:
        trc, vac = tr[cm[tr]], va[cm[va]]
        oof[vac] = LogisticRegression(C=1.0, max_iter=1000).fit(Z[trc], yb[trc]).predict_proba(Z[vac])[:, 1]
    oof_mean = 1 / (1 + np.exp(-Z.mean(axis=1)))
    method, metrics, both = choose_ensemble(yb[cm], oof[cm], oof_mean[cm])
    meta = LogisticRegression(C=1.0, max_iter=1000).fit(Z[cm], yb[cm])
    Zi = np.column_stack([logit(members[f]["p_inf"]) for f in members])
    if len(Zi) == 0:
        p_inf = np.zeros(0)
    elif method == "stack":
        p_inf = meta.predict_proba(Zi)[:, 1]
    else:
        p_inf = 1 / (1 + np.exp(-Zi.mean(axis=1)))
    weights = {f: round(float(c_), 4) for f, c_ in zip(members, meta.coef_[0])} if method == "stack" else {f: round(1 / len(members), 4) for f in members}
    return {"oof": oof if method == "stack" else np.where(cm, oof_mean, np.nan), "p_inf": p_inf, "metrics": metrics,
            "method": method, "candidates": both, "weights": weights, "importance": pd.Series(dtype=float)}


def stack_m2(m2d, members):
    y, lab, fold = m2d["y"], m2d["lab"], m2d["fold"]
    Z = np.column_stack([logit(np.nan_to_num(members[f]["p_dead"], nan=.5)) for f in members])
    yb = (y == 1).astype(int)
    oof_lab = np.full(len(lab), np.nan)
    for k in range(CONFIG["cv_folds"]):
        tr, va = fold[lab] != k, fold[lab] == k
        meta_k = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced").fit(Z[lab[tr]], yb[lab[tr]])
        oof_lab[va] = meta_k.predict_proba(Z[lab[va]])[:, 1]
    mean_all = 1 / (1 + np.exp(-Z.mean(axis=1)))
    method, metrics, both = choose_ensemble(yb[lab], oof_lab, mean_all[lab])
    meta = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced").fit(Z[lab], yb[lab])
    p_dead = np.full(N_FRAMES, np.nan, np.float32)
    p_dead[GRADED_FRAMES] = (meta.predict_proba(Z[GRADED_FRAMES])[:, 1] if method == "stack" else mean_all[GRADED_FRAMES])
    weights = {f: round(float(c_), 4) for f, c_ in zip(members, meta.coef_[0])} if method == "stack" else {f: round(1 / len(members), 4) for f in members}
    return {"p_dead": p_dead, "metrics": metrics, "method": method, "candidates": both, "weights": weights,
            "importance": pd.Series(dtype=float)}

## 9. Menjalankan pipeline per sumber bola

`run_source()` menjalankan tahap A–D untuk satu sumber bola dan **semua model** yang dibutuhkan skenario,
masing-masing dengan checkpoint sendiri (`m1_<model>_<sumber>`, `m2_<model>_<sumber>`).
Di sini dijalankan untuk sumber **ATD** (S1–S5). Sumber **YOLO** dijalankan di Bagian 12.

In [ ]:
def needed_families(source):
    fams = set()
    for spec in CONFIG["scenarios"].values():
        if spec["source"] == source:
            fams |= set(CONFIG["ensemble_members"]) if spec["model"] == "ensemble" else {spec["model"]}
    order = ["lgbm", "xgb", "cat", "rf"]
    return [f for f in order if f in fams] + sorted(fams - set(order))


def needs_ensemble(source):
    return any(s["source"] == source and s["model"] == "ensemble" for s in CONFIG["scenarios"].values())


PARAMS_PREP = {k: CONFIG[k] for k in ("ball_max_outside", "ball_max_speed", "ball_min_run", "savgol_window", "savgol_order")}
PARAMS_M1 = {k: CONFIG[k] for k in ("short_gap_max", "gap_aug_per_frame", "gap_max_train", "gap_cls_max_dist",
                                    "cv_folds", "cv_block_s", "lgbm", "seed", "n_threads")}
PARAMS_EPI = {k: CONFIG[k] for k in ("r_pz", "r_dz", "contact_merge_gap", "t_dwell", "pass_window")}
PARAMS_M2 = {k: CONFIG[k] for k in ("dead_lead_s", "dead_weak_len_s", "alive_weak_margin_s", "cv_folds", "cv_block_s",
                                    "m2_version", "goal_dead_cap_s")}


def family_params(f):
    base = {"seed": CONFIG["seed"], "n_threads": CONFIG["n_threads"], "gbdt_device": CONFIG["gbdt_device"]}
    if f in CONFIG:
        base["hp"] = CONFIG[f]
    return base


SOURCES = {}


def run_source(src, ball_raw, deps, extra_params=None, weak_mask=None):
    """weak_mask (v3): frame yang posisinya berasal dari YOLO -> bukti lebih lemah di tahap C dan tidak dipakai
    sebagai target latih M1. Parameter fusi harus ikut di extra_params agar kunci checkpoint berubah."""
    t0 = time.time()
    R = {"name": src}

    def _prep():
        p = preprocess_ball(ball_raw)
        if weak_mask is not None:
            p["weak"] = np.asarray(weak_mask, bool) & p["obs"]
        return p
    R["prep"] = CKPT.run("prep_" + src, _prep, dict(PARAMS_PREP, **(extra_params or {})), deps)
    R["m1d"] = CKPT.run("m1data_" + src, lambda: build_m1_data(R["prep"]), PARAMS_M1, ["prep_" + src, "player_feats"])
    R["epi"] = CKPT.run("episodes_" + src, lambda: build_episodes(R["prep"], R["m1d"]), PARAMS_EPI, ["m1data_" + src])
    R["m2d"] = CKPT.run("m2data_" + src, lambda: build_m2_data(R["prep"], R["m1d"], R["epi"]), PARAMS_M2,
                        ["episodes_" + src])
    R["m1"], R["m2"] = {}, {}
    for f in needed_families(src):
        print("--- model {} ({}) ---".format(FAMILY_NAMES.get(f, f), src))
        R["m1"][f] = CKPT.run("m1_{}_{}".format(f, src), lambda: fit_m1_family(R["m1d"], f), family_params(f),
                              ["m1data_" + src])
        R["m2"][f] = CKPT.run("m2_{}_{}".format(f, src), lambda: fit_m2_family(R["m2d"], f), family_params(f),
                              ["m2data_" + src])
    if needs_ensemble(src):
        mem = [f for f in CONFIG["ensemble_members"] if f in R["m1"]]
        R["m1"]["ensemble"] = CKPT.run("m1_ens_" + src, lambda: stack_m1(R["m1d"], {f: R["m1"][f] for f in mem}),
                                       {"members": mem, "method": CONFIG["ensemble_method"]}, ["m1_{}_{}".format(f, src) for f in mem])
        R["m2"]["ensemble"] = CKPT.run("m2_ens_" + src, lambda: stack_m2(R["m2d"], {f: R["m2"][f] for f in mem}),
                                       {"members": mem, "method": CONFIG["ensemble_method"]}, ["m2_{}_{}".format(f, src) for f in mem])
    R["L_episode"] = episode_evidence(R["epi"]["E"])
    print("Sumber '{}' selesai dalam {:.1f} menit".format(src, (time.time() - t0) / 60))
    return R


def episode_evidence(E):
    L = np.zeros(N_FRAMES, np.float32)
    lc, lu = logit(CONFIG["p_control"]), logit(CONFIG["p_contact"])
    wf = E["weak_frac"].values if "weak_frac" in E.columns else np.zeros(len(E))
    ew = CONFIG["yolo_fusion"]["evidence_weight"]
    for (s, e, c, conf, dur), fw in zip(E[["start", "end", "code", "confirmed", "dur"]].itertuples(index=False), wf):
        if c > 1:
            continue
        val = (lc if conf else lu) * max(1.0, CONFIG["min_mass_frames"] / dur) * (1.0 - (1.0 - ew) * fw)
        L[s:e + 1] += val if c == 0 else -val
    return L


SOURCES["atd"] = run_source("atd", BALL_RAW, deps=["atd"])

### 9.1 Diagnostik model (sumber ATD)
Semua angka **out-of-fold pada seluruh pertandingan** (bukan 5 menit publik), jadi layak dipakai membandingkan model.
- **M1**: akurasi/AUC/logloss menebak tim terdekat ke bola di dalam gap buatan. Pembanding sederhana:
  tim terdekat ke titik interpolasi, dan tim yang terakhir terlihat.
- **M2**: AUC/logloss terhadap **label lemah** Smart Tagging (indikator, bukan kebenaran mutlak).

In [ ]:
def model_diag_table(R):
    rows = {}
    for f, r in R["m1"].items():
        rows[(FAMILY_NAMES.get(f, f), "M1")] = r["metrics"]
    for f, r in R["m2"].items():
        rows[(FAMILY_NAMES.get(f, f), "M2")] = r["metrics"]
    return pd.DataFrame(rows).T


def plot_model_diagnostics(R, tag):
    m1d = R["m1d"]
    bins = [10, 25, 50, 100, 250, 500, 10 ** 7]
    names = ["0.4-1s", "1-2s", "2-4s", "4-10s", "10-20s", ">20s"]
    b = pd.cut(m1d["gap_len"], bins=bins, labels=names)
    cm = m1d["y"] >= 0
    yh = m1d["y"] == 0
    acc = {}
    for f, r in R["m1"].items():
        ok = cm & np.isfinite(r["oof"])
        acc[FAMILY_NAMES.get(f, f)] = pd.Series((r["oof"][ok] >= .5) == yh[ok]).groupby(b[ok], observed=False).mean()
    ok = cm
    acc["tim terdekat ke titik interpolasi"] = pd.Series(m1d["base_lerp"][ok] == m1d["y"][ok]).groupby(b[ok], observed=False).mean()
    acc["tim terakhir terlihat"] = pd.Series((m1d["base_prev"][ok] == 0) == yh[ok]).groupby(b[ok], observed=False).mean()
    acc = pd.DataFrame(acc)
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    acc.plot(kind="bar", ax=axes[0], rot=0, width=.85)
    axes[0].set_ylim(.4, 1)
    axes[0].set_ylabel("akurasi out-of-fold")
    axes[0].set_xlabel("panjang gap buatan")
    axes[0].set_title("M1 per model ({})".format(tag))
    axes[0].legend(fontsize=7)
    pos = pd.DataFrame({"regresi M1": m1d["err_model"], "interpolasi": m1d["err_lerp"]}).groupby(b, observed=False).median()
    pos.plot(kind="bar", ax=axes[1], rot=0)
    axes[1].set_ylabel("median galat posisi (m)")
    axes[1].set_title("Galat posisi bola (regresi dipakai bila gap > {} frame)".format(m1d["reg_min_gap"]))
    m2 = pd.DataFrame({FAMILY_NAMES.get(f, f): [r["metrics"]["auc"]] for f, r in R["m2"].items()}).T[0]
    m2.plot(kind="barh", ax=axes[2], color="#555")
    lo = np.nanmin(m2.values) if len(m2) else 0.5
    axes[2].set_xlim(max(0.5, lo - 0.02), 1.0)
    axes[2].set_title("M2: AUC out-of-fold vs label lemah")
    save_fig(fig, "06_diagnostik_model_{}".format(tag), "Diagnostik M1/M2 per model untuk sumber {}".format(tag))
    return acc


DIAG_ATD = model_diag_table(SOURCES["atd"])
print(DIAG_ATD.round(4).to_string())
save_table(DIAG_ATD, "diagnostik_model_atd")
ACC_ATD = plot_model_diagnostics(SOURCES["atd"], "atd")
for f in ("ensemble",):
    if f in SOURCES["atd"]["m1"]:
        for _t in ("m1", "m2"):
            _e = SOURCES["atd"][_t][f]
            print("Ensemble {}: metode terpilih = {} | bobot {} | logloss stack {:.4f} vs mean {:.4f}".format(
                _t.upper(), _e["method"], _e["weights"], _e["candidates"]["stack"]["logloss"], _e["candidates"]["mean"]["logloss"]))

## 10. Decoding Viterbi 3 keadaan (H, A, D)

Per frame: **L_tim** = bukti episode kontrol (Tahap C) + `gap_weight` × logit(M1) di dalam gap;
**L_mati** = `dead_temp` × logit(M2). Penalti transisi `lam_switch`, `lam_to_dead`, `lam_from_dead`.
Viterbi melihat seluruh babak, sehingga batas possession jatuh di sentuhan pertama tim baru. Frame 110.808–113.741
dipaksa `D` (fakta yang dipublikasikan). **Parameter decoding identik untuk semua skenario.**

In [ ]:
def log_sigmoid(x):
    return -np.logaddexp(0, -x)


def viterbi_half(lpH, lpA, lpD, lam_s, lam_td, lam_fd):
    H, A, D = lpH.tolist(), lpA.tolist(), lpD.tolist()
    n = len(H)
    bH, bA, bD = bytearray(n), bytearray(n), bytearray(n)
    sH, sA, sD = H[0], A[0], D[0]
    for t in range(1, n):
        x0, x1, x2 = sH, sA - lam_s, sD - lam_fd
        if x0 >= x1 and x0 >= x2:
            nH = x0
        elif x1 >= x2:
            nH, bH[t] = x1, 1
        else:
            nH, bH[t] = x2, 2
        x0, x1, x2 = sH - lam_s, sA, sD - lam_fd
        if x1 >= x0 and x1 >= x2:
            nA, bA[t] = x1, 1
        elif x0 >= x2:
            nA = x0
        else:
            nA, bA[t] = x2, 2
        x0, x1, x2 = sH - lam_td, sA - lam_td, sD
        if x2 >= x0 and x2 >= x1:
            nD, bD[t] = x2, 2
        elif x0 >= x1:
            nD = x0
        else:
            nD, bD[t] = x1, 1
        sH, sA, sD = nH + H[t], nA + A[t], nD + D[t]
    back = (bH, bA, bD)
    path = np.zeros(n, np.int8)
    state = 0 if (sH >= sA and sH >= sD) else (1 if sA >= sD else 2)
    path[-1] = state
    for t in range(n - 1, 0, -1):
        state = back[state][t]
        path[t - 1] = state
    return path


def viterbi_half_mindead(lpH, lpA, lpD, lam_s, lam_td, lam_fd, m):
    """Viterbi H/A/D dengan durasi minimum bola mati m frame (semi-Markov ringan).

    Keadaan D dipecah menjadi rantai D1..D(m-1) (wajib maju) dan Dm (boleh bertahan). Masuk D hanya lewat D1,
    keluar hanya dari Dm, sehingga setiap periode bola mati yang dimulai di dalam babak panjangnya >= m frame.
    Rantai disimpan sebagai antrean dengan skor relatif terhadap emisi kumulatif -> O(1) per frame."""
    if m <= 1:
        return viterbi_half(lpH, lpA, lpD, lam_s, lam_td, lam_fd)
    from collections import deque
    H, A, D = lpH.tolist(), lpA.tolist(), lpD.tolist()
    n = len(H)
    bH, bA, bD1, bDm = bytearray(n), bytearray(n), bytearray(n), bytearray(n)
    sH, sA, sDm = H[0], A[0], D[0]            # awal babak boleh langsung bola mati (mis. sebelum kick-off)
    C = D[0]
    chain = deque()                            # (skor - C saat masuk, frame masuk) untuk posisi 1..m-1
    for t in range(1, n):
        e = D[t]
        C_prev, C = C, C + e
        best, bdm = sDm, 1                     # Dm -> Dm
        if chain and chain[0][1] == t - m + 1:  # elemen di posisi m-1 maju ke Dm
            w, _ = chain.popleft()
            if w + C_prev > best:
                best, bdm = w + C_prev, 0
        nDm = best + e
        bDm[t] = bdm
        x0, x1, x2 = sH, sA - lam_s, sDm - lam_fd
        if x0 >= x1 and x0 >= x2:
            nH = x0
        elif x1 >= x2:
            nH, bH[t] = x1, 1
        else:
            nH, bH[t] = x2, 2
        x0, x1, x2 = sH - lam_s, sA, sDm - lam_fd
        if x1 >= x0 and x1 >= x2:
            nA, bA[t] = x1, 1
        elif x0 >= x2:
            nA = x0
        else:
            nA, bA[t] = x2, 2
        if sH >= sA:
            v = sH - lam_td
        else:
            v, bD1[t] = sA - lam_td, 1
        chain.append((v + e - C, t))
        sH, sA, sDm = nH + H[t], nA + A[t], nDm
    # keadaan akhir terbaik (termasuk di tengah rantai)
    best_state, best_val = ("H", None), sH
    if sA > best_val:
        best_state, best_val = ("A", None), sA
    if sDm > best_val:
        best_state, best_val = ("Dm", None), sDm
    for w, et in chain:
        if w + C > best_val:
            best_state, best_val = ("C", et), w + C
    path = np.zeros(n, np.int8)
    t, (state, et) = n - 1, best_state
    while t >= 0:
        if state == "C":                       # di dalam rantai sejak frame et
            path[et:t + 1] = 2
            t = et - 1
            state = "H" if bD1[et] == 0 else "A"
            continue
        if state == "H":
            path[t] = 0
            prev = bH[t]
        elif state == "A":
            path[t] = 1
            prev = bA[t]
        else:                                  # Dm
            path[t] = 2
            if t == 0:
                break
            if bDm[t] == 1:
                t -= 1
                continue
            state, et = "C", t - m + 1          # datang dari rantai yang masuk di frame t-m+1
            path[et:t] = 2
            t = et - 1
            state = "H" if bD1[et] == 0 else "A"
            continue
        if t == 0:
            break
        t -= 1
        state = ("H", "A", "Dm")[prev]
    return path


FORCE_DEAD = np.zeros(N_FRAMES, bool)
FORCE_DEAD[HALF_BLOCKS[1][1]:KICKOFF_2ND] = True


def smart_tag_evidence():
    L = np.zeros(N_FRAMES, np.float32)
    for _, r in ST.iterrows():
        if r["side"] < 0 or (r["code"] not in POSSESSION_CODES and r["code"] not in DEFENDING_CODES):
            continue
        sign = 1 if r["side"] == 0 else -1
        L[r["start"]:r["end"]] += -sign if r["code"] in DEFENDING_CODES else sign
    return L


L_ST_RAW = smart_tag_evidence()


def scenario_evidence(spec):
    R = SOURCES[spec["source"]]
    m1, m2 = R["m1"][spec["model"]], R["m2"][spec["model"]]
    p_home = np.full(N_FRAMES, np.nan, np.float32)
    p_home[R["m1d"]["longm"]] = m1["p_inf"]
    L_gap = np.where(R["m1d"]["src"] == 3, logit(np.nan_to_num(p_home, nan=.5)), 0).astype(np.float32)
    return {"L_episode": R["L_episode"], "L_gap": L_gap, "llr_dead": logit(np.nan_to_num(m2["p_dead"], nan=.5)),
            "src": R["m1d"]["src"]}


def decode(ev, lam_switch=None, dead_temp=None, gap_weight=None, st_weight=None, lam_to_dead=None, lam_from_dead=None):
    g = lambda v, k: CONFIG[k] if v is None else v  # noqa: E731
    Lep = ev["L_episode"]
    k = int(CONFIG["switch_delay_frames"])
    if k:
        Lep = np.r_[np.zeros(k, np.float32), Lep[:-k]] if k > 0 else np.r_[Lep[-k:], np.zeros(-k, np.float32)]
    Lt = Lep + g(gap_weight, "gap_weight") * ev["L_gap"] + g(st_weight, "st_weight") * L_ST_RAW
    Ld = np.where(FORCE_DEAD, 50.0, g(dead_temp, "dead_temp") * ev["llr_dead"])
    lpH = log_sigmoid(-Ld) + np.log(2) + log_sigmoid(Lt)
    lpA = log_sigmoid(-Ld) + np.log(2) + log_sigmoid(-Lt)
    lpD = log_sigmoid(Ld)
    labels = np.full(N_FRAMES, "X", dtype="<U1")
    for half, a, b in HALF_BLOCKS:
        path = viterbi_half_mindead(lpH[a:b + 1], lpA[a:b + 1], lpD[a:b + 1], g(lam_switch, "lam_switch"),
                                    g(lam_to_dead, "lam_to_dead"), g(lam_from_dead, "lam_from_dead"),
                                    int(round(CONFIG["min_dead_s"] * FPS)))
        labels[a:b + 1] = np.array(["H", "A", "D"])[path]
    return labels


def finalize_labels(labels, src):
    out = labels.copy()
    out[~GRADED] = "X"
    if CONFIG["abstain_in_gaps"]:
        for half, a2, b2 in HALF_BLOCKS:
            seg = out[a2:b2 + 1]
            ch = np.flatnonzero(seg[1:] != seg[:-1]) + 1
            for s, e in zip(np.r_[0, ch], np.r_[ch - 1, len(seg) - 1]):
                if seg[s] in ("H", "A") and e - s > 2:
                    inner = np.arange(a2 + s + 1, a2 + e)
                    out[inner[src[inner] == 3]] = "X"
    return out

## 11. Metrik evaluasi
- **Skor resmi pada 5 menit publik** (`possession_metrics.evaluate`): hanya cek kebenaran. 17 possession;
  selisih `spellF1` < 0,04 adalah noise.
- **Metrik proxy pertandingan penuh** (tanpa GT): kesesuaian tim dengan fase Smart Tagging (independen karena
  `st_weight = 0`), restart terdeteksi (konsistensi; M2 dilatih dari tag yang sama), dan laju possession.

In [ ]:
def spells(labels):
    out = []
    for half, a, b in HALF_BLOCKS:
        seg = labels[a:b + 1]
        ch = np.flatnonzero(seg[1:] != seg[:-1]) + 1
        for s, e in zip(np.r_[0, ch], np.r_[ch - 1, len(seg) - 1]):
            if seg[s] in ("H", "A"):
                out.append((a + s, a + e, seg[s]))
    return out


def score_against(gt_codes, labels, bootstrap=0):
    r = pm.evaluate(gt_codes, pm.label_codes(labels), bootstrap_n=bootstrap)
    tr = r["transitions"].get(25, {})
    return {"trans_f1_25": tr.get("f1"), "trans_matched_25": tr.get("n_matched"), "trans_gt": tr.get("n_gt"),
            "trans_pred_25": tr.get("n_pred"), "latency_median_25": tr.get("median_latency"),
            "spellF1": r["spell_f1_mean"], "macroF1": r["macro_f1"], "spell_lo": r["spell_bootstrap_low"],
            "spell_hi": r["spell_bootstrap_high"], "n_pred_spells": r["n_pred_spells"], "n_gt_spells": r["n_gt_spells"],
            "mean_dur_s": r["mean_spell_duration_s"], "alive_dead_f1": r["alive_dead_f1"],
            "team_acc": r["team_acc_both_alive"]}


SP_TAGS = ST[ST["code"] == "SET PIECES"]


def proxy_metrics(labels):
    agree, total = 0, 0
    for _, r in ST.iterrows():
        if r["side"] < 0 or (r["code"] not in POSSESSION_CODES and r["code"] not in DEFENDING_CODES):
            continue
        want = "H" if r["side"] == 0 else "A"
        if r["code"] in DEFENDING_CODES:
            want = "A" if want == "H" else "H"
        seg = labels[r["start"]:r["end"]]
        alive = seg[(seg == "H") | (seg == "A")]
        agree += int((alive == want).sum())
        total += len(alive)
    prev_lab = np.concatenate([np.array(["X"]), labels[:-1]])
    trans = (prev_lab == "D") & ((labels == "H") | (labels == "A"))
    tc = np.r_[0, np.cumsum(trans)]
    hit = [tc[min(r["end"] + 2 * FPS, N_FRAMES)] - tc[max(r["start"], 0)] > 0 for _, r in SP_TAGS.iterrows()]
    sp = spells(labels)
    alive_g = np.isin(labels[GRADED], ["H", "A"])
    durs = np.array([(e - s + 1) / FPS for s, e, _ in sp]) if sp else np.array([0.0])
    return {"team_agree": agree / max(total, 1), "restart_hit": float(np.mean(hit)) if hit else float("nan"),
            "n_spells": len(sp), "spells_per_min_alive": len(sp) / max(alive_g.sum() / FPS / 60, 1e-9),
            "median_spell_s": float(np.median(durs)), "dead_share": float((labels[GRADED] == "D").mean()),
            "home_share": float((labels[GRADED] == "H").sum() / max(alive_g.sum(), 1))}


GT_EXTRA = pm.label_codes(pm.read_label_csv(CONFIG["extra_gt_csv"])) if CONFIG["extra_gt_csv"] else None


def ribbon(ax, labels, y, a, b, h=0.8):
    seg = labels[a:b + 1]
    ch = np.flatnonzero(seg[1:] != seg[:-1]) + 1
    for s, e in zip(np.r_[0, ch], np.r_[ch, len(seg)]):
        ax.add_patch(Rectangle((s / FPS, y - h / 2), (e - s) / FPS, h, color=COLORS.get(seg[s], "w"), lw=0))


PUB = np.flatnonzero(GT_PUBLIC != pm.X)
PUB_A, PUB_B = PUB[0], PUB[-1]
BASE_SCORES = {k: score_against(GT_PUBLIC, v, CONFIG["bootstrap_n"]) for k, v in BASELINES.items()}
BASE_PROXY = {k: proxy_metrics(v) for k, v in BASELINES.items()}
print(pd.DataFrame(BASE_SCORES).T[["spellF1", "macroF1", "n_pred_spells"]].round(4).to_string())

## 12. Skenario S6 — Deteksi bola dengan YOLO dari video taktis (versi 3)

**Tujuan:** mengisi frame yang bolanya hilang dari tracking, lalu menjalankan ulang tahap A–D dengan sumber bola gabungan.
Sel pertama di bawah berisi fungsi dasar (runtime, video, dataset dasar v2, training, inferensi). Alur yang
**dijalankan** adalah alur versi 3 di Bagian 12.A.

| Langkah | Isi | Checkpoint |
|---|---|---|
| 12.1 | Cek runtime (GPU, PyTorch, Ultralytics) dan video (jumlah frame harus 199.319) | — |
| 12.2 | **Pseudo-label otomatis**: posisi bola ATD diproyeksikan ke piksel via homography → potongan 960×960 berlabel | `yolo_dataset` |
| 12.3 | **Fine-tuning** YOLO satu kelas (`ball`); `last.pt` disimpan tiap epoch -> bila terputus, **dilanjutkan dari epoch terakhir** | `yolo_train` + `last.pt` |
| 12.4 | **Inferensi full-frame** pada frame dinilai yang bolanya hilang + frame evaluasi (bola terlihat) | `yolo_detect` (bisa dilanjutkan) |
| 12.5 | Seleksi deteksi (confidence + dekat prior interpolasi), piksel → lapangan, evaluasi, penggabungan | — |

Model dilatih pada potongan beresolusi asli dan dipakai pada frame penuh beresolusi asli, sehingga ukuran bola
dalam piksel sama saat latih dan inferensi. **Keterbatasan:** homography mengasumsikan titik di permukaan lapangan,
jadi posisi bola yang melambung akan bergeser; bola yang tertutup pemain tetap tidak terlihat.

In [ ]:
def patch_ultralytics_imread():
    """Ultralytics membaca gambar sekali saja; di Google Drive pembacaan bisa gagal sesaat -> 'Image Not Found'.
    Fungsi ini mengganti pembaca gambar di loader training dengan versi yang mencoba ulang."""
    import ultralytics.data.base as ub
    if getattr(ub.imread, "_mtah_retry", False):
        return
    original = ub.imread

    def imread_retry(filename, flags=cv2.IMREAD_COLOR):
        for attempt in range(CONFIG["io_retries"] + 1):
            im = original(filename, flags)
            if im is not None:
                return im
            if attempt < CONFIG["io_retries"]:
                time.sleep(CONFIG["io_retry_wait_s"] * (attempt + 1))
        print("  [I/O] gambar tetap tidak terbaca setelah {} percobaan: {}".format(CONFIG["io_retries"], filename))
        return None
    imread_retry._mtah_retry = True
    ub.imread = imread_retry


class RobustVideo:
    """Pembaca video berurutan yang menyambung ulang (buka ulang + lompat ke frame) bila pembacaan dari Drive terputus."""

    def __init__(self, path):
        self.path, self.cap, self.pos = path, None, 0
        self._open(0)

    def _open(self, at):
        if self.cap is not None:
            self.cap.release()
        self.cap = cv2.VideoCapture(self.path)
        if not self.cap.isOpened():
            raise IOError("Gagal membaca frame video: file tidak bisa dibuka ({})".format(self.path))
        if at:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, at)
        self.pos = at

    def grab(self):
        """Maju satu frame. Indeks frame yang baru diambil = self.pos - 1."""
        for attempt in range(CONFIG["io_retries"] + 1):
            if self.cap.grab():
                self.pos += 1
                return True
            if self.pos >= N_FRAMES:
                return False
            if attempt < CONFIG["io_retries"]:
                print("  [video] gagal membaca frame {}, menyambung ulang ({}/{}) ...".format(
                    self.pos, attempt + 1, CONFIG["io_retries"]))
                time.sleep(CONFIG["io_retry_wait_s"] * (attempt + 1))
                self._open(self.pos)
        raise IOError("Gagal membaca frame video {} setelah {} percobaan".format(self.pos, CONFIG["io_retries"]))

    def retrieve(self):
        ok, img = self.cap.retrieve()
        return img if ok else None

    def release(self):
        if self.cap is not None:
            self.cap.release()


def yolo_runtime():
    cfg = CONFIG["yolo"]
    info = {"ok": False, "reason": ""}
    if not cfg["enabled"]:
        info["reason"] = "YOLO dinonaktifkan di CONFIG"
        return info
    if not any(s["source"] == "yolo" for s in CONFIG["scenarios"].values()):
        info["reason"] = "tidak ada skenario bersumber 'yolo'"
        return info
    if "video" not in ASSETS:
        info["reason"] = "video taktis tidak ditemukan"
        return info
    if importlib.util.find_spec("torch") is None:
        info["reason"] = ("PyTorch belum terpasang. Pasang versi CUDA dari https://pytorch.org/get-started/locally/ "
                          "lalu jalankan ulang notebook.")
        return info
    import torch
    cuda = torch.cuda.is_available()
    dev = cfg["device"]
    if dev == "auto":
        dev = 0 if cuda else "cpu"
    if dev == "cpu" and cfg["require_gpu"]:
        info["reason"] = "GPU CUDA tidak terdeteksi oleh PyTorch (require_gpu=True)"
        return info
    if importlib.util.find_spec("ultralytics") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    import ultralytics
    patch_ultralytics_imread()
    info.update(ok=True, device=dev, torch=torch.__version__, cuda=cuda,
                gpu=torch.cuda.get_device_name(0) if cuda else "cpu", ultralytics=ultralytics.__version__)
    return info


YOLO_RT = yolo_runtime()
print("YOLO runtime:", YOLO_RT)
VIDEO_INFO = {}
if YOLO_RT["ok"]:
    _cap = retry_io(lambda: cv2.VideoCapture(ASSETS["video"]), "membuka video")
    VIDEO_INFO = {"frames_meta": int(_cap.get(cv2.CAP_PROP_FRAME_COUNT)), "fps": float(_cap.get(cv2.CAP_PROP_FPS)),
                  "width": int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "height": int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}
    _cap.release()
    print("Video:", VIDEO_INFO)
    if abs(VIDEO_INFO["frames_meta"] - N_FRAMES) > 2:
        YOLO_RT.update(ok=False, reason="jumlah frame video {} != {} (lihat FAQ: jangan dipaksa cocok)".format(
            VIDEO_INFO["frames_meta"], N_FRAMES))
        print("S6 DILEWATI:", YOLO_RT["reason"])
else:
    print("S6 DILEWATI:", YOLO_RT["reason"])

In [ ]:
def pitch_to_px(f, xm, ym):
    q = HOM[f] @ np.array([xm / L_PITCH, ym / W_PITCH, 1.0])
    return q[0] / q[2], q[1] / q[2]


def px_scale(f, xm, ym):
    """Piksel per meter di sekitar titik (rata-rata arah x dan y)."""
    u0, v0 = pitch_to_px(f, xm, ym)
    u1, v1 = pitch_to_px(f, xm + 0.5, ym)
    u2, v2 = pitch_to_px(f, xm, ym + 0.5)
    return (np.hypot(u1 - u0, v1 - v0) + np.hypot(u2 - u0, v2 - v0)) / 2 / 0.5


def px_to_pitch(f, u, v):
    Hi = np.linalg.inv(HOM[f])
    q = Hi @ np.vstack([u, v, np.ones_like(u)])
    return q[0] / q[2] * L_PITCH, q[1] / q[2] * W_PITCH


def yolo_frame_plan():
    """Frame latih/validasi (bola ATD terlihat, 'di tanah') dan frame inferensi (bola hilang)."""
    cfg = CONFIG["yolo"]
    rng = np.random.default_rng(CONFIG["seed"])
    prep = SOURCES["atd"]["prep"]
    B, obs, spd = prep["B"], prep["obs"], prep["speed"]
    inside = (B[:, 0] >= 0) & (B[:, 0] <= L_PITCH) & (B[:, 1] >= 0) & (B[:, 1] <= W_PITCH)
    cand = GRADED & obs & HOM_VALID & inside & np.nan_to_num(spd < cfg["label_max_speed"], nan=False)
    block = np.arange(N_FRAMES) // (FPS * CONFIG["cv_block_s"])
    is_val = block % 5 == 0
    pick = lambda m, k: np.sort(rng.choice(np.flatnonzero(m), min(k, int(m.sum())), replace=False))  # noqa: E731
    train_f, val_f = pick(cand & ~is_val, cfg["n_train"]), pick(cand & is_val, cfg["n_val"])
    eval_f = pick(GRADED & obs & HOM_VALID & inside & is_val & ~np.isin(np.arange(N_FRAMES), val_f), cfg["n_eval"])
    target = np.flatnonzero(GRADED & np.isnan(BALL_RAW[:, 0]) & HOM_VALID)
    if cfg["max_infer_frames"]:
        target = target[:cfg["max_infer_frames"]]
    return {"train": train_f, "val": val_f, "eval": eval_f, "target": target}


def write_jpg(path, img, retries=3):
    """Tulis JPEG lalu verifikasi bisa dibaca ulang (cv2.imwrite gagal diam-diam di file system virtual)."""
    ok_enc, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, CONFIG["yolo"]["jpeg_quality"]])
    if not ok_enc:
        return False
    for attempt in range(max(retries, CONFIG["io_retries"])):
        try:
            with open(path, "wb") as fh:
                fh.write(buf.tobytes())
                fh.flush()
                os.fsync(fh.fileno())
            if os.path.getsize(path) == len(buf):
                return True
        except OSError as exc:
            if getattr(exc, "errno", None) == errno.ENOSPC or getattr(exc, "winerror", None) == 112:
                raise
            time.sleep(CONFIG["io_retry_wait_s"] * (attempt + 1))
    return False


def verify_yolo_images(root):
    """Daftar gambar yang tidak bisa dibaca (kosong/terpotong/hilang) atau tidak punya pasangan label."""
    bad = []
    for split in ("train", "val"):
        for ip in glob.glob(os.path.join(root, "images", split, "*.jpg")):
            lp = os.path.join(root, "labels", split, os.path.basename(ip)[:-4] + ".txt")
            im = None
            for attempt in range(3):
                try:
                    buf = np.fromfile(ip, dtype=np.uint8)
                    im = cv2.imdecode(buf, cv2.IMREAD_COLOR) if buf.size else None
                except OSError:
                    im = None
                if im is not None:
                    break
                time.sleep(CONFIG["io_retry_wait_s"])
            if im is None or not os.path.exists(lp):
                bad.append(ip)
    return bad


def estimate_yolo_gb():
    cfg = CONFIG["yolo"]
    per_img = cfg["crop"] ** 2 * 3 * (0.03 + 0.0012 * cfg["jpeg_quality"])   # perkiraan ukuran JPEG
    base = (cfg["n_train"] + cfg["n_val"]) * per_img / 1e9 + 0.3            # + folder runs (bobot per epoch)
    v3 = (cfg["n_train"] + cfg["n_val"]) * CONFIG["yolo_v3"]["neg_frac"] * per_img / 1e9 + 0.3   # negatif + runs_v3
    return base + v3                       # gambar positif v3 = hard link ke gambar v2 (bila tidak bisa: disalin)


def choose_yolo_dir():
    """Pilih lokasi dataset YOLO. Mode drive: temp lokal bila ruang cukup (cepat & tanpa error sinkron), selain itu Drive."""
    loc = CONFIG["yolo"]["dataset_location"]
    need = estimate_yolo_gb()
    temp_dir = os.path.join(tempfile.gettempdir(), "mtah2027_yolo")
    free_tmp = disk_free_gb(tempfile.gettempdir()) or 0
    reserve = CONFIG["min_free_gb_local"]
    if loc == "auto":
        if STORAGE_MODE != "drive":
            loc = "work_dir"
        else:
            loc = "local_temp" if free_tmp >= need + reserve else "work_dir"
    if loc == "local_temp":
        if free_tmp < need + 0.5:
            raise OSError(errno.ENOSPC, "ruang temp lokal {:.1f} GB < kebutuhan dataset YOLO {:.1f} GB".format(free_tmp, need))
        where, path = "local_temp", temp_dir
    else:
        where, path = "work_dir", os.path.join(FAST_DIR, "yolo")
    print("Lokasi dataset YOLO: {} -> {}\n  perkiraan kebutuhan {:.1f} GB | ruang temp lokal {:.1f} GB | cadangan untuk "
          "Drive {:.1f} GB".format(where, path, need, free_tmp, reserve))
    if where == "work_dir" and STORAGE_MODE == "drive":
        print("  Dataset ditulis ke Google Drive: setiap gambar diverifikasi setelah ditulis dan dibaca ulang otomatis "
              "saat training.")
    return path, where


def yolo_dataset_ok(ds, folder="dataset"):
    """Checkpoint dataset hanya sah bila foldernya ada di lokasi kerja SAAT INI dan semua gambar terbaca."""
    root = os.path.join(DIRS["yolo"], folder)
    if os.path.normcase(os.path.abspath(ds.get("root", ""))) != os.path.normcase(os.path.abspath(root)):
        return False
    n = {sp: len(glob.glob(os.path.join(root, "images", sp, "*.jpg"))) for sp in ("train", "val")}
    if n != ds["counts"] or not os.path.exists(ds["yaml"]):
        return False
    return not verify_yolo_images(root)


def find_previous_yolo_images(frames):
    """Cari gambar train/val dari run sebelumnya (temp lokal, folder YOLO saat ini, atau work_dir/yolo di Drive).
    Hasil: {frame: (path_gambar, cx_norm, cy_norm)} -- pusat bola dibaca dari label lama, jadi gambar bisa
    diberi label baru (kotak lebih besar) TANPA mendekode video lagi."""
    cands = [os.path.join(DIRS["yolo"], "dataset"), os.path.join(tempfile.gettempdir(), "mtah2027_yolo", "dataset"),
             os.path.join(WORK_DIR, "yolo", "dataset"), os.path.join(FAST_DIR, "yolo", "dataset")]
    seen, found = set(), {}
    want = set(int(f) for f in frames)
    for root in cands:
        key = os.path.normcase(os.path.abspath(root))
        if key in seen or not os.path.isdir(root):
            continue
        seen.add(key)
        n0 = len(found)
        for split in ("train", "val"):
            for lp in glob.glob(os.path.join(root, "labels", split, "f*.txt")):
                try:
                    f = int(os.path.basename(lp)[1:-4])
                except ValueError:
                    continue
                ip = os.path.join(root, "images", split, os.path.basename(lp)[:-4] + ".jpg")
                if f not in want or f in found or not os.path.exists(ip):
                    continue
                try:
                    vals = open(lp).read().split()
                    found[f] = (ip, float(vals[1]), float(vals[2]))
                except (OSError, IndexError, ValueError):
                    continue
        if len(found) > n0:
            print("  gambar lama ditemukan di {}: {}".format(root, len(found) - n0))
    return found


def build_yolo_dataset(plan):
    """Bangun dataset YOLO. Gambar dari run sebelumnya dipakai ulang (hanya labelnya ditulis ulang);
    hanya frame yang belum punya gambar yang diambil dari video."""
    cfg = CONFIG["yolo"]
    rng = np.random.default_rng(CONFIG["seed"] + 1)
    B = SOURCES["atd"]["prep"]["B"]
    c = cfg["crop"]
    root = os.path.join(DIRS["yolo"], "dataset")
    staging = root + "_baru"
    if os.path.isdir(staging):
        shutil.rmtree(staging)
    for split in ("train", "val"):
        os.makedirs(os.path.join(staging, "images", split), exist_ok=True)
        os.makedirs(os.path.join(staging, "labels", split), exist_ok=True)
    need = {int(f): "train" for f in plan["train"]}
    need.update({int(f): "val" for f in plan["val"]})
    prev = find_previous_yolo_images(list(need)) if cfg.get("reuse_previous_images", True) else {}
    counts = {"train": 0, "val": 0}
    box_sizes, failed, reused = [], [], 0

    def box_px(f):
        bs = max(cfg["min_box_px"], cfg["ball_diam_m"] * px_scale(f, B[f, 0], B[f, 1]) * cfg["box_scale"])
        return min(bs, c * 0.5)

    def write_label(split, name, cx, cy, bs):
        def _w():
            with open(os.path.join(staging, "labels", split, name + ".txt"), "w") as fh:
                fh.write("0 {:.6f} {:.6f} {:.6f} {:.6f}\n".format(cx, cy, bs / c, bs / c))
        retry_io(_w, "menulis label " + name)

    # 1) pakai ulang gambar lama: salin gambar, tulis label baru dengan pusat yang sama
    for f, (ip, cx, cy) in sorted(prev.items()):
        split, name = need[f], "f{:06d}".format(f)
        im = cv2.imdecode(np.fromfile(ip, dtype=np.uint8), cv2.IMREAD_COLOR) if os.path.getsize(ip) else None
        if im is None or im.shape[0] != c or im.shape[1] != c:
            continue                                   # ukuran crop berbeda/rusak -> diambil ulang dari video
        dst = os.path.join(staging, "images", split, name + ".jpg")
        retry_io(lambda: shutil.copy2(ip, dst), "menyalin gambar lama " + name)
        bs = box_px(f)
        write_label(split, name, cx, cy, bs)
        counts[split] += 1
        box_sizes.append(bs)
        reused += 1
    todo = {f: sp for f, sp in need.items() if not os.path.exists(
        os.path.join(staging, "images", sp, "f{:06d}.jpg".format(f)))}
    print("  dipakai ulang dari run sebelumnya: {} gambar | perlu diambil dari video: {}".format(reused, len(todo)))

    # 2) sisanya diambil dari video (dekode berurutan sampai frame terakhir yang dibutuhkan)
    if todo:
        last = max(todo)
        cap = RobustVideo(ASSETS["video"])
        t0 = time.time()
        for f in range(last + 1):
            if not cap.grab():
                raise IOError("Gagal membaca frame video {} (video berakhir lebih awal)".format(f))
            if f % 20000 == 0 and f:
                print("  dekode video: frame {:,} / {:,} ({:.0f} frame/s)".format(f, last, f / (time.time() - t0)))
            if f not in todo:
                continue
            img = cap.retrieve()
            if img is None:
                continue
            hh, ww = img.shape[:2]
            u, v = pitch_to_px(f, B[f, 0], B[f, 1])
            if not (0 <= u < ww and 0 <= v < hh):
                continue
            bs = box_px(f)
            lim = max(c / 2 - 2 * bs, 0)
            cxp, cyp = u + rng.uniform(-lim, lim), v + rng.uniform(-lim, lim)
            x0, y0 = int(round(cxp - c / 2)), int(round(cyp - c / 2))
            canvas = np.full((c, c, 3), 114, np.uint8)
            sx0, sy0, sx1, sy1 = max(x0, 0), max(y0, 0), min(x0 + c, ww), min(y0 + c, hh)
            canvas[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = img[sy0:sy1, sx0:sx1]
            split, name = todo[f], "f{:06d}".format(f)
            if not write_jpg(os.path.join(staging, "images", split, name + ".jpg"), canvas):
                failed.append(f)
                continue
            write_label(split, name, (u - x0) / c, (v - y0) / c, bs)
            counts[split] += 1
            box_sizes.append(bs)
        cap.release()
    if failed:
        print("  {} gambar gagal ditulis dan dilewati (frame: {}...)".format(len(failed), failed[:5]))
    bad = verify_yolo_images(staging)
    if bad:
        if len(bad) > 0.05 * max(sum(counts.values()), 1):
            raise IOError("Image Not Found: {} gambar dataset YOLO tidak bisa dibaca ulang, contoh {}".format(len(bad), bad[:3]))
        for ip in bad:
            for fp in (ip, ip.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)[:-4] + ".txt"):
                with contextlib.suppress(OSError):
                    os.remove(fp)
        print("  {} gambar rusak dibuang dari dataset".format(len(bad)))
    counts = {sp: len(glob.glob(os.path.join(staging, "images", sp, "*.jpg"))) for sp in ("train", "val")}
    # 3) ganti dataset lama dengan yang baru (gambar lama sudah disalin, aman dihapus)
    if os.path.isdir(root):
        shutil.rmtree(root, ignore_errors=True)
    os.replace(staging, root)
    yaml_path = os.path.join(root, "data.yaml")
    with open(yaml_path, "w") as fh:
        fh.write("path: {}\ntrain: images/train\nval: images/val\nnames:\n  0: ball\n".format(root.replace("\\", "/")))
    return {"root": root, "yaml": yaml_path, "counts": counts, "reused": reused,
            "box_px_median": float(np.median(box_sizes)) if box_sizes else float("nan"),
            "box_px_p10": float(np.percentile(box_sizes, 10)) if box_sizes else float("nan"),
            "box_px_p90": float(np.percentile(box_sizes, 90)) if box_sizes else float("nan")}


def yolo_ckpt_info(path):
    """(epoch_terakhir_tersimpan, train_args) dari last.pt Ultralytics. epoch -1 = training sudah selesai."""
    import torch
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        ck = torch.load(path, map_location="cpu")
    return int(ck.get("epoch", -1)), dict(ck.get("train_args") or {})


def find_resumable_yolo_run(run_dirs, ds, epochs=None):
    """Cari folder run Ultralytics yang cocok dengan dataset & konfigurasi saat ini.
    Cocok = epochs & imgsz sama, file data sama, dan training dimulai SETELAH dataset (label) terakhir dibuat."""
    cfg = dict(CONFIG["yolo"])
    if epochs is not None:
        cfg["epochs"] = epochs
    data_mtime = os.path.getmtime(ds["yaml"]) if os.path.exists(ds["yaml"]) else 0
    for d in run_dirs:
        last = os.path.join(d, "weights", "last.pt")
        if not os.path.exists(last):
            continue
        try:
            ep, args = yolo_ckpt_info(last)
        except Exception as exc:
            print("  last.pt di {} tidak terbaca ({}), diabaikan".format(d, type(exc).__name__))
            continue
        args_file = os.path.join(d, "args.yaml")
        started = os.path.getmtime(args_file) if os.path.exists(args_file) else os.path.getmtime(last)
        same = (int(args.get("epochs", -1)) == int(cfg["epochs"]) and int(args.get("imgsz", -1)) == int(cfg["crop"])
                and os.path.normcase(os.path.abspath(str(args.get("data", "")))) ==
                os.path.normcase(os.path.abspath(ds["yaml"])) and started >= data_mtime - 2)
        if not same:
            print("  run lama di {} tidak cocok dengan dataset/konfigurasi saat ini -> tidak dilanjutkan".format(d))
            continue
        return d, ep
    return None, None


def read_results_csv(run_dir):
    path = os.path.join(run_dir, "results.csv")
    if not os.path.exists(path):
        return {}
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    return {k: float(last[k]) for k in df.columns if k.startswith("metrics/")}


def train_yolo(ds, run_name="ball", weights=None, epochs=None, patience=None, project=None, stage_name="yolo_train",
               legacy_names=("ball",)):
    """Fine-tuning YOLO dengan checkpoint per epoch.

    Ultralytics menyimpan weights/last.pt di akhir SETIAP epoch. Bila training terputus (kernel restart, laptop
    mati, Ctrl+C), run berikutnya otomatis MELANJUTKAN dari epoch terakhir, bukan dari epoch 1.
    v3: bobot awal, jumlah epoch, folder run, dan nama tahap bisa diatur (training v3 terpisah dari v2)."""
    from ultralytics import YOLO
    import torch
    cfg = dict(CONFIG["yolo"])
    if weights is not None:
        cfg["weights"] = list(weights)
    if epochs is not None:
        cfg["epochs"] = int(epochs)
    if patience is not None:
        cfg["patience"] = int(patience)
    workers = cfg["workers"] if cfg["workers"] is not None else (0 if IS_WINDOWS else 4)
    project = project or os.path.join(DIRS["yolo"], "runs")
    run_dirs = [os.path.join(project, run_name)] + [os.path.join(project, n) for n in legacy_names]
    bad = verify_yolo_images(ds["root"])
    if bad:
        raise IOError("Image Not Found: dataset YOLO rusak sebelum training ({} gambar), contoh {}".format(len(bad), bad[:3]))

    run_dir, ep = find_resumable_yolo_run(run_dirs, ds, cfg["epochs"])
    if stage_name in CKPT.force:                       # training ulang diminta eksplisit -> mulai dari epoch 1
        print("force_stages berisi '{}' -> training YOLO dimulai dari awal (run lama diabaikan)".format(stage_name))
        run_dir, ep = None, None
        shutil.rmtree(os.path.join(project, run_name), ignore_errors=True)
    base, batch, model = None, cfg["batch"], None
    if run_dir is not None and ep == -1 and os.path.exists(os.path.join(run_dir, "weights", "best.pt")):
        print("Training YOLO sudah selesai sebelumnya di {} -> bobot dipakai tanpa training ulang".format(run_dir))
        best, metrics = os.path.join(run_dir, "weights", "best.pt"), read_results_csv(run_dir)
        base = "resume:" + os.path.basename(run_dir)
    else:
        while True:
            try:
                if run_dir is not None and ep >= 0:
                    last = os.path.join(run_dir, "weights", "last.pt")
                    print("Training YOLO DILANJUTKAN dari epoch {} / {} ({})".format(ep + 2, cfg["epochs"], last))
                    model, base = YOLO(last), "resume:" + os.path.basename(run_dir)
                    model.train(resume=True, data=ds["yaml"], batch=batch, device=YOLO_RT["device"], workers=workers,
                                plots=False)
                else:
                    for w in cfg["weights"]:
                        try:
                            model, base = YOLO(w), w
                            break
                        except Exception as exc:
                            print("  bobot {} tidak tersedia: {}".format(w, exc))
                    if model is None:
                        raise RuntimeError("Tidak ada bobot YOLO yang bisa dimuat dari {}".format(cfg["weights"]))
                    print("Training YOLO baru: {} epoch, progres disimpan tiap epoch di {}".format(
                        cfg["epochs"], os.path.join(project, run_name)))
                    model.train(data=ds["yaml"], epochs=cfg["epochs"], patience=cfg["patience"], imgsz=cfg["crop"],
                                batch=batch, device=YOLO_RT["device"], workers=workers, seed=CONFIG["seed"],
                                deterministic=True, single_cls=True, project=project, name=run_name, exist_ok=True,
                                plots=False, verbose=False, scale=0.2, mosaic=0.5, degrees=0.0, fliplr=0.5)
                break
            except (RuntimeError, torch.cuda.OutOfMemoryError) as exc:
                if "out of memory" not in str(exc).lower() or batch <= 2:
                    raise
                batch //= 2
                torch.cuda.empty_cache()
                print("  GPU kehabisan memori -> mengulang dengan batch", batch, "(progres epoch tetap dipakai)")
                run_dir, ep = find_resumable_yolo_run(run_dirs, ds, cfg["epochs"])
                model = None
        best = str(model.trainer.best)
        metrics = {}
        for k, v in (getattr(model.trainer, "metrics", None) or {}).items():
            try:
                metrics[k] = float(v)
            except Exception:
                pass
    sha = hashlib.sha256(open(best, "rb").read()).hexdigest()
    dst = os.path.join(DIRS["checkpoints"], "yolo_best_{}.pt".format(sha[:12]))
    retry_io(lambda: shutil.copy2(best, dst), "menyalin bobot YOLO ke work_dir")
    return {"weights": dst, "sha256": sha, "base_weights": base, "metrics": metrics, "workers": workers, "batch": batch}


def infer_imgsz():
    w, h = VIDEO_INFO.get("width", 1920), VIDEO_INFO.get("height", 1080)
    s = int(np.ceil(max(w, h) * CONFIG["yolo"]["upscale"] / 32) * 32)
    return s


def detect_yolo(tr, plan):
    """Inferensi full-frame dengan simpan-sementara berkala (bisa dilanjutkan bila terputus)."""
    from ultralytics import YOLO
    cfg = CONFIG["yolo"]
    model = YOLO(tr["weights"])
    frames = np.unique(np.r_[plan["target"], plan["eval"]]).astype(int)
    need = set(frames.tolist())
    part = os.path.join(DIRS["checkpoints"], "yolo_detect_partial_{}.pkl".format(tr["sha256"][:12]))
    state = {"done_upto": -1, "rows": []}
    if os.path.exists(part):
        try:
            state = retry_io(lambda: load_pickle(part), "memuat progres deteksi")
            print("  melanjutkan dari frame", state["done_upto"])
        except Exception:
            print("  file progres deteksi rusak, mulai dari awal")
    imgsz = infer_imgsz()
    cap = RobustVideo(ASSETS["video"])
    batch_img, batch_f = [], []
    last = int(frames.max())
    since_save, t0 = 0, time.time()

    def predict_safe(imgs):
        """Prediksi dengan pemecahan batch otomatis bila GPU kehabisan memori."""
        import torch
        try:
            return model.predict(imgs, imgsz=imgsz, conf=cfg["conf_store"], max_det=cfg["max_det"],
                                 device=YOLO_RT["device"], verbose=False)
        except (RuntimeError, torch.cuda.OutOfMemoryError) as exc:
            if "out of memory" not in str(exc).lower() or len(imgs) == 1:
                raise
            torch.cuda.empty_cache()
            h = len(imgs) // 2
            return list(predict_safe(imgs[:h])) + list(predict_safe(imgs[h:]))

    def flush():
        nonlocal batch_img, batch_f
        if not batch_img:
            return
        res = predict_safe(batch_img)
        for f, r in zip(batch_f, res):
            if r.boxes is None or len(r.boxes) == 0:
                continue
            xywh = r.boxes.xywh.cpu().numpy()
            conf = r.boxes.conf.cpu().numpy()
            xm, ym = px_to_pitch(f, xywh[:, 0], xywh[:, 1])
            for k in range(len(conf)):
                state["rows"].append((f, float(xm[k]), float(ym[k]), float(conf[k]), float(xywh[k, 0]), float(xywh[k, 1])))
        state["done_upto"] = batch_f[-1]
        batch_img, batch_f = [], []

    def save_partial():
        def _w():
            with open(part + ".tmp", "wb") as fh:
                pickle.dump(state, fh)
            os.replace(part + ".tmp", part)
        retry_io(_w, "menyimpan progres deteksi")

    for f in range(last + 1):
        if not cap.grab():
            raise IOError("Gagal membaca frame video {} (video berakhir lebih awal)".format(f))
        if f <= state["done_upto"] or f not in need:
            continue
        img = cap.retrieve()
        if img is None:
            continue
        batch_img.append(img)
        batch_f.append(f)
        if len(batch_img) >= cfg["infer_batch"]:
            flush()
            since_save += cfg["infer_batch"]
            if since_save >= cfg["save_every"]:
                save_partial()
                since_save = 0
                print("  {:,} / {:,} frame target | {:.0f} frame/s".format(
                    int(np.searchsorted(frames, f)), len(frames), np.searchsorted(frames, f) / (time.time() - t0 + 1e-9)))
    flush()
    cap.release()
    det = pd.DataFrame(state["rows"], columns=["frame", "x", "y", "conf", "u", "v"])
    return {"det": det, "frames": frames, "imgsz": imgsz}


def choose_conf_threshold(det, plan):
    """Ambang confidence yang memaksimalkan F1 deteksi pada frame evaluasi (bola ATD terlihat, tanpa prior).
    Frame evaluasi terpisah dari data latih YOLO (blok waktu validasi), jadi ini pemilihan yang sah."""
    cfg = CONFIG["yolo"]
    if cfg["conf_select"] != "auto":
        return float(cfg["conf_select"]), None
    B = SOURCES["atd"]["prep"]["B"]
    ev = np.asarray(plan["eval"])
    d = det[det["frame"].isin(ev)].sort_values(["frame", "conf"], ascending=[True, False]).drop_duplicates("frame")
    err = np.hypot(d["x"].values - B[d["frame"].values, 0], d["y"].values - B[d["frame"].values, 1])
    hit = err <= cfg["match_radius_m"]
    rows = []
    for thr in np.round(np.arange(0.05, 0.8, 0.05), 2):
        keep = d["conf"].values >= thr
        tp = int((hit & keep).sum())
        prec = tp / keep.sum() if keep.sum() else 0.0
        rec = tp / max(len(ev), 1)
        rows.append({"thr": thr, "precision": prec, "recall": rec,
                     "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0, "n_selected": int(keep.sum())})
    tab = pd.DataFrame(rows)
    best = tab.sort_values(["f1", "thr"], ascending=[False, False]).iloc[0]
    thr = float(best["thr"]) if best["f1"] > 0 else 0.5
    print("Ambang confidence otomatis: {:.2f} (F1 {:.3f}, precision {:.3f}, recall {:.3f})".format(
        thr, best["f1"], best["precision"], best["recall"]))
    return thr, tab


def select_yolo(det, plan, thr):
    """Pilih satu deteksi per frame: conf >= thr, di dalam batas, dekat prior interpolasi ATD."""
    cfg = CONFIG["yolo"]
    m = CONFIG["ball_max_outside"]
    d = det[(det["conf"] >= thr) & (det["x"] > -m) & (det["x"] < L_PITCH + m) &
            (det["y"] > -m) & (det["y"] < W_PITCH + m)].copy()
    prep = SOURCES["atd"]["prep"]
    B, obs = prep["B"], prep["obs"]
    prev, nxt, pok, nok = prev_next_observed(obs, HALF_ID)
    f = d["frame"].values
    p, n = prev[f], nxt[f]
    valid = (p >= 0) & (n < N_FRAMES) & pok[f] & nok[f]
    pc, nc = np.clip(p, 0, N_FRAMES - 1), np.clip(n, 0, N_FRAMES - 1)
    frac = np.where(valid, (f - pc) / np.maximum(nc - pc, 1), 0)
    lx = np.where(valid, B[pc, 0] + (B[nc, 0] - B[pc, 0]) * frac, np.nan)
    ly = np.where(valid, B[pc, 1] + (B[nc, 1] - B[pc, 1]) * frac, np.nan)
    d["prior_dist"] = np.hypot(d["x"].values - lx, d["y"].values - ly)
    is_eval = np.isin(f, plan["eval"])
    d.loc[is_eval, "prior_dist"] = np.nan          # frame evaluasi: tanpa prior (bola ATD-nya sendiri terlihat)
    d = d[~(d["prior_dist"] > cfg["max_prior_dist_m"])]
    d = d.sort_values(["frame", "conf"], ascending=[True, False]).drop_duplicates("frame")
    sel = np.full((N_FRAMES, 2), np.nan, np.float32)
    conf = np.full(N_FRAMES, np.nan, np.float32)
    sel[d["frame"].values] = d[["x", "y"]].values
    conf[d["frame"].values] = d["conf"].values
    return sel, conf

### 12.A Versi 3 — label bersih, kotak terkalibrasi, contoh negatif, ambang presisi, filter lintasan

| Langkah | Isi | Checkpoint (folder `checkpoints_v3`) |
|---|---|---|
| 12.A1 | **Kalibrasi ukuran bola**: diameter terukur (`ball_px_ref` = 13 px, sisi dekat kamera) dibagi skala homography di titik itu → diameter efektif (m). Ukuran bola tampak di posisi lain = diameter efektif × skala lokal | — |
| 12.A2 | **Guru** = bobot YOLO v2 terbaik (mAP50) dari `checkpoints/` (dibaca saja) | — |
| 12.A3 | **Label ulang**: guru mencari bola ≤ 1,5 m dari posisi ATD pada crop v2 → pusat dihaluskan detektor blob (DoG) → cek kontras. Crop yang tidak terkonfirmasi **dibuang**. Kotak = bola tampak + 2×1,5 px (min 15 px) | `yolo3_dataset` |
| 12.A4 | **Contoh negatif**: 20% crop diterima, bola dihapus dengan inpainting, label kosong | (bagian dari `yolo3_dataset`) |
| 12.A5 | Training baru (bobot COCO), `last.pt` tiap epoch → bisa dilanjutkan | `yolo3_train` |
| 12.A6 | Inferensi full-frame: frame target + **40 segmen evaluasi 2 s** (bola ATD terlihat, diperlakukan seperti gap) | `yolo3_detect` (bisa dilanjutkan) |
| 12.A7 | Ambang confidence terkecil dengan **presisi ≥ 0,90** (seluruh rantai seleksi) → **filter lintasan** → isi gap | — |
| 12.A8 | Sumber `yolo` dijalankan ulang dengan **fusi hati-hati** (bukti YOLO berbobot 0,5, dwell 16 frame, M1 latih dari ATD) | `prep_yolo`, `m1_*_yolo`, ... |

Gambar crop v2 di `<folder YOLO>/dataset/` **tidak diubah**; dataset v3 ditulis ke `dataset_v3/` (hard link bila
memungkinkan, jadi hampir tidak memakai ruang tambahan). Semua label tetap dibuat **algoritma** (aturan lomba).
Gambar contoh di bagian ini berisi potongan video — **jangan dibagikan ke asisten AI**; yang aman dibagikan hanya
tabel angka (`yolo_v3_relabel_ringkasan`, `yolo_v3_ambang_confidence`, JSON evaluasi).

In [ ]:
def ball_px_calibration():
    """Diameter bola efektif (m) dari ukuran terukur di piksel. Lebih besar dari 0,22 m karena blur gerak,
    kompresi video, dan halo; justru itu yang menentukan ukuran kotak yang benar."""
    v3 = CONFIG["yolo_v3"]
    fr = GRADED_FRAMES[HOM_VALID[GRADED_FRAMES]]
    if not len(fr):
        raise RuntimeError("homography tidak valid pada frame dinilai; kalibrasi ukuran bola tidak bisa dilakukan")
    fr = fr[np.linspace(0, len(fr) - 1, min(400, len(fr))).astype(int)]
    med = lambda y: float(np.nanmedian([px_scale(f, L_PITCH / 2, y) for f in fr]))  # noqa: E731
    s_y0, s_yw = med(0.0), med(W_PITCH)
    near_y = 0.0 if s_y0 >= s_yw else W_PITCH
    far_y = W_PITCH - near_y
    inset = v3["ball_px_ref_inset_m"]
    inward = lambda y: y + inset if y == 0.0 else y - inset  # noqa: E731
    ref_y = inward(near_y if v3["ball_px_ref_where"] == "near" else far_y)
    s_ref = med(ref_y)
    if not np.isfinite(s_ref) or s_ref <= 0:
        raise RuntimeError("skala homography di titik referensi tidak valid ({})".format(s_ref))
    d_eff = v3["ball_px_ref"] / s_ref
    return {"near_touchline_y_m": near_y, "ref_point_m": [round(L_PITCH / 2, 2), round(ref_y, 2)],
            "px_per_m_ref": s_ref, "ball_px_ref": v3["ball_px_ref"], "eff_ball_diam_m": d_eff,
            "ball_px_near": d_eff * med(inward(near_y)), "ball_px_mid": d_eff * med(W_PITCH / 2),
            "ball_px_far": d_eff * med(inward(far_y)), "geom_ball_px_ref": CONFIG["yolo"]["ball_diam_m"] * s_ref}


def box_px_v3(d_app):
    v3 = CONFIG["yolo_v3"]
    return float(np.clip(d_app + 2 * v3["box_margin_px"], v3["min_box_px"], v3["max_box_px"]))


def resolve_teacher():
    """Bobot guru: path di CONFIG, atau ("auto") bobot dengan mAP50 tertinggi dari checkpoint yolo_train v2."""
    tw = CONFIG["yolo_v3"]["teacher_weights"]
    if not tw:
        return None
    if tw != "auto":
        if not os.path.exists(tw):
            raise FileNotFoundError("bobot guru YOLO tidak ditemukan: {}".format(tw))
        return {"weights": tw, "sha256": hashlib.sha256(open(tw, "rb").read()).hexdigest(), "map50": None,
                "source": "CONFIG"}
    best = None
    for d in CKPT.shared + [CKPT.root]:
        for pk in sorted(glob.glob(os.path.join(d, "yolo_train_*.pkl"))):
            try:
                r = retry_io(lambda: load_pickle(pk), "memuat " + os.path.basename(pk))
            except Exception:
                continue
            w = r.get("weights") or ""
            if not os.path.exists(w):
                alt = os.path.join(d, os.path.basename(w.replace("\\", "/")))   # folder kerja dipindah
                if not (w and os.path.exists(alt)):
                    continue
                w = alt
            m = (r.get("metrics") or {}).get("metrics/mAP50(B)")
            m = float(m) if m is not None and np.isfinite(m) else -1.0
            if best is None or m > best["map50"]:
                best = {"weights": w, "sha256": r.get("sha256") or hashlib.sha256(open(w, "rb").read()).hexdigest(),
                        "map50": m, "source": pk}
    return best


def read_img(path):
    def _r():
        buf = np.fromfile(path, dtype=np.uint8)
        im = cv2.imdecode(buf, cv2.IMREAD_COLOR) if buf.size else None
        if im is None:
            raise OSError("gambar tidak terbaca: {}".format(path))
        return im
    return retry_io(_r, "membaca " + os.path.basename(path))


def teacher_predict(weights, paths, imgsz, conf):
    """Deteksi guru pada crop dataset. Hasil per path: array (u, v, conf) dalam piksel crop."""
    from ultralytics import YOLO
    import torch
    model = YOLO(weights)
    out, bs, t0 = {}, max(1, int(CONFIG["yolo"]["batch"])), time.time()
    i = 0
    while i < len(paths):
        chunk = paths[i:i + bs]
        imgs = [read_img(p) for p in chunk]
        try:
            res = model.predict(imgs, imgsz=imgsz, conf=conf, max_det=10, device=YOLO_RT["device"], verbose=False)
        except (RuntimeError, torch.cuda.OutOfMemoryError) as exc:
            if "out of memory" not in str(exc).lower() or bs == 1:
                raise
            torch.cuda.empty_cache()
            bs = max(1, bs // 2)
            print("  GPU kehabisan memori -> batch guru", bs)
            continue
        for p, r in zip(chunk, res):
            if r.boxes is None or len(r.boxes) == 0:
                out[p] = np.zeros((0, 3), np.float32)
            else:
                xywh, cf = r.boxes.xywh.cpu().numpy(), r.boxes.conf.cpu().numpy()
                out[p] = np.c_[xywh[:, 0], xywh[:, 1], cf].astype(np.float32)
        i += len(chunk)
        if (i // bs) % 100 == 0 or i >= len(paths):
            print("  guru: {:,} / {:,} crop ({:.0f} crop/s)".format(i, len(paths), i / (time.time() - t0 + 1e-9)))
    return out


def refine_center(gray, cx, cy, d, max_shift):
    """Haluskan pusat bola dengan Difference-of-Gaussians (blob terang seukuran bola) di sekitar (cx, cy).
    Hasil (cx, cy, kontras) atau None bila terlalu dekat tepi. kontras = rata-rata abu di dalam bola - cincin luar."""
    h, w = gray.shape
    R = int(np.ceil(1.5 * d + max_shift)) + 2
    x0, y0 = int(round(cx)) - R, int(round(cy)) - R
    xa, ya, xb, yb = max(x0, 0), max(y0, 0), min(x0 + 2 * R + 1, w), min(y0 + 2 * R + 1, h)
    if xb - xa < 7 or yb - ya < 7:
        return None
    patch = gray[ya:yb, xa:xb].astype(np.float32)
    dog = (cv2.GaussianBlur(patch, (0, 0), max(d / 4.0, 0.8)) -
           cv2.GaussianBlur(patch, (0, 0), max(d / 1.6, 1.6)))
    yy, xx = np.mgrid[ya:yb, xa:xb].astype(np.float32)
    win = np.hypot(xx - cx, yy - cy) <= max(1.5, max_shift)
    if not win.any():
        return None
    k = int(np.argmax(np.where(win, dog, -np.inf)))
    py, px = np.unravel_index(k, dog.shape)
    if dog[py, px] > 0:
        y1, y2, x1, x2 = max(py - 1, 0), min(py + 2, dog.shape[0]), max(px - 1, 0), min(px + 2, dog.shape[1])
        wgt = np.clip(dog[y1:y2, x1:x2], 0, None)
        nx = float((wgt * xx[y1:y2, x1:x2]).sum() / wgt.sum())
        ny = float((wgt * yy[y1:y2, x1:x2]).sum() / wgt.sum())
    else:
        nx, ny = float(cx), float(cy)                  # tidak ada blob terang: pakai pusat guru
    dist = np.hypot(xx - nx, yy - ny)
    inner, ring = patch[dist <= max(d / 2.0, 1.0)], patch[(dist > max(d / 2.0, 1.0)) & (dist <= max(d, 2.5))]
    contrast = float(inner.mean() - ring.mean()) if inner.size and ring.size else float("nan")
    return nx, ny, contrast


def _link_or_copy(src, dst):
    try:
        os.link(src, dst)
        return "link"
    except (OSError, AttributeError, NotImplementedError):
        retry_io(lambda: shutil.copy2(src, dst), "menyalin " + os.path.basename(dst))
        return "copy"


def build_yolo_dataset_v3(base, teacher, cal):
    """Dataset v3 dari gambar crop v2 (tidak mendekode video): pusat label diperbaiki guru + DoG, kotak dikalibrasi,
    crop tak terkonfirmasi dibuang, ditambah contoh negatif. Dataset v2 tidak diubah."""
    v3, c = CONFIG["yolo_v3"], CONFIG["yolo"]["crop"]
    B = SOURCES["atd"]["prep"]["B"]
    rng = np.random.default_rng(CONFIG["seed"] + 3)
    root = os.path.join(DIRS["yolo"], "dataset_v3")
    staging = root + "_baru"
    if os.path.isdir(staging):
        shutil.rmtree(staging)
    for split in ("train", "val"):
        os.makedirs(os.path.join(staging, "images", split), exist_ok=True)
        os.makedirs(os.path.join(staging, "labels", split), exist_ok=True)
    items = []
    for split in ("train", "val"):
        for lp in sorted(glob.glob(os.path.join(base["root"], "labels", split, "f*.txt"))):
            name = os.path.basename(lp)[:-4]
            ip = os.path.join(base["root"], "images", split, name + ".jpg")
            try:
                f, vals = int(name[1:]), open(lp).read().split()
                items.append((split, f, ip, float(vals[1]) * c, float(vals[2]) * c))
            except (ValueError, IndexError, OSError):
                continue
    if not items:
        raise IOError("Image Not Found: dataset dasar YOLO kosong di {}".format(base.get("root")))
    use_teacher = bool(teacher) and v3["relabel"]
    if not use_teacher:
        print("  PERINGATAN: tanpa guru -> pusat label tetap dari ATD, tanpa contoh negatif (hanya kotak dikalibrasi)")
    preds = teacher_predict(teacher["weights"], [it[2] for it in items], c, v3["relabel_conf_min"]) if use_teacher else {}

    def write_label(split, name, lines):
        def _w():
            with open(os.path.join(staging, "labels", split, name + ".txt"), "w") as fh:
                fh.write(lines)
        retry_io(_w, "menulis label " + name)

    rows, modes, accepted = [], {"link": 0, "copy": 0}, {"train": [], "val": []}
    for split, f, ip, ax, ay in items:
        s = px_scale(f, B[f, 0], B[f, 1])
        d_app = cal["eff_ball_diam_m"] * s
        row = {"split": split, "frame": f, "atd_u": ax, "atd_v": ay, "d_app_px": d_app, "px_per_m": s}
        cx, cy = ax, ay
        if use_teacher:
            Pt = preds.get(ip, np.zeros((0, 3), np.float32))
            dist = np.hypot(Pt[:, 0] - ax, Pt[:, 1] - ay) if len(Pt) else np.zeros(0)
            near = np.flatnonzero(dist <= v3["relabel_radius_m"] * s)
            row["n_teacher"] = int(len(Pt))
            if not len(near):
                rows.append(dict(row, status="tolak_guru_tidak_menemukan"))
                continue
            j = int(near[np.argmax(Pt[near, 2])])
            row.update(teacher_conf=float(Pt[j, 2]), teacher_dist_px=float(dist[j]))
            if Pt[j, 2] < v3["relabel_conf"]:
                rows.append(dict(row, status="tolak_conf_guru_rendah"))
                continue
            ref = refine_center(cv2.cvtColor(read_img(ip), cv2.COLOR_BGR2GRAY), float(Pt[j, 0]), float(Pt[j, 1]),
                                d_app, v3["relabel_max_shift_frac"] * d_app)
            if ref is None:
                rows.append(dict(row, status="tolak_di_tepi_crop"))
                continue
            cx, cy, contrast = ref
            row["contrast"] = contrast
            if v3["relabel_min_contrast"] is not None and not (contrast >= v3["relabel_min_contrast"]):
                rows.append(dict(row, status="tolak_kontras_rendah"))
                continue
        if not (0 <= cx < c and 0 <= cy < c):
            rows.append(dict(row, status="tolak_di_luar_crop"))
            continue
        bs = box_px_v3(d_app)
        name = "f{:06d}".format(f)
        modes[_link_or_copy(ip, os.path.join(staging, "images", split, name + ".jpg"))] += 1
        write_label(split, name, "0 {:.6f} {:.6f} {:.6f} {:.6f}\n".format(cx / c, cy / c, bs / c, bs / c))
        rows.append(dict(row, status="diterima", u=cx, v=cy, box_px=bs, shift_from_atd_px=float(np.hypot(cx - ax, cy - ay))))
        accepted[split].append((f, ip, cx, cy, d_app))

    n_neg = {"train": 0, "val": 0}
    if use_teacher and v3["neg_frac"] > 0:
        for split, acc in accepted.items():
            k = min(int(round(v3["neg_frac"] * len(acc))), len(acc))
            for idx in (np.sort(rng.choice(len(acc), size=k, replace=False)) if k else []):
                f, ip, cx, cy, d_app = acc[idx]
                im = read_img(ip)
                mask = np.zeros(im.shape[:2], np.uint8)
                cv2.circle(mask, (int(round(cx)), int(round(cy))), int(np.ceil(d_app / 2 + 3)), 255, -1)
                neg = cv2.inpaint(im, mask, 5, cv2.INPAINT_TELEA)
                name = "n{:06d}".format(f)
                if write_jpg(os.path.join(staging, "images", split, name + ".jpg"), neg):
                    write_label(split, name, "")
                    n_neg[split] += 1
    bad = verify_yolo_images(staging)
    if bad:
        if len(bad) > 0.05 * max(len(rows), 1):
            raise IOError("Image Not Found: {} gambar dataset v3 tidak bisa dibaca ulang, contoh {}".format(len(bad), bad[:3]))
        for ip in bad:
            for fp in (ip, ip.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)[:-4] + ".txt"):
                with contextlib.suppress(OSError):
                    os.remove(fp)
    counts = {sp: len(glob.glob(os.path.join(staging, "images", sp, "*.jpg"))) for sp in ("train", "val")}
    if os.path.isdir(root):
        shutil.rmtree(root, ignore_errors=True)
    os.replace(staging, root)
    yaml_path = os.path.join(root, "data.yaml")
    with open(yaml_path, "w") as fh:
        fh.write("path: {}\ntrain: images/train\nval: images/val\nnames:\n  0: ball\n".format(root.replace("\\", "/")))
    tab = pd.DataFrame(rows)
    acc_t = tab[tab["status"] == "diterima"]
    q = lambda col, p: float(np.nanpercentile(acc_t[col], p)) if col in acc_t and len(acc_t) else float("nan")  # noqa: E731
    summary = {"crop_dasar": int(len(items)), "diterima": int(len(acc_t)),
               "frac_diterima": float(len(acc_t) / max(len(items), 1)),
               "status": {k: int(v) for k, v in tab["status"].value_counts().items()},
               "negatif": n_neg, "pakai_guru": use_teacher, "hard_link": modes["link"], "salin": modes["copy"],
               "box_px_p10": q("box_px", 10), "box_px_median": q("box_px", 50), "box_px_p90": q("box_px", 90),
               "bola_tampak_px_median": q("d_app_px", 50),
               "geser_dari_atd_px_median": q("shift_from_atd_px", 50), "geser_dari_atd_px_p90": q("shift_from_atd_px", 90),
               "kontras_p10": q("contrast", 10), "kontras_median": q("contrast", 50)}
    return {"root": root, "yaml": yaml_path, "counts": counts, "summary": summary, "table": tab,
            "box_px_median": summary["box_px_median"], "reused": int(len(items)), "teacher": teacher, "calibration": cal}


def yolo_eval_segments(plan):
    """Segmen evaluasi: potongan berurutan di blok validasi yang bolanya terlihat ATD sepanjang segmen + 1 frame
    jangkar di kedua ujung. Dipakai seperti gap buatan sehingga SELURUH rantai seleksi (prior, lintasan) teruji."""
    v3 = CONFIG["yolo_v3"]
    prep = SOURCES["atd"]["prep"]
    B, obs = prep["B"], prep["obs"]
    inside = (B[:, 0] >= 0) & (B[:, 0] <= L_PITCH) & (B[:, 1] >= 0) & (B[:, 1] <= W_PITCH)
    good = GRADED & obs & HOM_VALID & inside & ((np.arange(N_FRAMES) // (FPS * CONFIG["cv_block_s"])) % 5 == 0)
    good[np.asarray(plan["train"], int)] = False
    good[np.asarray(plan["val"], int)] = False
    Ls = int(v3["eval_seg_len"])
    starts = []
    for a, b in zip(*run_lengths(good)):
        if b - a + 1 >= Ls + 2:
            starts.extend(range(a + 1, b - Ls + 1))
    rng = np.random.default_rng(CONFIG["seed"] + 7)
    chosen, taken = [], np.zeros(N_FRAMES, bool)
    for s in (rng.permutation(np.asarray(starts, np.int64)) if starts else []):
        if len(chosen) >= v3["eval_segments"]:
            break
        if taken[max(s - Ls, 0):min(s + 2 * Ls, N_FRAMES)].any():
            continue
        taken[s:s + Ls] = True
        chosen.append((int(s), int(s + Ls - 1)))
    chosen.sort()
    frames = np.concatenate([np.arange(a, b + 1) for a, b in chosen]) if chosen else np.zeros(0, np.int64)
    runs = [(a, b, a - 1, b + 1) for a, b in chosen]
    return runs, frames


def yolo_candidates(det, thr, frames=None):
    """{frame: array (x, y, conf)} -- maksimal track_topk kandidat per frame, conf menurun."""
    m = CONFIG["ball_max_outside"]
    d = det[(det["conf"] >= thr) & (det["x"] > -m) & (det["x"] < L_PITCH + m) & (det["y"] > -m) & (det["y"] < W_PITCH + m)]
    if frames is not None:
        d = d[d["frame"].isin(np.asarray(frames))]
    d = d.sort_values(["frame", "conf"], ascending=[True, False]).groupby("frame", sort=False).head(
        CONFIG["yolo_v3"]["track_topk"])
    fr, xyz, out = d["frame"].values, d[["x", "y", "conf"]].values.astype(np.float64), {}
    if len(fr):
        cut = np.flatnonzero(np.diff(fr)) + 1
        for a_, b_ in zip(np.r_[0, cut], np.r_[cut, len(fr)]):
            out[int(fr[a_])] = xyz[a_:b_]
    return out


def target_runs(frames, obs):
    """Potongan berurutan frame target + jangkar (frame bola ATD terdekat sebelum/sesudah pada babak yang sama)."""
    frames = np.sort(np.asarray(frames, np.int64))
    if not len(frames):
        return []
    prev, nxt, pok, nok = prev_next_observed(obs, HALF_ID)
    cut = np.flatnonzero((np.diff(frames) != 1) | (HALF_ID[frames[1:]] != HALF_ID[frames[:-1]])) + 1
    runs = []
    for a_, b_ in zip(np.r_[0, cut], np.r_[cut, len(frames)]):
        a, b = int(frames[a_]), int(frames[b_ - 1])
        runs.append((a, b, int(prev[a]) if pok[a] else -1, int(nxt[b]) if nok[b] else -1))
    return runs


def track_fill(cands, runs, B, use_tracks=True):
    """Seleksi deteksi per gap. use_tracks=False: deteksi conf tertinggi per frame (cara v2, untuk pembanding).
    use_tracks=True: kandidat dirangkai menjadi lintasan (jarak antar-deteksi <= v_max*dt + toleransi, boleh
    melompati beberapa frame); hanya lintasan yang cukup panjang ATAU tersambung ke bola ATD di ujung gap diterima."""
    v3 = CONFIG["yolo_v3"]
    max_prior = CONFIG["yolo"]["max_prior_dist_m"]
    vstep, tol, skip = v3["track_vmax"] / FPS, v3["track_tol_m"], v3["track_max_skip"]
    out = {}
    st = {"gap": len(runs), "frame_berkandidat": 0, "lintasan": 0, "lintasan_diterima": 0, "lintasan_berjangkar": 0}
    for a, b, pa, pb in runs:
        A = B[pa].astype(np.float64) if pa >= 0 else None
        Z = B[pb].astype(np.float64) if pb >= 0 else None
        tracks, active = [], []
        for f in range(a, b + 1):
            C = cands.get(f)
            if C is None:
                continue
            if A is not None and Z is not None and max_prior:
                pr = A + (Z - A) * ((f - pa) / max(pb - pa, 1))
                C = C[np.hypot(C[:, 0] - pr[0], C[:, 1] - pr[1]) <= max_prior]
                if not len(C):
                    continue
            st["frame_berkandidat"] += 1
            if not use_tracks:
                out[f] = (C[0, 0], C[0, 1], C[0, 2], C[0, 2])
                continue
            active = [t for t in active if f - t["f"][-1] <= skip + 1]
            used = np.zeros(len(C), bool)
            for t in sorted(active, key=lambda t: -t["cs"]):
                dd = np.hypot(C[:, 0] - t["p"][-1][0], C[:, 1] - t["p"][-1][1])
                dd[used] = np.inf
                j = int(np.argmin(dd))
                if dd[j] <= vstep * (f - t["f"][-1]) + tol:
                    t["f"].append(f)
                    t["p"].append(C[j, :2])
                    t["c"].append(C[j, 2])
                    t["cs"] += float(C[j, 2])
                    used[j] = True
            for j in np.flatnonzero(~used):
                t = {"f": [f], "p": [C[j, :2]], "c": [C[j, 2]], "cs": float(C[j, 2])}
                tracks.append(t)
                active.append(t)
        for t in tracks:
            st["lintasan"] += 1
            n, anc = len(t["f"]), False
            if A is not None and t["f"][0] - pa <= skip + 1:
                anc |= bool(np.hypot(*(t["p"][0] - A)) <= vstep * (t["f"][0] - pa) + tol)
            if Z is not None and pb - t["f"][-1] <= skip + 1:
                anc |= bool(np.hypot(*(t["p"][-1] - Z)) <= vstep * (pb - t["f"][-1]) + tol)
            if n >= v3["track_min_len"] or (anc and n >= v3["track_min_len_anchored"]):
                st["lintasan_diterima"] += 1
                st["lintasan_berjangkar"] += int(anc)
                for f, p, cf in zip(t["f"], t["p"], t["c"]):
                    if f not in out or out[f][3] < t["cs"]:
                        out[f] = (float(p[0]), float(p[1]), float(cf), t["cs"])
    return out, st


def eval_selection(out, frames, B):
    frames = np.asarray(frames, np.int64)
    err = np.array([np.hypot(out[f][0] - B[f, 0], out[f][1] - B[f, 1]) if f in out else np.nan for f in frames])
    sel = np.isfinite(err)
    hit = np.nan_to_num(err, nan=1e9) <= CONFIG["yolo"]["match_radius_m"]
    return {"precision": float(hit.sum() / sel.sum()) if sel.any() else float("nan"),
            "recall": float(hit.mean()) if len(frames) else float("nan"), "n_selected": int(sel.sum()),
            "median_err_m": float(np.nanmedian(err)) if sel.any() else float("nan")}


def choose_conf_threshold_v3(det, runs, frames):
    """Ambang TERKECIL yang mencapai presisi target pada segmen evaluasi (rantai seleksi lengkap)."""
    v3 = CONFIG["yolo_v3"]
    B = SOURCES["atd"]["prep"]["B"]
    rows = []
    for thr in v3["thr_grid"]:
        cands = yolo_candidates(det, thr, frames)
        for mode, ut in (("lintasan", True), ("per_frame_v2", False)):
            out, _ = track_fill(cands, runs, B, ut)
            rows.append(dict(thr=thr, mode=mode, **eval_selection(out, frames, B)))
    tab = pd.DataFrame(rows)
    tr = tab[tab["mode"] == "lintasan"].copy()
    tr["precision"] = tr["precision"].fillna(0.0)
    ok = tr[(tr["precision"] >= v3["target_precision"]) & (tr["n_selected"] >= 20)]
    if len(ok):
        best, note = ok.sort_values("thr").iloc[0], "presisi target {:.2f} tercapai".format(v3["target_precision"])
    else:
        cand = tr[tr["recall"] >= 0.05]
        best = (cand if len(cand) else tr).sort_values(["precision", "thr"], ascending=[False, True]).iloc[0]
        note = ("PERINGATAN: presisi {:.2f} tidak tercapai di grid ambang; dipakai ambang dengan presisi tertinggi "
                "(recall >= 0,05)".format(v3["target_precision"]))
    return float(best["thr"]), tab, note


YOLO_RESULT, YDS, YOLO_DS_WHERE, YDS_BASE, YTEACHER, YCAL = None, {}, None, {}, None, None


def _yolo_block():
    global YPLAN, YDS, YDS_BASE, YTR, YDET, YSEL, YCONF, YOLO_EVAL, BALL_YOLO, YOLO_RESULT, DIAG_YOLO, YOLO_DS_WHERE
    global YTEACHER, YCAL, YTHR, YTHR_TAB, EVAL_RUNS, EVAL_FRAMES
    _yc, v3, fz = CONFIG["yolo"], CONFIG["yolo_v3"], CONFIG["yolo_fusion"]
    _pick = lambda d, keys: {k: d[k] for k in keys}  # noqa: E731
    # dataset dasar = dataset v2 (parameter identik -> checkpoint & gambar v2 dipakai ulang)
    _ds_params = dict(_pick(_yc, ("crop", "n_train", "n_val", "label_max_speed", "ball_diam_m", "box_scale",
                                  "min_box_px", "jpeg_quality")), seed=CONFIG["seed"], video=os.path.getsize(ASSETS["video"]))
    YPLAN = yolo_frame_plan()
    EVAL_RUNS, EVAL_FRAMES = yolo_eval_segments(YPLAN)
    YPLAN["eval_v2"], YPLAN["eval"] = YPLAN["eval"], EVAL_FRAMES
    print({k: len(v) for k, v in YPLAN.items()}, "| segmen evaluasi:", len(EVAL_RUNS))
    if len(EVAL_RUNS) < 5:
        print("  PERINGATAN: segmen evaluasi sangat sedikit ({}); ambang confidence kurang andal".format(len(EVAL_RUNS)))
    DIRS["yolo"], YOLO_DS_WHERE = choose_yolo_dir()
    os.makedirs(DIRS["yolo"], exist_ok=True)
    _ds_key = CKPT.key_for("yolo_dataset", _ds_params, ["prep_atd"])
    CKPT.keys["yolo_dataset"] = _ds_key

    with stage("kalibrasi ukuran bola"):
        YCAL = ball_px_calibration()
    print("Kalibrasi bola: diameter efektif {:.3f} m | bola tampak dekat {:.1f} px, tengah {:.1f} px, jauh {:.1f} px "
          "(acuan {} px di {} m; ukuran geometris 0,22 m = {:.1f} px)".format(
              YCAL["eff_ball_diam_m"], YCAL["ball_px_near"], YCAL["ball_px_mid"], YCAL["ball_px_far"],
              YCAL["ball_px_ref"], YCAL["ref_point_m"], YCAL["geom_ball_px_ref"]))
    print("  kotak label v3 = bola tampak + 2 x {} px, dibatasi [{}, {}] px".format(
        v3["box_margin_px"], v3["min_box_px"], v3["max_box_px"]))
    with stage("mencari bobot guru (YOLO v2)"):
        YTEACHER = resolve_teacher() if v3["relabel"] else None
    if YTEACHER:
        print("Guru:", YTEACHER["weights"], "| mAP50 v2 = {:.3f}".format(YTEACHER["map50"])
              if YTEACHER["map50"] is not None else "")
    elif v3["relabel"]:
        print("PERINGATAN: bobot YOLO v2 tidak ditemukan di {} -> label ulang tanpa guru (pusat dari ATD).".format(
            CKPT.shared + [CKPT.root]))

    _ds3_params = dict(_pick(v3, ("relabel", "relabel_conf", "relabel_conf_min", "relabel_radius_m",
                                  "relabel_max_shift_frac", "relabel_min_contrast", "box_margin_px", "min_box_px",
                                  "max_box_px", "neg_frac")),
                       teacher=(YTEACHER or {}).get("sha256"), eff_diam=round(YCAL["eff_ball_diam_m"], 5), v=3)
    _ds3_key = CKPT.make_key("yolo3_dataset", _ds3_params, [_ds_key])
    if v3["init_weights"] == "teacher" and YTEACHER:
        _init = [YTEACHER["weights"]]
        _init_id = "teacher:" + YTEACHER["sha256"][:12]
    else:
        _init, _init_id = list(_yc["weights"]), list(_yc["weights"])
    _tr3_params = {"init": _init_id, "epochs": v3["epochs"], "patience": v3["patience"], "batch": _yc["batch"],
                   "crop": _yc["crop"]}
    _tr3_key = CKPT.make_key("yolo3_train", _tr3_params, [_ds3_key])
    _ok_weights = lambda r: os.path.exists(r["weights"])  # noqa: E731
    _ok_ds3 = lambda r: yolo_dataset_ok(r, "dataset_v3")  # noqa: E731
    YTR = None
    if os.path.exists(CKPT.locate("yolo3_train", _tr3_key)[0]) and not ({"yolo3_train", "yolo3_dataset"} & CKPT.force):
        CKPT.keys["yolo3_dataset"] = _ds3_key        # model v3 sudah terlatih: dataset tidak perlu diperiksa ulang
        YTR = CKPT.peek("yolo3_train", _tr3_params, ["yolo3_dataset"], validate=_ok_weights)
    if YTR is None:
        YDS = None if "yolo3_dataset" in CKPT.force else CKPT.peek("yolo3_dataset", _ds3_params, ["yolo_dataset"],
                                                                   validate=_ok_ds3)
        if YDS is None:
            YDS_BASE = CKPT.run("yolo_dataset", lambda: build_yolo_dataset(YPLAN), _ds_params, ["prep_atd"],
                                validate=yolo_dataset_ok)
            print("Dataset dasar (crop v2):", YDS_BASE["counts"], "|", YDS_BASE["root"])
            YDS = CKPT.run("yolo3_dataset", lambda: build_yolo_dataset_v3(YDS_BASE, YTEACHER, YCAL), _ds3_params,
                           ["yolo_dataset"], validate=_ok_ds3)
        YTR = CKPT.run("yolo3_train", lambda: train_yolo(
            YDS, "ball3_" + _tr3_key[:8], weights=_init, epochs=v3["epochs"], patience=v3["patience"],
            project=os.path.join(DIRS["yolo"], "runs_v3"), stage_name="yolo3_train", legacy_names=()),
            _tr3_params, ["yolo3_dataset"], validate=_ok_weights)
    else:
        _p = CKPT.locate("yolo3_dataset", _ds3_key)[0]
        YDS = retry_io(lambda: load_pickle(_p), "memuat info dataset v3") if os.path.exists(_p) else {
            "counts": {}, "box_px_median": float("nan"), "root": "", "summary": {}}

    # --- ringkasan label ulang (angka saja: aman dibagikan) ---
    if YDS.get("summary"):
        sm = YDS["summary"]
        print("\nLabel ulang v3: {diterima:,} / {crop_dasar:,} crop diterima ({frac_diterima:.1%}) | negatif {negatif} | "
              "kotak median {box_px_median:.1f} px (p10 {box_px_p10:.1f}, p90 {box_px_p90:.1f})".format(**sm))
        print("  geser pusat dari ATD: median {:.1f} px, p90 {:.1f} px | kontras median {:.1f}".format(
            sm["geser_dari_atd_px_median"], sm["geser_dari_atd_px_p90"], sm["kontras_median"]))
        print("  status:", sm["status"])
        if sm["frac_diterima"] < 0.4:
            print("  PERINGATAN: < 40% crop diterima. Coba turunkan yolo_v3.relabel_min_contrast (mis. 0 atau None) "
                  "atau relabel_conf (mis. 0.05), lalu jalankan ulang (hanya tahap v3 yang dihitung ulang).")
        save_table(pd.Series(sm, dtype=object).to_frame("nilai"), "yolo_v3_relabel_ringkasan")
        if isinstance(YDS.get("table"), pd.DataFrame):
            save_table(YDS["table"].groupby(["split", "status"]).size().unstack(fill_value=0), "yolo_v3_relabel_status")
    print("Bobot YOLO v3:", YTR["weights"], "| metrik:", {k: round(v, 4) for k, v in YTR["metrics"].items()})
    with stage("menyalin bobot YOLO ke folder run"):
        retry_io(lambda: shutil.copy2(YTR["weights"], os.path.join(DIRS["models"], "yolo_v3_ball_best.pt")),
                 "menyalin bobot")

    _det_params = dict(_pick(_yc, ("conf_store", "max_det", "upscale", "max_infer_frames")),
                       eval_segments=v3["eval_segments"], eval_seg_len=v3["eval_seg_len"])
    YDET = CKPT.run("yolo3_detect", lambda: detect_yolo(YTR, YPLAN), _det_params, ["yolo3_train"])
    if YOLO_DS_WHERE == "local_temp" and _yc["cleanup_local_temp"]:
        for _sub in ("dataset_v3", "runs_v3"):
            shutil.rmtree(os.path.join(DIRS["yolo"], _sub), ignore_errors=True)

    # --- ambang presisi + filter lintasan ---
    Batd, obs_atd = SOURCES["atd"]["prep"]["B"], SOURCES["atd"]["prep"]["obs"]
    YTHR, YTHR_TAB, _note = choose_conf_threshold_v3(YDET["det"], EVAL_RUNS, EVAL_FRAMES)
    save_table(YTHR_TAB.set_index(["thr", "mode"]), "yolo_v3_ambang_confidence")
    print(YTHR_TAB.pivot(index="thr", columns="mode", values=["precision", "recall"]).round(3).to_string())
    print("Ambang confidence v3: {:.2f} ({})".format(YTHR, _note))
    tgt = YPLAN["target"]
    runs_t = target_runs(tgt, obs_atd)
    cands_t = yolo_candidates(YDET["det"], YTHR, tgt)
    out_t, st_t = track_fill(cands_t, runs_t, Batd, True)
    out_f, _ = track_fill(cands_t, runs_t, Batd, False)
    YSEL = np.full((N_FRAMES, 2), np.nan, np.float32)
    YCONF = np.full(N_FRAMES, np.nan, np.float32)
    if out_t:
        _fs = np.fromiter(out_t.keys(), np.int64)
        _v = np.array([out_t[f] for f in _fs])
        YSEL[_fs], YCONF[_fs] = _v[:, :2], _v[:, 2]
    cands_e = yolo_candidates(YDET["det"], YTHR, EVAL_FRAMES)
    ev_tr = eval_selection(track_fill(cands_e, EVAL_RUNS, Batd, True)[0], EVAL_FRAMES, Batd)
    ev_pf = eval_selection(track_fill(cands_e, EVAL_RUNS, Batd, False)[0], EVAL_FRAMES, Batd)
    r_m = _yc["match_radius_m"]
    YOLO_EVAL = {
        "eval_segments": len(EVAL_RUNS), "eval_frames": int(len(EVAL_FRAMES)),
        "recall_top@{}m".format(r_m): ev_tr["recall"], "precision_top@{}m".format(r_m): ev_tr["precision"],
        "median_err_m": ev_tr["median_err_m"],
        "per_frame_recall": ev_pf["recall"], "per_frame_precision": ev_pf["precision"],
        "target_frames": int(len(tgt)),
        "target_filled_frac": float(np.mean(np.isfinite(YSEL[tgt, 0]))) if len(tgt) else float("nan"),
        "target_filled_frac_per_frame": float(len(out_f) / max(len(tgt), 1)),
        "tracks": st_t, "conf_select_used": YTHR, "conf_note": _note,
        "box_px_median": YDS.get("box_px_median"), "reused_images": YDS.get("reused"),
        "relabel": YDS.get("summary"), "calibration": YCAL,
        "teacher": {k: v for k, v in (YTEACHER or {}).items() if k != "source"},
        "imgsz_infer": YDET["imgsz"], "train_metrics": YTR["metrics"], "weights_sha256": YTR["sha256"],
        "base_weights": YTR["base_weights"], "dataset": YDS.get("counts", {}),
    }
    print(json.dumps({k: v for k, v in YOLO_EVAL.items() if k not in ("relabel", "train_metrics")}, indent=2, default=str))

    fill = np.isnan(BALL_RAW[:, 0]) & np.isfinite(YSEL[:, 0]) & GRADED
    BALL_YOLO = BALL_RAW.copy()
    BALL_YOLO[fill] = YSEL[fill]
    print("Frame dinilai dengan bola: ATD {:.1%} -> ATD+YOLO v3 {:.1%} (per-frame tanpa filter lintasan akan {:.1%})".format(
        (~np.isnan(BALL_RAW[GRADED, 0])).mean(), (~np.isnan(BALL_YOLO[GRADED, 0])).mean(),
        (~np.isnan(BALL_RAW[GRADED, 0])).mean() + len(out_f) / max(int(GRADED.sum()), 1)))
    YOLO_RESULT = {"eval": YOLO_EVAL, "fill": fill}

    # --- gambar: kurva ambang, evaluasi, cakupan ---
    fig, axes = plt.subplots(1, 4, figsize=(19, 3.8))
    for mode, ls_ in (("lintasan", "-"), ("per_frame_v2", "--")):
        s_ = YTHR_TAB[YTHR_TAB["mode"] == mode]
        axes[0].plot(s_["thr"], s_["precision"], "o" + ls_, color="#c44e52", label="presisi " + mode, ms=3)
        axes[0].plot(s_["thr"], s_["recall"], "o" + ls_, color="#4c72b0", label="recall " + mode, ms=3)
    axes[0].axhline(v3["target_precision"], color="k", ls=":", lw=1)
    axes[0].axvline(YTHR, color="r", ls="--", lw=1)
    axes[0].set_xlabel("ambang confidence")
    axes[0].set_title("Segmen evaluasi: presisi & recall")
    axes[0].legend(fontsize=7)
    axes[0].grid(alpha=.3)
    axes[1].hist(YDET["det"]["conf"], bins=50, color="#555")
    axes[1].axvline(YTHR, color="r", ls="--")
    axes[1].set_title("Distribusi confidence deteksi")
    _err = np.array([np.hypot(*(np.array(out_e[:2]) - Batd[f])) for f, out_e in
                     track_fill(cands_e, EVAL_RUNS, Batd, True)[0].items()])
    axes[2].hist(_err, bins=np.linspace(0, 10, 51), color="#555")
    axes[2].axvline(r_m, color="r", ls="--")
    axes[2].set_xlabel("galat posisi (m) vs bola ATD")
    axes[2].set_title("Galat terpilih (segmen evaluasi)")
    gl = np.zeros(N_FRAMES, int)
    for a, b in zip(*run_lengths(np.isnan(BALL_RAW[:, 0]) & GRADED)):
        gl[a:b + 1] = b - a + 1
    bins_g = [0, 10, 25, 50, 100, 250, 10 ** 7]
    names_g = ["<0.4s", "0.4-1s", "1-2s", "2-4s", "4-10s", ">10s"]
    _cut = pd.cut(gl[tgt], bins_g, labels=names_g)
    pd.DataFrame({"v3 lintasan": pd.Series(np.isfinite(YSEL[tgt, 0])).groupby(_cut, observed=False).mean(),
                  "per-frame": pd.Series(np.isin(tgt, list(out_f.keys()))).groupby(_cut, observed=False).mean()}
                 ).plot(kind="bar", ax=axes[3], rot=0, color=["#4c72b0", "#bbbbbb"])
    axes[3].set_ylim(0, 1)
    axes[3].set_title("Frame gap yang terisi, per panjang gap")
    save_fig(fig, "07_yolo_v3_evaluasi", "Evaluasi YOLO v3: ambang presisi, galat, cakupan", group="yolo")

    # --- gambar contoh label ulang (BERISI POTONGAN VIDEO: jangan dibagikan) ---
    tab = YDS.get("table")
    if isinstance(tab, pd.DataFrame) and len(tab) and YDS.get("root") and os.path.isdir(YDS["root"]):
        acc_ = tab[tab["status"] == "diterima"].sort_values("shift_from_atd_px", ascending=False)
        pick = pd.concat([acc_.head(4), acc_.sample(min(4, len(acc_)), random_state=CONFIG["seed"])]) if len(acc_) else acc_
        rej = tab[tab["status"] != "diterima"].head(4)
        shows = [(r, "diterima") for _, r in pick.iterrows()] + [(r, "ditolak") for _, r in rej.iterrows()]
        shows += [(r, "negatif") for _, r in acc_.head(12).iterrows()
                  if os.path.exists(os.path.join(YDS["root"], "images", r["split"], "n{:06d}.jpg".format(int(r["frame"]))))][:3]
        if shows:
            fig, axes = plt.subplots(2, int(np.ceil(len(shows) / 2)), figsize=(2.6 * np.ceil(len(shows) / 2), 5.6))
            for ax in np.atleast_1d(axes).ravel():
                ax.axis("off")
            for ax, (r, kind) in zip(np.atleast_1d(axes).ravel(), shows):
                name = ("n" if kind == "negatif" else "f") + "{:06d}.jpg".format(int(r["frame"]))
                ip = os.path.join(YDS["root"], "images", r["split"], name)
                if kind == "ditolak":
                    ip = os.path.join(YDS_BASE.get("root", "") or os.path.join(DIRS["yolo"], "dataset"), "images",
                                      r["split"], "f{:06d}.jpg".format(int(r["frame"])))
                if not os.path.exists(ip):
                    continue
                im = cv2.cvtColor(read_img(ip), cv2.COLOR_BGR2RGB)
                cx0, cy0 = float(r["atd_u"]), float(r["atd_v"])
                x0_, y0_ = int(max(cx0 - 40, 0)), int(max(cy0 - 40, 0))
                ax.imshow(im[y0_:int(cy0 + 40), x0_:int(cx0 + 40)])
                ax.plot(cx0 - x0_, cy0 - y0_, "r+", ms=10, mew=1.5)
                if kind == "diterima":
                    bw = float(r["box_px"])
                    ax.add_patch(Rectangle((r["u"] - bw / 2 - x0_, r["v"] - bw / 2 - y0_), bw, bw, fill=False,
                                           ec="yellow", lw=1.2))
                ax.set_title("{} {}\n{}".format(kind, name[:-4], r["status"] if kind == "ditolak" else
                                                "geser {:.1f} px".format(r.get("shift_from_atd_px", np.nan))), fontsize=7)
            fig.suptitle("Label ulang v3: + merah = posisi ATD, kotak kuning = label baru (JANGAN DIBAGIKAN)", fontsize=9)
            save_fig(fig, "08_yolo_v3_contoh_label", "Contoh label ulang v3, ditolak, dan negatif - JANGAN DIBAGIKAN",
                     group="yolo")

    _yparams = {"v": 3, "conf_select": YTHR, "max_prior_dist_m": _yc["max_prior_dist_m"],
                "track": _pick(v3, ("track_topk", "track_vmax", "track_tol_m", "track_max_skip", "track_min_len",
                                    "track_min_len_anchored")), "fusion": dict(fz)}
    SOURCES["yolo"] = run_source("yolo", BALL_YOLO, deps=["atd", "yolo3_detect"], extra_params={"yolo": _yparams},
                                 weak_mask=fill)
    _E = SOURCES["yolo"]["epi"]["E"]
    if "weak_frac" in _E.columns:
        print("Episode kontak sumber yolo: {:,} | sebagian besar dari posisi YOLO: {:,} ({:.1%})".format(
            len(_E), int((_E["weak_frac"] > 0.5).sum()), float((_E["weak_frac"] > 0.5).mean())))
    DIAG_YOLO = model_diag_table(SOURCES["yolo"])
    print(DIAG_YOLO.round(4).to_string())
    save_table(DIAG_YOLO, "diagnostik_model_yolo")
    plot_model_diagnostics(SOURCES["yolo"], "yolo")


if YOLO_RT["ok"]:
    try:
        _yolo_block()
    except KeyboardInterrupt:
        raise
    except Exception as _exc:
        if not isinstance(_exc, PipelineError):
            print(explain_error("YOLO v3 (S6)", _exc))
        YOLO_RT.update(ok=False, reason="tahap YOLO gagal ({})".format(getattr(_exc, "short", "{}: {}".format(
            type(_exc).__name__, str(_exc).strip()[:160]))))
        SOURCES.pop("yolo", None)
        print("\n>>> S6 DILEWATI untuk run ini; S1–S5 tetap diproses. Perbaiki penyebab di atas lalu 'Run all' lagi —\n"
              ">>> tahap YOLO yang sudah selesai (dataset/training/deteksi v3) dimuat dari checkpoint.\n")

## 13. Menjalankan semua skenario dan membuat paket submission

Untuk setiap skenario: decoding → `<team_name>.csv` → validator resmi → skor publik & metrik proxy → sapuan
sensitivitas → gambar → **draf dokumen teknis (English)** `<team_name>.md` → `scores.json`.

In [ ]:
def run_sweep(ev):
    rows = []
    for ls, dt in itertools.product(CONFIG["sweep"]["lam_switch"], CONFIG["sweep"]["dead_temp"]):
        lab = decode(ev, lam_switch=ls, dead_temp=dt)
        row = {"lam_switch": ls, "dead_temp": dt}
        row.update(proxy_metrics(lab))
        row["pub_spellF1"] = score_against(GT_PUBLIC, lab)["spellF1"]
        rows.append(row)
    return pd.DataFrame(rows)


def write_submission(path, labels):
    with open(path, "w", newline="\n") as fh:
        fh.write("frame,label\n")
        fh.writelines("{},{}\n".format(i, l) for i, l in enumerate(labels))
    return hashlib.sha256(open(path, "rb").read()).hexdigest()


def scenario_figures(sid, spec, labels, sweep, folder):
    folder = os.path.join(folder, "figures")
    a, b = PUB_A, PUB_B
    fig, ax = plt.subplots(figsize=(15, 3.0))
    rows_rb = [("GT publik", GT_PUB_LAB), (sid, labels), ("baseline_smooth50", BASELINES["baseline_smooth50"])]
    for i, (name, lab) in enumerate(rows_rb):
        ribbon(ax, lab, len(rows_rb) - i, a, b)
    ax.set_xlim(0, (b - a + 1) / FPS)
    ax.set_ylim(.4, len(rows_rb) + .6)
    ax.set_yticks(range(len(rows_rb), 0, -1))
    ax.set_yticklabels([r[0] for r in rows_rb])
    ax.set_xlabel("detik sejak kick-off babak 1")
    ax.set_title("{} {}: 5 menit publik (merah H, biru A, abu D)".format(sid, spec["name"]))
    save_fig(fig, "{}_01_pita_publik".format(sid), "Pita possession 5 menit publik", folder, group=sid)

    fig, axes = plt.subplots(2, 1, figsize=(16, 4.2), constrained_layout=True)
    for ax, (half, a2, b2) in zip(axes, HALF_BLOCKS):
        ribbon(ax, labels, 1, a2, b2)
        sub = SP_TAGS[(SP_TAGS["start"] >= a2) & (SP_TAGS["start"] <= b2)]
        ax.scatter((sub["start"].values - a2) / FPS, np.full(len(sub), 1.55), marker="v", s=12, color="k")
        ax.set_xlim(0, (b2 - a2 + 1) / FPS)
        ax.set_ylim(.5, 1.7)
        ax.set_yticks([])
        ax.set_title("{} — babak {} (▼ = tag SET PIECES)".format(sid, half), fontsize=9)
    axes[-1].set_xlabel("detik sejak awal babak")
    save_fig(fig, "{}_02_timeline_full_match".format(sid), "Label sepanjang pertandingan", folder, group=sid)

    fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
    sp = spells(labels)
    durs = np.array([(e - s + 1) / FPS for s, e, _ in sp]) if sp else np.array([0.04])
    axes[0].hist(durs, bins=np.logspace(np.log10(max(durs.min(), .04)), np.log10(durs.max() + 1), 40), color="#555")
    axes[0].set_xscale("log")
    axes[0].set_xlabel("durasi possession (s, log)")
    axes[0].set_title("Durasi possession (n={})".format(len(sp)))
    for dt in CONFIG["sweep"]["dead_temp"]:
        s = sweep[sweep["dead_temp"] == dt]
        axes[1].plot(s["lam_switch"], s["n_spells"], "o-", label="dead_temp={}".format(dt))
        axes[2].plot(s["lam_switch"], s["team_agree"], "o-", label="dead_temp={}".format(dt))
    for ax, t in ((axes[1], "jumlah possession vs lam_switch"), (axes[2], "kesesuaian tim vs lam_switch")):
        ax.axvline(CONFIG["lam_switch"], color="k", ls=":")
        ax.set_xscale("log")
        ax.set_title(t, fontsize=9)
        ax.grid(alpha=.3)
        ax.legend(fontsize=7)
    save_fig(fig, "{}_03_durasi_dan_sensitivitas".format(sid), "Durasi possession dan sensitivitas decoding",
             folder, group=sid)


def run_scenario(sid, spec):
    folder = os.path.join(DIRS["scenarios"], "{}_{}".format(sid, spec["name"]))
    os.makedirs(folder, exist_ok=True)
    ev = scenario_evidence(spec)
    labels = finalize_labels(decode(ev), ev["src"])
    again = finalize_labels(decode(ev), ev["src"])
    csv_path = os.path.join(folder, "{}.csv".format(CONFIG["team_name"]))
    sha = retry_io(lambda: write_submission(csv_path, labels), "menulis CSV submission")
    val = subprocess.run([sys.executable, VALIDATOR, csv_path], capture_output=True, text=True)
    retry_io(lambda: write_text(os.path.join(folder, "validation.txt"), val.stdout + val.stderr), "menulis validation.txt")
    valid_ok = val.returncode == 0 and "Valid" in val.stdout
    pub = score_against(GT_PUBLIC, labels, CONFIG["bootstrap_n"])
    prox = proxy_metrics(labels)
    sweep = run_sweep(ev)
    save_table(sweep, "{}_sweep".format(sid), folder)
    scenario_figures(sid, spec, labels, sweep, folder)
    R = SOURCES[spec["source"]]
    res = {"spec": spec, "folder": folder, "csv": csv_path, "sha256": sha, "valid": valid_ok,
           "deterministic": bool(np.array_equal(labels, again)), "public": pub, "proxy": prox,
           "m1": R["m1"][spec["model"]]["metrics"], "m2": R["m2"][spec["model"]]["metrics"],
           "weights": {"m1": R["m1"][spec["model"]].get("weights"), "m2": R["m2"][spec["model"]].get("weights")},
           "label_counts": pd.Series(labels).value_counts().to_dict(), "labels": labels,
           "sweep_n_spells_range": [int(sweep["n_spells"].min()), int(sweep["n_spells"].max())],
           "yolo": YOLO_RESULT["eval"] if spec["source"] == "yolo" and YOLO_RESULT else None}
    if GT_EXTRA is not None:
        res["extra"] = score_against(GT_EXTRA, labels)
    print("valid={} | deterministik={} | publik spellF1={:.4f} macroF1={:.4f} | possession={} | team_agree={:.4f}".format(
        valid_ok, res["deterministic"], pub["spellF1"], pub["macroF1"], prox["n_spells"], prox["team_agree"]))
    if GT_EXTRA is not None:
        print("  [uji] GT tambahan: spellF1={:.4f} macroF1={:.4f}".format(res["extra"]["spellF1"], res["extra"]["macroF1"]))
    return res


SCENARIO_RESULTS = {}
for sid, spec in CONFIG["scenarios"].items():
    print("\n" + "=" * 70 + "\n{} — {}  (sumber: {}, model: {})".format(sid, spec["name"], spec["source"], spec["model"]))
    if spec["source"] not in SOURCES:
        reason = YOLO_RT.get("reason") if spec["source"] == "yolo" else "sumber tidak tersedia"
        print("DILEWATI:", reason)
        SCENARIO_RESULTS[sid] = {"spec": spec, "skipped": reason}
        continue
    try:
        SCENARIO_RESULTS[sid] = run_scenario(sid, spec)
    except KeyboardInterrupt:
        raise
    except Exception as _exc:
        print(explain_error("skenario {} ({})".format(sid, spec["name"]), _exc))
        SCENARIO_RESULTS[sid] = {"spec": spec, "skipped": "error: {}: {}".format(type(_exc).__name__, str(_exc).strip()[:120])}

### 13.1 Draf dokumen teknis per skenario
Dokumen (English, sesuai kebutuhan `SUBMISSION.md`: approach, assets used/dropped, possession rules, limitations,
reproduction) dibuat otomatis dari konfigurasi dan hasil run. **Periksa dan lengkapi** sebelum dikirim
(misalnya URL repo publik dan nama anggota tim), lalu ekspor ke PDF bila perlu.

In [ ]:
def fmt(x, n=4):
    try:
        return "{:.{}f}".format(float(x), n)
    except Exception:
        return str(x)


def md_table(df):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join([""] + cols) + " |", "|" + "---|" * (len(cols) + 1)]
    for idx, row in df.iterrows():
        lines.append("| " + " | ".join([str(idx)] + [fmt(v) if isinstance(v, (float, np.floating)) else str(v)
                                                     for v in row.values]) + " |")
    return "\n".join(lines)


def tech_doc(sid, res):
    spec = res["spec"]
    model_name = FAMILY_NAMES.get(spec["model"], spec["model"])
    members = ", ".join(FAMILY_NAMES[m] for m in CONFIG["ensemble_members"])
    model_desc = ("an ensemble of {} whose out-of-fold probabilities are combined on the logit scale, either by a "
                  "logistic-regression meta-learner (stacking) or by a plain average, whichever has the lower "
                  "out-of-fold log-loss (chosen: M1 {}, M2 {})".format(
                      members, SOURCES[spec["source"]]["m1"]["ensemble"]["method"],
                      SOURCES[spec["source"]]["m2"]["ensemble"]["method"]) if spec["model"] == "ensemble"
                  else "a {} classifier".format(model_name))
    y = res.get("yolo")
    c = CONFIG
    pub, prox = res["public"], res["proxy"]
    yolo_section = ""
    if spec["source"] == "yolo" and y:
        rk = [k for k in y if k.startswith("recall_top")][0]
        pk = [k for k in y if k.startswith("precision_top")][0]
        sm = y.get("relabel") or {}
        cal = y.get("calibration") or {}
        tch = y.get("teacher") or {}
        fz = c["yolo_fusion"]
        v3 = c["yolo_v3"]
        yolo_section = """
### Stage 0 — Ball detection from the tactical video (YOLO, version 3)
The automatic tracking misses the ball on 41% of graded frames, so this scenario adds a single-class detector
(`{base}`, Ultralytics, AGPL-3.0). All training labels are generated **automatically**; no frame was labelled by hand.

*Pseudo-labels.* On frames where the tracking sees a low-speed (ground) ball, its pitch position is projected to pixels
with the provided homography and a {crop}x{crop} native-resolution crop is cut ({ncrop} crops; validation crops come
from separate 2-minute blocks). Because the tracked position is not always exactly on the ball, a first detector trained
on these raw labels ({tmap}) is used as a *teacher*: it searches for the ball within {rr} m of the tracked position
(confidence >= {rc}); the centre is refined with a Difference-of-Gaussians blob detector at the expected ball size and
kept only if the blob is brighter than its surrounding ring (contrast >= {mc}). Crops without a confirmed ball are
discarded ({acc} of {ncrop} kept; median correction of the centre {shift:.1f} px). The apparent ball size is calibrated
from one measured ball diameter ({bref:.0f} px near the camera) divided by the local homography scale, giving an effective
diameter of {deff:.3f} m (motion blur and compression make the ball look larger than 0.22 m); each box is the apparent
diameter plus {mg} px per side, at least {mb} px (median {box:.1f} px). Hard negatives are added by removing the ball
from {nneg} accepted crops with inpainting (empty labels), so that feet and grass without a ball are explicitly negative.

*Inference and selection.* The detector runs on full frames at `imgsz={imgsz}` on every graded frame whose tracked ball
is missing, and on {nseg} evaluation segments of {sl} frames (2 s) in held-out blocks where the tracking does see the
ball; these segments are treated exactly like gaps. Detections are mapped back to the pitch with the inverse homography
and must lie within {mpd} m of the interpolation between the tracked positions at both ends of the gap. They are then
linked into tracks (step <= {vmax:.0f} m/s x dt + {tol} m, up to {skip} skipped frames); a track is kept if it has at least
{tml} detections, or at least {tma} if it connects to the tracked ball at either end of the gap. The confidence threshold
({cs:.2f}) is the smallest one reaching a precision of {tp:.2f} on the evaluation segments. There, the selected positions
are within {mr} m of the tracked ball with precision {prec:.1%} and recall {rec:.1%} (frame-wise selection without
tracks: precision {pfp:.1%}, recall {pfr:.1%}). The detector fills {fill:.1%} of the missing-ball frames.

*Fusion.* Detector positions are treated as weaker evidence than tracked positions: contact evidence from frames whose
ball comes mostly from the detector is weighted by {ew}, control by dwell needs {tdw} frames instead of {td}, and the
gap model M1 is trained only on tracked-ball frames (the detector only shortens the gaps it has to bridge).
""".format(base=y["base_weights"], crop=c["yolo"]["crop"], ncrop=sm.get("crop_dasar", "n/a"),
           tmap=("mAP50 {:.3f}".format(tch["map50"]) if tch.get("map50") is not None else "previous version"),
           rr=v3["relabel_radius_m"], rc=v3["relabel_conf"], mc=v3["relabel_min_contrast"], acc=sm.get("diterima", "n/a"),
           shift=float(sm.get("geser_dari_atd_px_median", float("nan"))), bref=float(cal.get("ball_px_ref", float("nan"))),
           deff=float(cal.get("eff_ball_diam_m", float("nan"))), mg=v3["box_margin_px"], mb=v3["min_box_px"],
           box=float(y.get("box_px_median") or float("nan")), nneg=sum((sm.get("negatif") or {}).values()),
           imgsz=y["imgsz_infer"], nseg=y["eval_segments"], sl=v3["eval_seg_len"], mpd=c["yolo"]["max_prior_dist_m"],
           vmax=v3["track_vmax"], tol=v3["track_tol_m"], skip=v3["track_max_skip"], tml=v3["track_min_len"],
           tma=v3["track_min_len_anchored"], cs=float(y.get("conf_select_used", float("nan"))),
           tp=v3["target_precision"], mr=c["yolo"]["match_radius_m"], prec=y[pk], rec=y[rk],
           pfp=y.get("per_frame_precision", float("nan")), pfr=y.get("per_frame_recall", float("nan")),
           fill=y["target_filled_frac"], ew=fz["evidence_weight"], tdw=fz["t_dwell_weak"], td=c["t_dwell"])
    w = res["weights"]
    weights_txt = ("\nEnsemble weights (logit scale): M1 {}, M2 {}.\n".format(w["m1"], w["m2"])
                   if spec["model"] == "ensemble" else "")
    doc = """# {team} — More than a Hack 2027, Phase 1 (Ball Possession)
## Technical document — scenario {sid}: {name}

**Submission file:** `{team}.csv` (SHA-256 `{sha}`)
**Code:** single Jupyter notebook `MTAH2027_Possession_Scenarios_v3.ipynb` in our public repository (URL: _to be filled_)

## 1. Approach
We treat possession as a sequence-labelling problem with three states (H, A, D) and decode it globally with a
Viterbi algorithm, because the ranking metric (`spellF1`) scores whole possessions rather than frames. Frame-level
evidence comes from football rules (IFAB Laws 8, 9, 13–17), from self-supervised models trained on the tracking data
itself, and from weak labels derived from the provided Smart Tagging. No possession label was ever produced by hand,
and the public 5-minute ground truth was used only to check that the pipeline is correct, never to choose parameters.
In this scenario the learned components use {model_desc}.
{yolo_section}
### Stage A — Ball pre-processing
Ball samples farther than {bmo} m outside the pitch, isolated jumps faster than {bms} m/s, and detections shorter
than {bmr} frames are removed; continuous runs are smoothed with a Savitzky–Golay filter (window {sgw}, order {sgo}).

### Stage B — Who is closest to the ball when it is missing (model M1, self-supervised)
On frames where the ball is visible we cut artificial gaps whose lengths follow the real gap distribution, and train the
model to predict which team has the player closest to the ball (within {gcd} m) from the ball positions at both ends
of the gap and the configuration of all tracked players (team centroids, spreads, fastest players, densest cluster,
distances to the interpolated ball). Gaps of up to {sgm} frames are linearly interpolated. Validation uses
{k}-fold GroupKFold over 2-minute blocks of the whole match. Out-of-fold: accuracy {m1acc}, AUC {m1auc},
log-loss {m1ll} (n = {m1n}).

### Stage C — Contact and control
A contact is a frame where the nearest player is within {rpz} m of a reliably located ball; a duel is when players of
both teams are within {rdz} m (possession then stays with the previous team). Consecutive contacts form episodes.
An episode counts as **control** if it lasts at least {td} frames ({tds:.2f} s, "dwell"), or if the next team touch
within {pw} frames ({pws:.1f} s) is by a team-mate ("chain"). Unconfirmed touches (deflections) carry weak evidence.

### Stage D — Ball out of play (model M2, weakly supervised)
Dead-ball features follow the Laws of the Game: ball beyond the lines, collective slow-down of players (with a
{dls:.1f}-s lead, since players react after the whistle and after the restart kick), ball not tracked for long,
throw-in takers beyond the touchline, players at corner arcs, kick-off formation. Weak labels: the first {dwl} s of
every `SET PIECES` tag are "dead", the interior of possession phases is "alive", the time between a `GOALS` tag and the
following kick-off tag is "dead", and frames 110,808–113,741 (before the second-half kick-off, published) are dead.
Additional features describe the ball being stationary before a restart and the distance of the last ball position
to restart spots (corner arcs, goal area, touchlines, centre spot). Out-of-fold AUC against the weak labels {m2auc} (log-loss {m2ll}).
{weights_txt}
### Stage E — Decoding
Team log-odds = control evidence + {gw} × M1 log-odds inside long gaps; dead log-odds = {dt} × logit(M2).
Transition penalties: H↔A {ls}, alive→dead {ltd}, dead→alive {lfd}. A dead-ball stretch must last at least {mds} s
(a restart needs the ball to be placed), enforced exactly with a semi-Markov Viterbi in which the dead state is a
chain of {mdf} frames. Because Viterbi sees the whole half, possession boundaries are placed at the first touch of the
new team rather than when control is confirmed.

## 2. Assets used and dropped
| Asset | Use |
|---|---|
| Tactical ATD (players + ball) | all stages |
| Smart Tagging | weak labels for M2; independent consistency check of team labels |
| Homography (tactical) | {hom_use} |
| Tactical video | {vid_use} |
| Broadcast video / ATD | **dropped** — not synchronised to the tactical timeline |

## 3. Possession rules implemented
- A change of possession requires **control** by the other team (dwell or chain); single touches, deflections and
  duels keep the previous possessor.
- Dead ball (`D`) is never a possession; after a restart, the team in possession is decided by the next controlled
  touch in the tracking, not copied from the Smart Tagging.
- Loose and contested balls stay with the last controlling team.
- Ungraded frames (before kick-off, half-time, after the final whistle) are written as `X`.

## 4. Results available before submission
Public 5-minute window (sanity check only; 17 possessions, differences < 0.04 are noise): spellF1 {pspell}
(90% bootstrap {plo}–{phi}), macroF1 {pmacro}, {pn} predicted vs {pgt} true possessions.
Full-match proxies: {nsp} possessions ({spm:.2f} per minute of live ball, median {msd:.1f} s), team agreement with
Smart Tagging phases {ta}, set-piece restarts recovered {rh}, dead-ball share {ds}.

## 5. Limitations
- M2 learns from automatically generated tags, so systematic tagging errors propagate to dead-ball detection.
- Very long ball gaps (> 10 s) remain hard; M1 accuracy drops with gap length.
- Control thresholds were set from football reasoning and literature, not tuned on hidden labels; they may not match
  the exact ownership rule of the corrected tracking used for the ground truth.
{yolo_lim}
- With more time: event-type-specific restart models (throw-in, corner, goal kick with last-touch rules) and a
  sequence model learned end-to-end on top of the current features.

## 6. Reproduction
1. Put the official data bundle in `data/tactical/` (or set `data_dir`).
2. Install the pinned versions (`requirements_used.txt` in the run folder); Python {py}.
3. Run all cells of the notebook with `scenarios` containing `{sid}`; the file is written to
   `<work_dir>/{rdir}/<RUN_ID>/scenarios/{sid}_{name}/{team}.csv`.

Seed {seed}, `n_threads` = {nt}, GBDT device `{gdev}`, official repository commit `{commit}`.
Library versions: numpy {np_v}, pandas {pd_v}, scikit-learn {sk_v}, LightGBM {lgb_v}, XGBoost {xgb_v},
CatBoost {cat_v}{torch_v}. Re-running with identical versions and thread count reproduces the same file
(checked by SHA-256 within our runs){gpu_note}.

## References
- IFAB, *Laws of the Game 2025/26*, Laws 8 and 9 (The FA edition).
- Daniel Link and Martin Hoernig, "Individual ball possession in soccer," *PLoS ONE*, vol. 12, no. 7, e0179953, 2017,
  doi:10.1371/journal.pone.0179953.
- Ferran Vidal-Codina, Nicolas Evans, Bahaeddine El Fakir and Johsan Billingham, "Automatic event detection in football
  using tracking data," *Sports Engineering*, vol. 25, no. 1, art. 18, 2022, doi:10.1007/s12283-022-00381-6.
""".format(
        team=c["team_name"], sid=sid, name=spec["name"], sha=res["sha256"], model_desc=model_desc,
        rdir=c["results_dirname"],
        yolo_section=yolo_section, bmo=c["ball_max_outside"], bms=c["ball_max_speed"], bmr=c["ball_min_run"],
        sgw=c["savgol_window"], sgo=c["savgol_order"], gcd=c["gap_cls_max_dist"], sgm=c["short_gap_max"],
        k=c["cv_folds"], m1acc=fmt(res["m1"]["acc"]), m1auc=fmt(res["m1"]["auc"]), m1ll=fmt(res["m1"]["logloss"]),
        m1n=res["m1"]["n"], rpz=c["r_pz"], rdz=c["r_dz"], td=c["t_dwell"], tds=c["t_dwell"] / FPS,
        pw=c["pass_window"], pws=c["pass_window"] / FPS, dls=c["dead_lead_s"], dwl=c["dead_weak_len_s"],
        m2auc=fmt(res["m2"]["auc"]), m2ll=fmt(res["m2"]["logloss"]), weights_txt=weights_txt,
        gw=c["gap_weight"], dt=c["dead_temp"], ls=c["lam_switch"], ltd=c["lam_to_dead"], lfd=c["lam_from_dead"],
        mds=c["min_dead_s"], mdf=int(round(c["min_dead_s"] * FPS)),
        hom_use=("pixel ↔ pitch mapping for YOLO pseudo-labels and detections" if spec["source"] == "yolo"
                 else "error analysis only"),
        vid_use=("ball detection (YOLO)" if spec["source"] == "yolo" else "not used in this scenario"),
        pspell=fmt(pub["spellF1"]), plo=fmt(pub["spell_lo"]), phi=fmt(pub["spell_hi"]), pmacro=fmt(pub["macroF1"]),
        pn=pub["n_pred_spells"], pgt=pub["n_gt_spells"], nsp=prox["n_spells"], spm=prox["spells_per_min_alive"],
        msd=prox["median_spell_s"], ta=fmt(prox["team_agree"]), rh=fmt(prox["restart_hit"]), ds=fmt(prox["dead_share"]),
        yolo_lim=("- YOLO positions assume the ball is on the ground (homography); aerial balls are displaced, and "
                  "occluded balls stay undetected. The evaluation segments use frames where the tracking sees the ball, "
                  "which are easier than real gaps, so precision inside real gaps is likely lower."
                  if spec["source"] == "yolo" else ""),
        py=platform.python_version(), seed=c["seed"], nt=c["n_threads"], gdev=c["gbdt_device"], commit=OFFICIAL_COMMIT,
        np_v=np.__version__, pd_v=pd.__version__, sk_v=sklearn.__version__, lgb_v=lgb.__version__,
        xgb_v=xgb.__version__, cat_v=catboost.__version__,
        torch_v=(", PyTorch {}, Ultralytics {}".format(YOLO_RT["torch"], YOLO_RT["ultralytics"])
                 if spec["source"] == "yolo" and YOLO_RT["ok"] else ""),
        gpu_note=("; the YOLO weights (SHA-256 `{}`) are stored with the run because GPU training is not bit-exact "
                  "across hardware".format(y["weights_sha256"]) if spec["source"] == "yolo" and y else ""))
    return doc


for sid, res in SCENARIO_RESULTS.items():
    if res.get("skipped"):
        continue
    doc = tech_doc(sid, res)
    md_path = os.path.join(res["folder"], "{}.md".format(CONFIG["team_name"]))
    retry_io(lambda: write_text(md_path, doc), "menulis dokumen teknis " + sid)
    res["doc"] = md_path
    slim = {k: v for k, v in res.items() if k not in ("labels",)}
    retry_io(lambda: write_json(os.path.join(res["folder"], "scores.json"), slim), "menulis scores.json " + sid)
    print("{}: {}".format(sid, md_path))

## 14. Laporan transisi resmi (matched / missed / spurious)
Scorer resmi dijalankan pada semua skenario + baseline terhadap 5 menit publik; laporan per transisi tersimpan
di `comparison/official_eval/reports/` untuk analisis kesalahan.

In [ ]:
_eval_dir = os.path.join(DIRS["comparison"], "official_eval")
os.makedirs(os.path.join(_eval_dir, "results"), exist_ok=True)
for sid, res in SCENARIO_RESULTS.items():
    if not res.get("skipped"):
        shutil.copy2(res["csv"], os.path.join(_eval_dir, "results", "{}_{}.csv".format(sid, res["spec"]["name"])))
for name, lab in BASELINES.items():
    write_submission(os.path.join(_eval_dir, "results", name + ".csv"), lab)
_r = subprocess.run([sys.executable, os.path.join(OFFICIAL_DIR, "evaluation", "run_evaluation.py"), "--results",
                     os.path.join(_eval_dir, "results"), "--out", _eval_dir, "--no-detail"], capture_output=True, text=True)
print(_r.stdout[-3000:] or _r.stderr[-3000:])

## 15. Perbandingan skenario

### Cara membaca dan memilih
Tidak ada leaderboard dan hanya satu file yang dinilai, jadi pilih berdasarkan bukti yang **dihitung pada seluruh
pertandingan**, dengan urutan prioritas:
1. **M1 log-loss** (lebih kecil lebih baik) dan **M2 AUC** (lebih besar lebih baik) — out-of-fold, paling andal.
   *Catatan v3:* M1 pada S6 kini dilatih hanya dari frame bola ATD (`m1_train_on_atd_only`), sehingga angkanya lebih
   setara dengan S1–S5 daripada di v2 (v2: AUC 0,976 karena gap buatan lebih mudah).
2. **Kesesuaian tim** dengan Smart Tagging (lebih tinggi lebih baik) dan **restart terdeteksi**.
3. **Laju possession** yang wajar dan stabil terhadap `lam_switch` (lihat gambar sensitivitas tiap skenario).
4. **Skor 5 menit publik** hanya sebagai pemeriksaan: skenario yang jauh di bawah baseline di sini patut dicurigai,
   tetapi selisih kecil tidak berarti apa-apa.

Kolom `rank_core` = rata-rata peringkat pada M1 log-loss, M2 AUC, dan kesesuaian tim. Ini **alat bantu**, bukan aturan.
Matriks kesepakatan menunjukkan seberapa mirip dua skenario (spellF1 satu skenario terhadap yang lain): bila semua
di atas ~0,9, pilihan antar-skenario tersebut hampir tidak berpengaruh.

In [ ]:
rows = {}
for sid, res in SCENARIO_RESULTS.items():
    spec = res["spec"]
    if res.get("skipped"):
        rows[sid] = {"nama": spec["name"], "status": "dilewati: " + str(res["skipped"])}
        continue
    r = {"nama": spec["name"], "status": "ok" if res["valid"] else "TIDAK VALID",
         "M1_logloss": res["m1"]["logloss"], "M1_auc": res["m1"]["auc"], "M1_acc": res["m1"]["acc"],
         "M2_auc_weak": res["m2"]["auc"], "M2_logloss_weak": res["m2"]["logloss"],
         "team_agree": res["proxy"]["team_agree"], "restart_hit": res["proxy"]["restart_hit"],
         "n_spells": res["proxy"]["n_spells"], "spells_per_min": res["proxy"]["spells_per_min_alive"],
         "median_spell_s": res["proxy"]["median_spell_s"], "dead_share": res["proxy"]["dead_share"],
         "pub_spellF1": res["public"]["spellF1"], "pub_spell_lo": res["public"]["spell_lo"],
         "pub_spell_hi": res["public"]["spell_hi"], "pub_macroF1": res["public"]["macroF1"],
         "pub_transF1": res["public"]["trans_f1_25"], "pub_trans_match": res["public"]["trans_matched_25"],
         "pub_latency_med": res["public"]["latency_median_25"],
         "deterministic": res["deterministic"], "sha256": res["sha256"][:12]}
    if res.get("yolo"):
        r["yolo_recall"] = [v for k, v in res["yolo"].items() if k.startswith("recall_top")][0]
        r["yolo_precision"] = [v for k, v in res["yolo"].items() if k.startswith("precision_top")][0]
        r["yolo_fill"] = res["yolo"]["target_filled_frac"]
    if "extra" in res:
        r["UJI_extra_spellF1"] = res["extra"]["spellF1"]
        r["UJI_extra_macroF1"] = res["extra"]["macroF1"]
    rows[sid] = r
for name in ("baseline_smooth50",):
    rows["baseline"] = {"nama": "baseline resmi", "status": "referensi", "team_agree": BASE_PROXY[name]["team_agree"],
                  "restart_hit": BASE_PROXY[name]["restart_hit"], "n_spells": BASE_PROXY[name]["n_spells"],
                  "spells_per_min": BASE_PROXY[name]["spells_per_min_alive"],
                  "median_spell_s": BASE_PROXY[name]["median_spell_s"], "dead_share": BASE_PROXY[name]["dead_share"],
                  "pub_spellF1": BASE_SCORES[name]["spellF1"], "pub_spell_lo": BASE_SCORES[name]["spell_lo"],
                  "pub_spell_hi": BASE_SCORES[name]["spell_hi"], "pub_macroF1": BASE_SCORES[name]["macroF1"]}
    if GT_EXTRA is not None:
        sx = score_against(GT_EXTRA, BASELINES[name])
        rows["baseline"].update({"UJI_extra_spellF1": sx["spellF1"], "UJI_extra_macroF1": sx["macroF1"]})
COMP = pd.DataFrame(rows).T
for _c in COMP.columns:
    try:
        COMP[_c] = pd.to_numeric(COMP[_c])
    except (ValueError, TypeError):
        pass
ok_rows = [s for s in SCENARIO_RESULTS if not SCENARIO_RESULTS[s].get("skipped")]
if ok_rows:
    sub = COMP.loc[ok_rows]
    ranks = pd.concat([sub["M1_logloss"].astype(float).rank(), (-sub["M2_auc_weak"].astype(float)).rank(),
                       (-sub["team_agree"].astype(float)).rank()], axis=1)
    COMP.loc[ok_rows, "rank_core"] = ranks.mean(axis=1)
save_table(COMP, "comparison_table", DIRS["comparison"])
show_cols = [c for c in ["nama", "status", "M1_logloss", "M1_auc", "M2_auc_weak", "team_agree", "restart_hit", "n_spells",
                         "median_spell_s", "pub_spellF1", "pub_spell_lo", "pub_spell_hi", "pub_macroF1", "pub_transF1",
                         "pub_trans_match", "pub_latency_med", "yolo_precision", "yolo_recall", "yolo_fill", "rank_core",
                         "UJI_extra_spellF1"] if c in COMP.columns]
print(COMP[show_cols].to_string(float_format=lambda v: "{:.4f}".format(v)))

In [ ]:
fig_dir = os.path.join(DIRS["comparison"], "figures")
metrics_plot = [("M1_logloss", "M1 log-loss (↓)"), ("M2_auc_weak", "M2 AUC vs label lemah (↑)"),
                ("team_agree", "kesesuaian tim (↑)"), ("restart_hit", "restart terdeteksi (↑)"),
                ("n_spells", "jumlah possession"), ("pub_spellF1", "spellF1 publik (cek saja)")]
fig, axes = plt.subplots(2, 3, figsize=(17, 7))
for ax, (col, title) in zip(axes.ravel(), metrics_plot):
    s = COMP[col].dropna().astype(float) if col in COMP.columns else pd.Series(dtype=float)
    ax.bar(range(len(s)), s.values, color=[SC_COLORS[i % len(SC_COLORS)] for i in range(len(s))])
    ax.set_xticks(range(len(s)))
    ax.set_xticklabels(s.index, rotation=30, fontsize=8)
    if col == "pub_spellF1" and len(s):
        lo = COMP.loc[s.index, "pub_spell_lo"].astype(float).values
        hi = COMP.loc[s.index, "pub_spell_hi"].astype(float).values
        ax.errorbar(range(len(s)), s.values, yerr=[np.clip(s.values - lo, 0, None), np.clip(hi - s.values, 0, None)],
                    fmt="none", ecolor="k", capsize=3)
    if len(s) and col not in ("n_spells",):
        rng_ = s.max() - s.min()
        ax.set_ylim(max(0, s.min() - rng_ * 1.5 - 1e-3), s.max() + rng_ * 0.5 + 1e-3)
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=.3, axis="y")
save_fig(fig, "20_perbandingan_metrik", "Perbandingan metrik utama antar-skenario", fig_dir, group="perbandingan")

ids = ok_rows
AGREE = pd.DataFrame(index=ids, columns=ids, dtype=float)
for i in ids:
    gi = pm.label_codes(SCENARIO_RESULTS[i]["labels"])
    for j in ids:
        AGREE.loc[i, j] = 1.0 if i == j else pm.evaluate(gi, pm.label_codes(SCENARIO_RESULTS[j]["labels"]))["spell_f1_mean"]
save_table(AGREE, "agreement_spellF1", DIRS["comparison"])
if len(ids) > 1:
    fig, ax = plt.subplots(figsize=(1.2 * len(ids) + 3, 1.0 * len(ids) + 2))
    im = ax.imshow(AGREE.values.astype(float), vmin=min(0.5, np.nanmin(AGREE.values.astype(float))), vmax=1, cmap="viridis")
    ax.set_xticks(range(len(ids)))
    ax.set_xticklabels(ids)
    ax.set_yticks(range(len(ids)))
    ax.set_yticklabels(ids)
    for a_ in range(len(ids)):
        for b_ in range(len(ids)):
            _v = float(AGREE.values[a_, b_])
            ax.text(b_, a_, "{:.2f}".format(_v), ha="center", va="center", color="k" if _v > 0.85 else "w", fontsize=8)
    fig.colorbar(im, ax=ax)
    ax.set_title("Kesepakatan antar-skenario (spellF1 kolom terhadap baris)")
    save_fig(fig, "21_kesepakatan_skenario", "Matriks kesepakatan spellF1 antar-skenario", fig_dir, group="perbandingan")

fig, ax = plt.subplots(figsize=(15, 0.55 * (len(ids) + 2) + 1))
rows_rb = [("GT publik", GT_PUB_LAB)] + [(s, SCENARIO_RESULTS[s]["labels"]) for s in ids] + \
          [("baseline_smooth50", BASELINES["baseline_smooth50"])]
for i, (name, lab) in enumerate(rows_rb):
    ribbon(ax, lab, len(rows_rb) - i, PUB_A, PUB_B)
ax.set_xlim(0, (PUB_B - PUB_A + 1) / FPS)
ax.set_ylim(.4, len(rows_rb) + .6)
ax.set_yticks(range(len(rows_rb), 0, -1))
ax.set_yticklabels([r[0] for r in rows_rb])
ax.set_xlabel("detik sejak kick-off babak 1")
ax.set_title("5 menit publik: semua skenario (merah H, biru A, abu D)")
save_fig(fig, "22_pita_publik_semua_skenario", "Pita possession semua skenario", fig_dir, group="perbandingan")

if ids:
    fig, ax = plt.subplots(figsize=(10, 4))
    data = [[(e - s + 1) / FPS for s, e, _ in spells(SCENARIO_RESULTS[i]["labels"])] for i in ids]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(ids) + 1))
    ax.set_xticklabels(ids)
    ax.set_yscale("log")
    ax.set_ylabel("durasi possession (s, log)")
    ax.set_title("Distribusi durasi possession per skenario")
    save_fig(fig, "23_durasi_possession_semua", "Durasi possession per skenario", fig_dir, group="perbandingan")

### 15.1 Perbandingan dengan run sebelumnya
Tabel `vs_previous.csv` membandingkan skenario yang sama antara run ini dan run terbaru di `runs_v2/`
(`compare_prefer_dirname`). Kolom `Δ` = nilai run ini − run v2. **S1–S5 seharusnya Δ = 0** (dimuat dari cache v2);
yang relevan adalah baris **S6**.

In [ ]:
def find_previous_comparison():
    opt = CONFIG["compare_with_previous"]
    if not opt:
        return None
    if opt != "auto":
        cand = os.path.join(opt, "comparison", "comparison_table.csv") if os.path.isdir(opt) else opt
        return cand if os.path.exists(cand) else None
    here = os.path.normcase(os.path.abspath(DIRS["run"]))
    prefer = CONFIG.get("compare_prefer_dirname")
    cands = []
    for pth in glob.glob(os.path.join(WORK_DIR, "*", "*", "comparison", "comparison_table.csv")):
        run_dir = os.path.normcase(os.path.abspath(os.path.dirname(os.path.dirname(pth))))
        if run_dir != here:
            try:
                n_ok = int((pd.read_csv(pth, index_col=0)["status"] == "ok").sum())
            except Exception:
                n_ok = 0
            pref = int(bool(prefer) and os.path.basename(os.path.dirname(run_dir)) == os.path.normcase(prefer))
            cands.append((pref, n_ok, os.path.getmtime(pth), pth))   # folder pilihan, lalu skenario selesai terbanyak
    return sorted(cands)[-1][-1] if cands else None


PREV_PATH = find_previous_comparison()
VS_PREV = None
if PREV_PATH:
    print("Run pembanding:", os.path.dirname(os.path.dirname(PREV_PATH)))
    PREV = pd.read_csv(PREV_PATH, index_col=0)
    cols = [c for c in ["pub_spellF1", "pub_macroF1", "pub_transF1", "M2_auc_weak", "restart_hit", "team_agree",
                        "n_spells", "median_spell_s", "yolo_precision", "yolo_recall", "yolo_fill"]
            if c in PREV.columns and c in COMP.columns]
    common = [i for i in COMP.index if i in PREV.index]
    rows_vs = {}
    for i in common:
        r = {}
        for c in cols:
            a_, b_ = pd.to_numeric(COMP.loc[i, c], errors="coerce"), pd.to_numeric(PREV.loc[i, c], errors="coerce")
            r[c + "_lama"], r[c + "_baru"], r[c + "_Δ"] = b_, a_, a_ - b_
        rows_vs[i] = r
    VS_PREV = pd.DataFrame(rows_vs).T
    save_table(VS_PREV, "vs_previous", DIRS["comparison"])
    print(VS_PREV[[c for c in VS_PREV.columns if c.endswith("_Δ")]].to_string(float_format=lambda v: "{:+.4f}".format(v)))
    plot_cols = [c for c in ["pub_spellF1", "pub_transF1", "M2_auc_weak", "restart_hit"] if c + "_baru" in VS_PREV.columns]
    if plot_cols and len(common):
        fig, axes = plt.subplots(1, len(plot_cols), figsize=(4.2 * len(plot_cols), 3.6))
        for ax, c in zip(np.atleast_1d(axes), plot_cols):
            x_ = np.arange(len(common))
            ax.bar(x_ - 0.2, VS_PREV[c + "_lama"].astype(float), 0.4, label="sebelumnya", color="#bbbbbb")
            ax.bar(x_ + 0.2, VS_PREV[c + "_baru"].astype(float), 0.4, label="run ini", color="#4c72b0")
            ax.set_xticks(x_)
            ax.set_xticklabels(common, rotation=30, fontsize=8)
            ax.set_title(c, fontsize=10)
            ax.grid(alpha=.3, axis="y")
        np.atleast_1d(axes)[0].legend(fontsize=8)
        save_fig(fig, "24_vs_run_sebelumnya", "Perbandingan metrik dengan run sebelumnya",
                 os.path.join(DIRS["comparison"], "figures"), group="perbandingan")
else:
    print("Tidak ada run sebelumnya untuk dibandingkan.")

In [ ]:
readme = ["# Perbandingan skenario — run {}".format(RUN_ID), "",
          "Tim: **{}** | commit repo resmi `{}`".format(CONFIG["team_name"], OFFICIAL_COMMIT), "",
          md_table(COMP[show_cols]), "", "## Kesepakatan antar-skenario (spellF1)", "",
          md_table(AGREE) if len(ids) else "-", "",
          "## Dibanding run sebelumnya", "", "Pembanding: `{}`".format(PREV_PATH) if PREV_PATH else "-", "",
          md_table(VS_PREV[[c for c in VS_PREV.columns if c.endswith("_Δ")]]) if VS_PREV is not None else "-", "",
          "## Cara memilih", "",
          "1. Utamakan M1 log-loss rendah dan M2 AUC tinggi (out-of-fold, seluruh pertandingan).",
          "2. Lalu kesesuaian tim dan restart terdeteksi yang tinggi.",
          "3. Laju possession yang wajar dan stabil di sapuan sensitivitas.",
          "4. Skor 5 menit publik hanya untuk memastikan tidak ada yang rusak (selisih < 0,04 = noise).", "",
          "Paket setiap skenario ada di `scenarios/<ID>_<nama>/` berisi `{0}.csv`, `{0}.md`, `validation.txt`, "
          "`scores.json`, dan `figures/`.".format(CONFIG["team_name"])]
open(os.path.join(DIRS["comparison"], "README.md"), "w", encoding="utf-8").write("\n".join(readme))
print("Ringkasan perbandingan:", os.path.join(DIRS["comparison"], "README.md"))

if CONFIG["submit_scenario"]:
    sid = CONFIG["submit_scenario"]
    res = SCENARIO_RESULTS.get(sid)
    if res and not res.get("skipped") and res["valid"]:
        final = os.path.join(WORK_DIR, "submission_final")
        os.makedirs(final, exist_ok=True)
        for p in (res["csv"], res["doc"], os.path.join(res["folder"], "validation.txt")):
            shutil.copy2(p, os.path.join(final, os.path.basename(p)))
        print("Skenario {} disalin ke {}".format(sid, final))
    else:
        print("submit_scenario={} tidak tersedia atau tidak valid".format(sid))

## 16. Analisis kesalahan dengan video (opsional)
Set `enable_video_overlay = True` untuk memproyeksikan pemain dan bola ke frame video pada frame yang salah di
5 menit publik. **Gambar ini berisi potongan video — jangan dibagikan ke asisten AI.**

In [ ]:
def video_overlay(frames, labels, title_prefix=""):
    if "video" not in ASSETS:
        print("Video tidak tersedia.")
        return
    cap = cv2.VideoCapture(ASSETS["video"])
    B_ = SOURCES["atd"]["m1d"]["B_est"]
    fig, axes = plt.subplots(1, len(frames), figsize=(6 * len(frames), 3.6))
    for ax, f in zip(np.atleast_1d(axes), frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f))
        ok, img = cap.read()
        if not ok:
            continue
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        for k in (0, 1):
            pts = P[f, TEAM_OF_SLOT == k]
            pts = pts[~np.isnan(pts[:, 0])]
            if len(pts):
                uv = np.array([pitch_to_px(f, x_, y_) for x_, y_ in pts])
                ax.scatter(uv[:, 0], uv[:, 1], s=25, facecolors="none", edgecolors=COLORS["HA"[k]], lw=1.5)
        if not np.isnan(B_[f, 0]):
            u, v = pitch_to_px(f, B_[f, 0], B_[f, 1])
            ax.scatter(u, v, s=40, color="yellow", edgecolors="k")
        ax.set_title("{}frame {} | model {} | GT {}".format(title_prefix, f, labels[f], GT_PUB_LAB[f]), fontsize=8)
        ax.axis("off")
    cap.release()
    save_fig(fig, "24_overlay_video_{}".format(frames[0]), "Overlay video (JANGAN DIBAGIKAN)", fig_dir, group="perbandingan")


if CONFIG["enable_video_overlay"] and ok_rows:
    best = COMP.loc[ok_rows, "rank_core"].astype(float).idxmin()
    lab = SCENARIO_RESULTS[best]["labels"]
    wrong = np.flatnonzero((GT_PUB_LAB != "X") & (GT_PUB_LAB != lab))
    if len(wrong):
        video_overlay(list(wrong[np.linspace(0, len(wrong) - 1, 3).astype(int)]), lab, "{} salah: ".format(best))

## 17. Manifest reproduksi, galeri, dan ringkasan run

In [ ]:
with open(os.path.join(DIRS["run"], "requirements_used.txt"), "w") as fh:
    for pkg, ver in (("numpy", np.__version__), ("pandas", pd.__version__), ("scipy", scipy.__version__),
                     ("scikit-learn", sklearn.__version__), ("lightgbm", lgb.__version__), ("xgboost", xgb.__version__),
                     ("catboost", catboost.__version__), ("opencv-python-headless", cv2.__version__),
                     ("matplotlib", matplotlib.__version__)):
        fh.write("{}=={}\n".format(pkg, ver))
    if YOLO_RT.get("ok"):
        fh.write("torch=={}\nultralytics=={}\n".format(YOLO_RT["torch"], YOLO_RT["ultralytics"]))
for res in SCENARIO_RESULTS.values():
    if not res.get("skipped"):
        shutil.copy2(os.path.join(DIRS["run"], "requirements_used.txt"), os.path.join(res["folder"], "requirements_used.txt"))

MANIFEST = {"run_id": RUN_ID, "config": CONFIG, "env": ENV_INFO, "official_repo_commit": OFFICIAL_COMMIT,
            "checkpoint_keys": CKPT.keys, "checkpoint_dir": DIRS["checkpoints"],
            "loaded_from_shared_cache": CKPT.loaded_from_shared, "parser_check": PARSER_CHECK, "yolo_runtime": YOLO_RT, "video": VIDEO_INFO,
            "scenarios": {s: {k: v for k, v in r.items() if k != "labels"} for s, r in SCENARIO_RESULTS.items()}}
with open(os.path.join(DIRS["run"], "run_manifest.json"), "w") as fh:
    json.dump(MANIFEST, fh, indent=2, default=str, ensure_ascii=False)

pdf_path = os.path.join(DIRS["figures"], "figures_all.pdf")
with PdfPages(pdf_path) as pdf:
    for key, meta in FIG_MANIFEST.items():
        img = plt.imread(meta["path"])
        h_, w_ = img.shape[:2]
        fig = plt.figure(figsize=(11, 11 * h_ / w_ + 0.6))
        plt.imshow(img)
        plt.axis("off")
        plt.title("{} — {}".format(key, meta["caption"]), fontsize=9)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)
html = ["<html><head><meta charset='utf-8'><title>MTAH 2027 run {}</title>".format(RUN_ID),
        "<style>body{font-family:sans-serif;max-width:1150px;margin:auto}img{max-width:100%;border:1px solid #ddd}"
        "table{border-collapse:collapse;font-size:12px}td,th{border:1px solid #ccc;padding:3px 6px}</style></head><body>",
        "<h1>MTAH 2027 — run {}</h1>".format(RUN_ID), COMP[show_cols].to_html(float_format=lambda v: "{:.4f}".format(v))]
groups = {}
for key, meta in FIG_MANIFEST.items():
    groups.setdefault(meta["group"], []).append((key, meta))
for g, items in groups.items():
    html.append("<h2>{}</h2>".format(g))
    for key, meta in items:
        html.append("<h4>{}</h4><p>{}</p><img src='{}'>".format(key, meta["caption"],
                                                               os.path.relpath(meta["path"], DIRS["figures"]).replace("\\", "/")))
html.append("</body></html>")
open(os.path.join(DIRS["figures"], "index.html"), "w", encoding="utf-8").write("\n".join(html))

print("=== RINGKASAN RUN {} ===".format(RUN_ID))
print(COMP[show_cols].to_string(float_format=lambda v: "{:.4f}".format(v)))
print("\nPaket per skenario :", DIRS["scenarios"])
print("Perbandingan       :", os.path.join(DIRS["comparison"], "README.md"))
print("Galeri / PDF       :", os.path.join(DIRS["figures"], "index.html"), "|", pdf_path)
print("Manifest           :", os.path.join(DIRS["run"], "run_manifest.json"))

if RESULTS_MIRROR:
    _dst = os.path.join(RESULTS_MIRROR, "runs", RUN_ID)
    t0 = time.time()
    try:
        retry_io(lambda: shutil.copytree(DIRS["run"], _dst, dirs_exist_ok=True), "menyalin hasil ke results_mirror_dir")
        print("Hasil run disalin ke: {} ({:.0f} s)".format(_dst, time.time() - t0))
    except Exception as _exc:
        print(explain_error("menyalin hasil ke results_mirror_dir (hasil tetap ada di work_dir)", _exc))
_skipped = {s_: r_["skipped"] for s_, r_ in SCENARIO_RESULTS.items() if r_.get("skipped")}
print("\n=== STATUS AKHIR ===")
print("Skenario selesai :", [s_ for s_, r_ in SCENARIO_RESULTS.items() if not r_.get("skipped")])
if _skipped:
    print("Skenario dilewati:")
    for s_, why in _skipped.items():
        print("  {} -> {}".format(s_, why))
    print("Perbaiki penyebabnya lalu 'Run all' lagi; tahap yang sudah selesai dimuat dari checkpoint.")

## 18. Catatan dan langkah berikutnya
- **Yang boleh dilaporkan ke asisten AI:** tabel perbandingan (Bagian 15), `vs_previous.csv`, diagnostik model (9.1),
  metrik YOLO v3 (angka/JSON: ringkasan label ulang, tabel ambang, evaluasi), dan output validator.
  **Jangan** gambar Bagian 12 atau 16.
- **Menyetel v3 tanpa training ulang dari awal:** parameter seleksi (`target_precision`, `track_*`) dan fusi
  (`yolo_fusion`) hanya menghitung ulang seleksi + sumber `yolo` (menit, bukan jam). Parameter label ulang
  (`relabel_*`, `min_box_px`, `neg_frac`) membangun ulang dataset v3 dan **melatih ulang** YOLO v3.
- Bila hasil S6 v3 tetap di bawah S5, itu temuan yang sah: kualitas posisi bola bukan penghambat utama; batas
  akurasi ada pada timing transisi dan aturan kepemilikan GT.
- Menambah skenario tanpa biaya latih ekstra: model anggota untuk S6 sudah dilatih pada sumber YOLO, jadi
  `"S7": {"name": "YOLO_LightGBM", "source": "yolo", "model": "lgbm"}` hanya menambah decoding.
- Pengembangan berikutnya: restart per jenis (throw-in/corner/goal kick dengan aturan sentuhan terakhir) dan model
  sekuens (TCN/BiGRU) sebagai pengganti emisi buatan tangan.